In [ ]:
#图5、附图5-10网络图
import pandas as pd         # 用于数据处理和分析，特别是表格型数据 (DataFrame)。
import subprocess     # 用于创建和管理新的进程，比如在 Python 中运行系统命令。
import matplotlib.pyplot as plt # 用于创建静态、交互式和动画的可视化图表。通常简写为 plt。
import seaborn as sns       # 基于 matplotlib 的数据可视化库，提供更高级、美观的统计图表。通常简写为 sns。
import statistics     # 提供数学统计函数，用于计算数据的均值、中位数、方差等。
import numpy as np          # 用于进行科学计算，特别是数组和矩阵操作。通常简写为 np。
import math           # 提供标准的数学函数，如三角函数、对数、平方根等。
import os             # 用于与操作系统进行交互，如文件路径操作、目录创建等。
import re
from skbio.stats import subsample_counts
#  辅助函数：计算生态位宽度 (Bj) 
def calculate_levins_bj(df):
    """
    计算每个物种的 Levin's 生态位宽度 Bj。
    
    公式: Bj = 1 / sum(Pij^2)
    其中 Pij 为物种 j 在样本 i 中的相对丰度。
    
    参数:
        df (pd.DataFrame): 行 = OTU/物种, 列 = 样本, 值为相对丰度
    
    返回:
        pd.Series: 每个物种对应的 Bj 值
    """
    matrix = df.astype(float)
    
    # 对每个物种（行），跨所有样本计算 Pij² 的加和
    sum_sq = (matrix ** 2).sum(axis=1)
    
    # 避免除以零，将0替换为NaN后再填回0
    bj = 1.0 / sum_sq.replace(0, np.nan)
    
    return bj.fillna(0)


def classify_generalist_specialist(
    relative_abundance_matrix,
    num_permutations=1000,
    seed=42
):
    """
    基于生态位宽度（Levin's Bj）的置换检验，将物种分类为：
    广适种（Generalist）、专适种（Specialist）或中性种（Neutral）。
    
    分类标准：
        Generalist : 观测 Bj 显著大于零分布（p_generalist < 0.05）
        Specialist : 观测 Bj 显著小于零分布（p_specialist < 0.05）
        Neutral    : 其余物种
    
    参数:
        relative_abundance_matrix (pd.DataFrame):
            行 = OTU/物种, 列 = 样本/生境, 值为相对丰度
        num_permutations (int):
            置换次数，默认 1000 次
        seed (int):
            随机种子，保证结果可重复，默认 42
    
    返回:
        pd.DataFrame: index = 物种名，包含以下列：
            - Bj_Observed  : 观测生态位宽度
            - Bj_Expected  : 零分布的平均生态位宽度
            - Bj_Std       : 零分布的标准差
            - Z_Score      : 标准化得分，衡量偏离零分布的程度
            - p_generalist : 右尾 p 值（观测值是否显著偏大）
            - p_specialist : 左尾 p 值（观测值是否显著偏小）
            - Ecotype      : 分类结果（Generalist / Specialist / Neutral）
    """
    # ---------- 1. 数据准备 ----------
    mat = relative_abundance_matrix.fillna(0).astype(float)
    species = mat.index
    n_species, n_samples = mat.shape
    
    print(f"输入矩阵维度: {n_species} 个物种 × {n_samples} 个样本")

    # ---------- 2. 计算观测 Bj ----------
    observed_bj = calculate_levins_bj(mat)

    # ---------- 3. 置换检验，构建零分布 ----------
    # 使用独立随机生成器，避免影响全局随机状态
    rng = np.random.default_rng(seed=seed)
    
    # 预分配结果数组：行 = 物种，列 = 每次置换的 Bj
    permuted_bj_array = np.zeros((n_species, num_permutations), dtype=float)

    for i in range(num_permutations):
        # 将整个相对丰度矩阵展平后随机打乱
        # 目的：彻底破坏物种与样本之间的对应关系，生成零假设下的分布
        shuffled_flat = mat.values.flatten().copy()
        rng.shuffle(shuffled_flat)
        
        # 重塑回原始矩阵形状，计算该置换下每个物种的 Bj
        shuffled_df = pd.DataFrame(
            shuffled_flat.reshape(n_species, n_samples),
            index=species,
            columns=mat.columns
        )
        permuted_bj_array[:, i] = calculate_levins_bj(shuffled_df).values

    # ---------- 4. 零分布统计量 ----------
    # 零分布的均值：代表随机期望下的 Bj
    expected_bj = permuted_bj_array.mean(axis=1)
    
    # 零分布的标准差（使用无偏估计 ddof=1）
    std_null = permuted_bj_array.std(axis=1, ddof=1)
    
    # Z-score：观测值偏离零分布均值的标准化程度
    # 标准差为 0 时（所有置换结果相同）设为 NaN，避免除以零
    z_score = (observed_bj.values - expected_bj) / np.where(
        std_null == 0, np.nan, std_null
    )

    # ---------- 5. 计算置换 p 值 ----------
    # 使用 +1 修正（Phipson & Smyth 2010），避免 p 值为 0 的情况
    obs_val = observed_bj.values.reshape(-1, 1)
    
    # 右尾 p 值：零分布中 Bj >= 观测值的比例
    # p 值越小，说明观测 Bj 显著偏大 → 倾向于 Generalist
    p_gen = (
        (permuted_bj_array >= obs_val).sum(axis=1) + 1
    ) / (num_permutations + 1)
    
    # 左尾 p 值：零分布中 Bj <= 观测值的比例
    # p 值越小，说明观测 Bj 显著偏小 → 倾向于 Specialist
    p_spe = (
        (permuted_bj_array <= obs_val).sum(axis=1) + 1
    ) / (num_permutations + 1)

    # ---------- 6. 汇总结果 ----------
    results = pd.DataFrame({
        'Bj_Observed' : observed_bj.values,
        'Bj_Expected' : expected_bj,
        'Bj_Std'      : std_null,
        'Z_Score'     : z_score,
        'p_generalist': p_gen,
        'p_specialist': p_spe,
    }, index=species)

    # ---------- 7. 物种分类 ----------
    results['Ecotype'] = 'Neutral'
    results.loc[results['p_generalist'] < 0.05, 'Ecotype'] = 'Generalist'
    results.loc[results['p_specialist'] < 0.05, 'Ecotype'] = 'Specialist'

    # ---------- 8. 输出统计摘要 ----------
    print("\n物种分类统计:")
    print(results['Ecotype'].value_counts().to_string())

    return results



#细菌共线性
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
bacteria_ASV = pd.read_csv("/mnt/d/study/master/meiji/bacteria_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 bacteria_ASV DataFrame 中选择指定的列
b_ASV = bacteria_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
b_ASV.set_index(keys="ASV", inplace=True)
b_tax = bacteria_ASV.iloc[:, 1:9].copy()
b_tax.set_index(keys="ASV", inplace=True)
print(len(b_ASV))
# 计算 ASV 相对丰度
b_ASV_df = b_ASV.copy()

#按种
b_species = b_ASV.join(b_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
b_species = b_species.groupby('Species')[sample_ids].sum()
#排序
b_species = b_species.sort_index()
b_species_df = b_species.copy()
print(len(b_species_df))
# 将当前的 ASV 索引重置为一个普通列。
b_tax_reset_index = b_tax.reset_index()
# 将 'Species' 列设置为新的索引。
b_tax_species = b_tax_reset_index.groupby('Species').first()
b_tax_species = b_tax_species.drop(columns=['ASV'])
b_tax_species = b_tax_species.sort_index()
# 进行替换操作
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__unclassified_k__norank_d__Bacteria', 'Unclassified')
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__SAR324_cladeMarine_group_B', 'SAR324')
# 去除 'p__' 前缀
b_tax_species['Phylum'] = b_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
b_unique_species_names = b_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
b_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(b_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 b_species 的索引
b_species = b_species.rename(index= b_species_to_numeric_id)
# 转换 b_tax_species 的索引
b_tax_species = b_tax_species.rename(index= b_species_to_numeric_id)
b_species_df = b_species.copy()
print(len(b_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
b_species_rel = b_species_df.div(b_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
b_rel_long = b_species_rel.stack().reset_index()
b_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
b_rel_long['Origin'] = b_rel_long['Sample'].map(sample_origins_map)
b_rel_long['Niche'] = b_rel_long['Sample'].map(sample_niches_map)
b_species_total_counts = b_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
b_total_counts_across_all_species = b_species_total_counts.sum() # 计算所有 species 的总丰度
b_species_relative_abundance = b_species_total_counts / b_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
b_species_above_overall_threshold = b_species_relative_abundance[b_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
b_rel_long_filtered_by_overall_abundance = b_rel_long[b_rel_long['species'].isin(b_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
b_species_origin_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
b_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
b_species_in_origin = b_species_origin_mean_filtered[b_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
b_species_origin_count = b_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
b_species_niche_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
b_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
b_species_in_niche = b_species_niche_mean_filtered[b_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
b_species_niche_count = b_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
b_species_counts_combined = pd.merge(b_species_origin_count, b_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 3
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
b_core_species_names = b_species_counts_combined[
    (b_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (b_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
b_core_abundance = b_species_rel.loc[b_core_species_names]
b_core_species_raw_counts = b_species.loc[b_core_species_names]
print(len(b_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
b_median_relative_abundance = b_species_relative_abundance.median()
print(b_median_relative_abundance)
# 定义稀有类群的丰度阈值 
b_rare_threshold = 0.00001
# 筛选出稀有 species 
b_rare_species = b_species_relative_abundance[b_species_relative_abundance < b_rare_threshold]
# 获取稀有 species 的名称（索引）
b_rare_species_names = b_rare_species.index.tolist()
len(b_rare_species_names)
b_species_db_transposed = b_species_df.T 
b_speciesdfmelt = b_species_db_transposed.stack().reset_index()# 长宽表格转换
b_speciesdfmelt.columns = ['Sample', 'species', 'value']
b_speciesdfmelt = b_speciesdfmelt[b_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
b_control_species = b_speciesdfmelt.copy()
b_control_species['Group'] = b_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
b_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
b_controlspecies_unique = b_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
b_controlspecies_unique = b_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
b_controlspecies_unique = b_controlspecies_unique[b_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
b_unique_species_names_single_group = b_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
b_set_rare_species = set(b_rare_species_names)
b_set_unique_species_single_group = set(b_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
b_rare_unique_species_names = list(b_set_rare_species.intersection(b_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
b_rare_unique_abundance = b_species_df.loc[b_rare_unique_species_names]
b_rare_unique_species_raw_counts = b_species.loc[b_rare_unique_species_names]
print(len(b_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度置换检验与 Z-score 计算
num_permutations = 1000 
b_niche_breadth_results = classify_generalist_specialist(
    b_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (含 Z-score, 前5个 species) ")
print(b_niche_breadth_results.head())
b_generalists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
b_specialists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(b_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(b_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_generalists_species:
    print("无广适类群 species 数据。")
else:
    b_generalists_raw_counts = b_species.loc[b_generalists_species]
    b_generalists_abundance = b_species_rel.loc[b_generalists_species]
    print(b_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_specialists_species:
    print("无专适类群 species 数据。")
else:
    b_specialists_raw_counts = b_species.loc[b_specialists_species]
    b_specialists_abundance = b_species_rel.loc[b_specialists_species]
print(b_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")


# 加载相对丰度、丰度物种、稀有物种、广适种和狭适种数据
b_db_OTU_rel = b_species_rel 
b_db_abun = b_core_abundance 
b_db_rare = b_rare_unique_abundance
b_db_gen = b_generalists_abundance
b_db_spe = b_specialists_abundance
# 加载分类学数据
b_otu_taxon = b_tax_species
# 从 otu_taxon 中删除不需要的分类学列，只保留 OTU ID 和 Phylum 相关的信息
# 此处drop了除Phylum之外的所有分类学层级和OTUs列
b_otu_network_color = b_otu_taxon.drop(["Kingdom", "Class", "Order", "Family", "Genus"], axis=1)
b_otu_network_color = b_otu_network_color.reset_index(drop=True)
# 为 DataFrame 创建一个新的 'id' 列，其值为当前索引
b_otu_network_color["id"] = b_otu_network_color.index
# 重新排序并只保留 'id' 和 'Phylum' 列
b_otu_network_color = b_otu_network_color[['id', 'Phylum']]
# 将 'id' 列的值加 1 (通常是为了使 ID 从 1 开始而不是从 0 开始，这在某些网络分析软件中很常见)
b_otu_network_color["id"] += 1
b_otu_network_color_phyla = b_otu_network_color.copy()
# 将结果保存为 Excel 文件，不包含索引
b_otu_network_color_phyla.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_otu_network_color_phyla.xlsx", index=False)
# 根据丰度 (abundance) 进行颜色编码
# 读取稀有化后的丰度数据 (或包含Ecotype信息的数据)
# 将 b_species 的索引 (species ID) 转换为一个新列
b_species_ecotype_df = b_species.reset_index()
b_species_ecotype_df = b_species_ecotype_df[['Species']].copy() # 在这里明确创建副本
# 添加 'Ecotype' 列并初始化为 'Others'
b_species_ecotype_df['Ecotype'] = 'Others'
# 根据 b_core_species_names 标记 'Cores' species
b_species_ecotype_df.loc[b_species_ecotype_df['Species'].isin(b_core_species_names), 'Ecotype'] = 'Cores'
# 根据 b_rare_unique_species_names 标记 'Uniques' species
b_species_ecotype_df.loc[b_species_ecotype_df['Species'].isin(b_rare_unique_species_names), 'Ecotype'] = 'Uniques'
b_otu_network_color = b_species_ecotype_df.copy()
# 为 DataFrame 创建一个新的 'id' 列，其值为当前索引
b_otu_network_color["id"] = b_otu_network_color.index
# 重新排序并只保留 'id' 和 'Ecotype' 列
b_otu_network_color = b_otu_network_color[['id', 'Ecotype']].copy()
# 将 'id' 列的值加 1
b_otu_network_color["id"] += 1
b_otu_network_color_abundance = b_otu_network_color.copy()
# 将结果保存为 Excel 文件，不包含索引
b_otu_network_color_abundance.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_otu_network_color_abundance.xlsx", index=False)
# 根据生态位宽度 (niche breadth) 进行颜色编码
# 读取表示基因-物种关系（或包含Ecotype信息）的数据
# 初始化一个新的 DataFrame，只包含 species ID，并作为独立副本
b_species_niche_ecotype_df = pd.DataFrame({'Species': b_species.index.tolist()}).copy()
# 添加 'Ecotype' 列并初始化为 'Others'
b_species_niche_ecotype_df['Ecotype'] = 'Others'
# 标记 'Generalists' species
# 使用 .loc[] 安全地进行条件赋值
b_species_niche_ecotype_df.loc[
    b_species_niche_ecotype_df['Species'].isin(b_generalists_species),
    'Ecotype'
] = 'Generalists'
# 标记 'Specialists' species
b_species_niche_ecotype_df.loc[
    (b_species_niche_ecotype_df['Species'].isin(b_specialists_species)),
    'Ecotype'
] = 'Specialists'
b_otu_network_color = b_species_niche_ecotype_df.copy()
# 为 DataFrame 创建一个新的 'id' 列，其值为当前索引
b_otu_network_color["id"] = b_otu_network_color.index
# 重新排序并只保留 'id' 和 'Ecotype' 列
b_otu_network_color = b_otu_network_color[['id', 'Ecotype']]
# 将 'id' 列的值加 1
b_otu_network_color["id"] += 1
b_otu_network_color_breadth = b_otu_network_color.copy()
# 将结果保存为 Excel 文件，不包含索引
b_otu_network_color_breadth.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_otu_network_color_breadth.xlsx", index=False)
# 处理所有 OTUs
# 读取 OTU 稀有化后的相对丰度数据，并将第一列设置为索引
b_otu_relative = b_species_rel.copy()
# 对 DataFrame 进行转置，即行和列互换
b_otu_relative = b_otu_relative.transpose()
# 将转置后的 DataFrame 保存为一个新的 CSV 文件，保留索引作为第一列
b_otu_relative.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_transposed_OTU_rar_relative.csv", index=True)
# 处理 OTUs 的子集用于测试
# 创建数据子集用于测试
# 读取转置后的 OTU 相对丰度数据
b_transposed_OTU_rar_relative = b_otu_relative.copy()
# 选取 DataFrame 的前 100 行和前 100 列作为子集
b_subset = b_transposed_OTU_rar_relative.iloc[:100, :100]
b_subset_transposed_OTU_rar_relative = b_subset.copy()
# 将子集保存为 CSV 文件，不包含索引
b_subset_transposed_OTU_rar_relative.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_subset_transposed_OTU_rar_relative.csv", index=False)
# 为测试目的填充颜色矩阵
# 读取 OTU 门类别的颜色编码文件
b_otu_network_color = b_otu_network_color_phyla.copy()
# 选取颜色矩阵的前 99 行作为子集
b_otu_network_color_subset = b_otu_network_color.iloc[:99,:]
b_subset_otu_network_color = b_otu_network_color_subset.copy()
# 将颜色矩阵子集保存为 Excel 文件，不包含索引
b_subset_otu_network_color.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_subset_otu_network_color.xlsx", index=False)
# OTU 网络度量函数
# 定义一个函数，用于解析网络分析结果并整理成 DataFrame
# result: 包含网络分析原始输出的字符串
# network_color_file: 包含 ASV id 和颜色信息的 DataFrame (通常是 ASV_network_color_phyla.xlsx 的内容)
def network_metrics(result, network_color_file):
    # 从 network_color_file 中提取 'id' 列并转换为列表
    network_color_list = network_color_file["id"].tolist()

    # 将原始结果字符串按行分割
    lines = result.split('\n')
    # 从第一行中提取边密度 (b_edge_density) 的浮点数值
    # 假设第一行格式为 '[1] 0.1234'，通过切片去除 '[1] '
    b_edge_density = float(lines[0][len('[1] '):])

    # 初始化空字典用于存储度 (degree)、接近中心性 (closeness) 和中介中心性 (betweenness)
    degree = {}
    closeness = {}
    betweenness = {}
    current_stage = '' # 用于标记当前正在解析哪种网络度量
    lines = lines[1:] # 跳过已经处理过的第一行 (边密度)
    a = iter(lines) # 创建一个迭代器，以便可以每次取两行
    # 遍历处理后的行，每次取两行 (line1 包含键，line2 包含值)
    for line1, line2 in zip(a, a):
        # 检查当前行是否包含网络度量类型的标识（'degree', 'closeness', 'betweenness'）
        if 'degree' in line1 or 'closeness' in line1 or 'betweenness' in line1:
            current_stage = line1.strip()  # 更新当前正在处理的度量类型，并去除空白
            continue # 跳过当前行，进入下一轮循环读取数据行
        keys = line1.split() # 将第一行按空格分割，得到键 (ASV ID)
        vals = line2.split() # 将第二行按空格分割，得到值 (度量数值)
        # 遍历键值对，将数据存入相应的字典中
        for k, v in zip(keys, vals):
            if 'degree' in current_stage:
                degree[k] = int(v) # 度为整数
            elif 'closeness' in current_stage:
                closeness[k] = float(v) # 接近中心性为浮点数
            elif 'betweenness' in current_stage:
                betweenness[k] = float(v) # 中介中心性为浮点数

    # 将所有键（ASV ID）转换为 'ASV' 格式，并在排序后添加前缀
    # 收集所有字典中的键，去重并转换为整数后排序 (确保按数字顺序排序)
    all_keys = sorted(set(degree.keys()).union(closeness.keys(), betweenness.keys()), key=int)
    # 为排序后的键添加 'ASV' 前缀
    all_keys = ["ASV" + k for k in all_keys]  # 在排序后添加 'ASV' 前缀

    # 准备用于创建 DataFrame 的数据
    db_data = {
        'id': network_color_list, # 使用传入的颜色文件中的 'id' 列表作为 ID
        # 获取 degree 值。由于 all_keys 带有 'ASV' 前缀，所以需要 k[3:] 去除前缀来匹配字典中的键。
        # 如果键不存在，则默认为 0。
        'degree': [degree.get(k[3:], 0) for k in all_keys],
        'closeness': [closeness.get(k[3:], 0) for k in all_keys],
        'betweenness': [betweenness.get(k[3:], 0) for k in all_keys]
    }
    # 使用准备好的数据创建 Pandas DataFrame
    b_df = pd.DataFrame(db_data)
# 返回边密度和包含网络度量值的 DataFrame
    return b_edge_density, b_df


# 用于测试的子集数据 (Subset for testing)
# 读取子集 OTU 网络颜色数据到 DataFrame
b_subset_otu_network_color_df = b_subset_otu_network_color.copy()
# 定义子集 OTU 网络颜色文件的路径
b_subset_otu_network_color = "/mnt/d/study/master/Main_Figure_tables/Figure_5/b_subset_otu_network_color.xlsx"
# 定义子集 OTU 稀有化相对丰度文件的路径
b_subset_otu_rar_relative = "/mnt/d/study/master/Main_Figure_tables/Figure_5/b_subset_transposed_OTU_rar_relative.csv"
# 生成网络度量指标 (Generate network metrics)
# 调用 R 脚本 'SpiecEasi_stage_python_phyla.R' 来生成网络度量。
# 传入稀有化相对丰度子集和网络颜色子集文件作为参数。
# universal_newlines=True: 将输出解释为文本。
b_subset_otu_network_metrics = subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_python_phyla.R', b_subset_otu_rar_relative, b_subset_otu_network_color],
    universal_newlines=True,
    stderr=subprocess.PIPE
)
# 调用自定义的 network_metrics 函数解析 R 脚本的输出，获取边密度和网络度量 DataFrame
b_edge_density, b_df = network_metrics(b_subset_otu_network_metrics, b_subset_otu_network_color_df)
# 打印边密度
print("B_edge_density: ", b_edge_density)
# 预期输出：B_edge_density:
# 绘制网络图 (Plot network)
# 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制网络图。
# 传入稀有化相对丰度子集、网络颜色子集文件，以及输出 PDF 和边权重 CSV 文件的路径。
subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R',
     b_subset_otu_rar_relative, b_subset_otu_network_color,
     '/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_SpiecEasi_subset.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_weight_subset.csv'],
    universal_newlines=True,
    stderr=subprocess.DEVNULL
)
# 预期输出：'null device \n          1 \n' (R 脚本的默认输出，表示绘图设备已激活)
# 处理所有 OTUs (All OTUs)
# 读取所有 OTU 网络颜色数据到 DataFrame (这里使用的是按门类的数据作为主颜色文件)
b_otu_network_color_df = b_otu_network_color_phyla.copy()
# 定义所有 OTU 网络颜色文件的路径 (按门类)
b_otu_network_color_phyla = "/mnt/d/study/master/Main_Figure_tables/Figure_5/b_otu_network_color_phyla.xlsx"
# 定义所有 OTU 网络颜色文件的路径 (按丰度/分组)
b_otu_network_color_abundance = "/mnt/d/study/master/Main_Figure_tables/Figure_5/b_otu_network_color_abundance.xlsx"
# 定义所有 OTU 网络颜色文件的路径 (按生态位宽度)
b_otu_network_color_breadth= "/mnt/d/study/master/Main_Figure_tables/Figure_5/b_otu_network_color_breadth.xlsx"
# 定义所有 OTU 稀有化相对丰度文件的路径
b_otu_rar_relative = "/mnt/d/study/master/Main_Figure_tables/Figure_5/b_transposed_OTU_rar_relative.csv"
# 生成网络度量指标 (Generate network metrics)
# 调用 R 脚本 'SpiecEasi_stage_python_phyla.R' 来生成所有 OTU 的网络度量。
# 传入所有 OTU 的稀有化相对丰度文件和按门类的颜色文件作为参数。
b_otu_network_metrics = subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_python_phyla.R', b_otu_rar_relative, b_otu_network_color_phyla], # 注意这里使用了b_otu_network_color_phyla
    universal_newlines=True,
    stderr=subprocess.DEVNULL
)
# 调用自定义的 network_metrics 函数解析 R 脚本的输出，获取边密度和网络度量 DataFrame
b_edge_density, b_df = network_metrics(b_otu_network_metrics, b_otu_network_color_df)
# 打印边密度
print("B_edge_density: ", b_edge_density)
# 预期输出：B_edge_density: 
# 将计算出的边密度保存为 CSV 文件
b_edge_density_df = pd.DataFrame(data={'edge density': [b_edge_density]})
b_edge_density_df.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_density_all_OTUs.csv", index=False)
# 将计算出的网络度量 DataFrame (df) 保存为 CSV 文件
b_df.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_metrics_all_OTUs.csv", index = False)
# 绘制网络图，按门类进行颜色编码 (Plot network color code by phyla)
# 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制网络图。
# 传入所有 OTU 的稀有化相对丰度文件、按门类的颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R',
     b_otu_rar_relative, b_otu_network_color_phyla,
     '/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_SpiecEasi_all_OTUs_phyla.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_weight_all_OTUs.csv'],
    universal_newlines=True,
    stderr=subprocess.DEVNULL
)
# 预期输出：'null device \n          1 \n'

# 绘制网络图，按丰度进行颜色编码 (Plot network color code by abundance)
# 调用 R 脚本 'SpiecEasi_stage_network_ecotypes.R' 来绘制网络图。
# 注意：这里调用的是 'SpiecEasi_stage_network_ecotypes.R' 脚本。
# 传入所有 OTU 的稀有化相对丰度文件、按丰度的颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_ecotypes.R',
     b_otu_rar_relative, b_otu_network_color_abundance,
     '/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_SpiecEasi_all_OTUs_abundance.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_weight_all_OTUs.csv'],
    universal_newlines=True,
    stderr=subprocess.DEVNULL
)
# 预期输出：'null device \n          1 \n'

# 绘制网络图，按生态位宽度进行颜色编码 (Plot network color code by niche breadth)
# 调用 R 脚本 'SpiecEasi_stage_network_ecotypes.R' 来绘制网络图。
# 传入所有 OTU 的稀有化相对丰度文件、按生态位宽度的颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_ecotypes.R',
     b_otu_rar_relative, b_otu_network_color_breadth,
     '/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_SpiecEasi_all_OTUs_breadth.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_weight_all_OTUs.csv'],
    universal_newlines=True,
    stderr=subprocess.DEVNULL
)
# 预期输出：'null device \n          1 \n'

# 根据需要过滤 OTU 以改善网络可视化效果
# 由于网络图过于混乱，通过以下两步过程减少用于分析的 OTU 数量。
# 1. 剔除在 X% 样本中存在的 OTU。
# 2. 剔除与所有其他 OTU 的 Spearman 相关性小于 y 的 OTU。
# # 读取原始的转置 OTU 稀有化相对丰度数据
# b_df = pd.read_csv('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_transposed_OTU_rar_relative.csv', index_col=0)
# # 将 DataFrame 中的所有 0 值替换为 NaN (这样在计数时不会被计算，方便统计存在性)
# b_df.replace(0, np.nan, inplace=True)
# # 第一步：根据 OTU 在样本中的存在比例进行过滤
# # 计算阈值：0.05（5%）乘以总样本数 618，向下取整。
# threshold = math.floor(0.05*618)
# columns_to_drop_1 = [] # 初始化列表，用于存储要剔除的列名
# for col in b_df.columns: # 遍历 DataFrame 的每一列 (即每一个 OTU)
#     # 计算非零条目（非 NaN 值）的数量
#     non_zero_count = b_df[col].count()
#     # 如果非零条目的数量小于阈值，则将该列标记为待剔除
#     if non_zero_count < threshold:
#         columns_to_drop_1.append(col)
# # 打印根据存在性阈值需要剔除的列数
# print("Fol threshold: ", threshold,", the number of cols to be dropped is ", len(columns_to_drop_1))
# # 重新读取原始数据，因为之前的操作替换了 0 为 NaN
# b_df = pd.read_csv('/mnt/d/study/master/Main_Figure_tables/Figure_5/transposed_OTU_rar_relative.csv', index_col=0)
# # 剔除第一步中标记的列
# b_df = b_df.drop(columns=columns_to_drop_1)
# # 第二步：根据 OTU 间的 Spearman 相关性进行过滤
# # 计算 DataFrame 中所有列之间的 Spearman 相关性矩阵
# correlation_matrix = b_df.corr(method='spearman')
# threshold = 0.4 # 设定相关性阈值
# tolerance_list = [0] # 允许具有高相关性的 OTU 数量 (这里是 0，表示如果一个 OTU 没有任何与它高相关的其他 OTU，则被剔除)
# for tolerance in tolerance_list: # 遍历容忍度列表 (这里只有一个值 0)
#     columns_to_drop_2 = [] # 初始化列表，用于存储第二步要剔除的列名
#     for col in correlation_matrix.columns: # 遍历相关性矩阵的每一列 (即每一个 OTU)
#         # 计算与当前 OTU 相关性绝对值大于阈值的其他 OTU 的数量
#         # .drop(col) 是为了排除自身与自身的相关性 (总是 1)
#         high_corr_count = (correlation_matrix[col].drop(col).abs() > threshold).sum()
#         # 如果高相关性 OTU 的数量小于或等于容忍度，则标记该列为待剔除
#         if high_corr_count <= tolerance:
#             columns_to_drop_2.append(col)
#     # 打印根据相关性阈值和容忍度需要剔除的列数
#     print("Fol tolerance: ", tolerance,", the number of cols to be dropped is ", len(columns_to_drop_2))
# # 剔除第二步中标记的列
# b_df = b_df.drop(columns=columns_to_drop_2)
# # 将经过两步过滤后的 DataFrame 保存为新的 CSV 文件
# b_df.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/less_than_5_and_4_OTU_data.csv")
# 上述被注释代码的运行结果摘要：
# 表示第一步过滤后，如果 OTU 在少于 30 个样本中存在，将剔除 1974 个 OTU。
print("Fol threshold:  30 , the number of cols to be dropped is  1974")
# 表示第二步过滤后，如果一个 OTU 与任何其他 OTU 的 Spearman 相关性低于 0.4 且高相关性 OTU 数量小于等于 0，将剔除 405 个 OTU。
print("Fol tolerance:  0 , the number of cols to be dropped is  405")
# # (被注释掉的代码块 - 以下代码用于更新颜色文件)
# # 从颜色 CSV 文件中剔除相同的 OTU
# # 合并两步过滤后需要剔除的 OTU 列名列表
# columns_to_drop = columns_to_drop_1 + columns_to_drop_2
# # 将要剔除的 OTU 列名转换为整数 ID (假设列名格式为 "OTU_ID")
# ids_to_drop = [int(col[4:]) for col in columns_to_drop]
# # 读取原始的 OTU 网络颜色文件 (按门类)
# otu_network_color = pd.read_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/otu_network_color_phyla.xlsx")
# # 根据需要剔除的 ID 过滤颜色 DataFrame
# b_db_color_filtered = otu_network_color[~otu_network_color['id'].isin(ids_to_drop)]
# # 将过滤后的颜色 DataFrame 保存为新的 Excel 文件
# b_db_color_filtered.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/less_than_5_and_4_otu_network_color.xlsx", index=False)
# # (被注释掉的代码块 - 以下代码用于绘制过滤后的网络图)
# # 加载用于网络构建的输入文件
# # 定义过滤后的 OTU 网络颜色文件路径
# filtered_otu_network_color = "/mnt/d/study/master/Main_Figure_tables/Figure_5/less_than_5_and_4_otu_network_color.xlsx"
# # 定义过滤后的 OTU 稀有化相对丰度文件路径
# filtered_otu_rar_relative = "/mnt/d/study/master/Main_Figure_tables/Figure_5/less_than_5_and_4_OTU_data.csv"
# # 绘制网络图
# # 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制过滤后的网络图。
# # 传入过滤后的数据文件、颜色文件以及输出 PDF 和边权重 CSV 文件的路径。
# subprocess.check_output(['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R',
#                          filtered_otu_rar_relative, filtered_otu_network_color,
#                          '/mnt/d/study/master/Main_Figure_tables/Figure_5/network_SpiecEasi_filtered_OTUs.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/edge_weight_filtered_OTUs.csv'], universal_newlines=True, stderr=subprocess.DEVNULL)
# # 预期输出：'null device \n          1 \n' (R 脚本的默认输出，表示绘图设备已激活)

# 为居群构建网络 (Network for ecosystems)
b_ecosystem = metadata[['Sample ID', 'Origin']]
# 读取转置后的 OTU 稀有化相对丰度数据
b_transposed_OTU_rar_relative = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_transposed_OTU_rar_relative.csv")
# 合并数据以获取所有样本的生态系统类别
# 将 ecosystem DataFrame 与 transposed_OTU_rar_relative DataFrame 合并。
# 使用 "Sample ID " 和 "Unnamed: 0" 列进行连接。
b_merged_data = pd.merge(b_ecosystem, b_transposed_OTU_rar_relative, left_on="Sample ID", right_on="Unnamed: 0")
# 为每个生态系统类别创建单独的 DataFrame
# 获取 'Origin' 列中所有独特的生态系统类别
b_ecosystem_classes = b_merged_data['Origin'].unique()
# 使用字典推导式为每个生态系统类别创建一个 DataFrame，并删除原始的生态系统编码和样本 ID 列
b_ecosystems_dfs = {b_ecosystem: b_merged_data[b_merged_data['Origin'] == b_ecosystem].drop(columns=['Origin', 'Sample ID']) for b_ecosystem in b_ecosystem_classes}
# 显示每个类别的初始样本数量
b_initial_counts = {b_ecosystem: len(b_df) for b_ecosystem, b_df in b_ecosystems_dfs.items()}
print("Initial Sample Counts:", b_initial_counts)
# 预期输出：Initial Sample Counts: 
# 找到所有 DataFrame 中最小的样本数量
b_min_sample_count = min(b_initial_counts.values())
# 打印最小样本数量
print(b_min_sample_count)
# 预期输出：
# 随机剔除样本，使所有生态系统类别的样本数量与最小数量相等。
b_equalized_dfs = {} # 初始化字典，用于存储样本数量均等化后的 DataFrame
for b_ecosystem, b_df in b_ecosystems_dfs.items():
    if len(b_df) > b_min_sample_count:
        # 如果当前生态系统的样本数量大于最小数量，则随机抽样，使数量等于最小数量
        # random_state=42 保证结果可复现
        b_equalized_dfs[b_ecosystem] = b_df.sample(n= b_min_sample_count, random_state=42)
    else:
        # 否则，直接使用原始 DataFrame (因为它的样本数量已经等于或小于最小数量)
        b_equalized_dfs[b_ecosystem] = b_df
# 移除每个 DataFrame 中所有样本中出现次数为零的 OTU
b_final_dfs = {} # 初始化字典，用于存储最终处理后的 DataFrame
b_final_otu_counts = {} # 初始化字典，用于存储最终的 OTU 数量
for b_ecosystem, b_df in b_equalized_dfs.items():
    # 筛选列：只保留至少在一个样本中存在（非零）的 OTU (列)
    b_df = b_df.loc[:, (b_df != 0).any(axis=0)]
    b_final_dfs[b_ecosystem] = b_df
    # 记录每个生态系统处理后的 OTU 数量
    b_final_otu_counts[b_ecosystem] = b_df.shape[1]
# 显示最终 DataFrame 结构和样本数量
b_final_counts = {b_ecosystem: len(b_df) for b_ecosystem, b_df in b_final_dfs.items()}
print("Final Sample Counts:", b_final_counts)
print("Final OTU Counts:", b_final_otu_counts)
# 预期输出：
# Final Sample Counts:
# Final OTU Counts: 
# 由于移除了在所有样本中出现次数为零的 OTU 后，OTU 数量仍然很高
# 移除在少于或等于阈值数量样本中存在的 OTU
b_final_dfs = {} # 重新初始化，因为下面会根据新的阈值再次过滤
b_otu_presence_counts = {} # 初始化字典，用于存储 OTU 在不同阈值下的存在计数
b_thresholds = [0, 1, 2, 3, 4, 5] # 定义要测试的样本存在阈值列表
for b_ecosystem, b_df in b_equalized_dfs.items():
    # 再次筛选列：只保留至少在一个样本中存在（非零）的 OTU (列)
    b_df = b_df.loc[:, (b_df != 0).any(axis=0)]
    b_final_dfs[b_ecosystem] = b_df
    b_num_samples = len(b_df) # 当前生态系统的样本数量
    b_otu_presence_counts[b_ecosystem] = {} # 初始化当前生态系统的 OTU 存在计数字典
    for b_threshold in b_thresholds: # 遍历每个阈值
        # 计算每个 OTU 在超过阈值数量样本中存在的 OTU 数量
        b_count = (b_df != 0).sum(axis=0) > b_threshold
        b_otu_presence_counts[b_ecosystem][f"{int(b_threshold)} count"] = b_count.sum()
# 显示最终的 DataFrame 结构和 OTU 非零计数
print("OTU Presence with non zero Counts:")
for b_ecosystem, b_counts in b_otu_presence_counts.items():
    print(f"{b_ecosystem}: {b_counts}")
# 预期输出：
# OTU Presence with non zero Counts:

# 最终的计数表明，阈值设置为 0 可以得到一个可管理的 OTU 数量。
# 最终过滤 OTU 并为每个生态系统创建文件和目录
b_final_dfs = {}
b_otu_presence_counts = {}
b_ecosystem_directories = [] # 列表用于存储已创建的生态系统目录路径
b_threshold = 1  # 设置感兴趣的阈值 (OTU 至少在 1 个样本中出现)
for b_ecosystem, b_df in b_equalized_dfs.items():
    if b_ecosystem == "Unknown": # 跳过 'Unknown' 生态系统
        continue
    # 再次过滤掉在所有样本中出现次数为零的 OTU
    b_df = b_df.loc[:, (b_df != 0).any(axis=0)]
    b_final_dfs[b_ecosystem] = b_df
    # 计算每个 OTU 在超过阈值数量样本中存在的 OTU 数量
    b_presence_count = (b_df != 0).sum(axis=0) > b_threshold
    b_otu_presence_counts[b_ecosystem] = b_presence_count.sum() # 存储满足阈值的 OTU 数量
    # 检查生态系统对应的目录是否存在，如果不存在则创建
    directory = f'/mnt/d/study/master/Main_Figure_tables/Figure_5/{b_ecosystem}/'
    if not os.path.exists(directory):
        os.makedirs(directory)
    b_ecosystem_directories.append(directory) # 将新创建的目录添加到列表中
    # 过滤 DataFrame，只保留在超过阈值数量样本中存在的 OTU
    b_filtered_df = b_df.loc[:, (b_df != 0).sum(axis=0) > b_threshold]
    # 将过滤后的 DataFrame 保存为 CSV 文件，不包含索引
    b_filtered_df.to_csv(f'{directory}b_filtered_otus.csv', index = False)
# 显示满足阈值  的 OTU 数量
print("OTU Presence with non-zero counts above the threshold of 0:")
for b_ecosystem, b_count in b_otu_presence_counts.items():
    print(f"{b_ecosystem}: {b_count}")
# 预期输出：
# OTU Presence with non-zero counts above the threshold of 1:

# 修复网络颜色文件 (针对每个生态系统)
for directory in b_ecosystem_directories: # 遍历每个生态系统的目录
    # 重新读取原始的 OTU 网络颜色文件 (按门类)
    b_otu_network_color = pd.read_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_otu_network_color_phyla.xlsx")
    # 读取当前过滤后的 OTU 文件 (这个文件中的列名是 OTU IDs)
    b_db_otu = pd.read_csv(f"{directory}b_filtered_otus.csv")
    # 从列名中提取 OTU ID
    # 假设第一列是样本 ID，所以从第二列开始才是 OTU ID。
    # 提取 OTU ID 的数字部分并转换为整数。
    b_otus = b_db_otu.columns.tolist()[1:]
    b_otu_ids = [int(b_otu.split('_')[1]) for b_otu in b_otus] # 假设 OTU_ID 格式为 "OTU_数字"
    # 过滤颜色数据，只保留 'id' 列与当前文件中 OTU ID 匹配的行
    b_db_color_filtered = b_otu_network_color[b_otu_network_color['id'].isin(b_otu_ids)]
    # 将过滤后的颜色数据保存为 Excel 文件
    b_db_color_filtered.to_excel(f'{directory}b_filtered_network_color.xlsx', index=False)

# 这个函数将使用 SpiecEasi_stage_python.R 文件找到网络度量指标。
# 这个函数使用 OS subprocess 来调用 R 脚本。R 脚本必须与这个 Python 文件在同一个文件夹中。
for directory in b_ecosystem_directories: # 遍历每个生态系统的目录
    # 构建输入和输出文件的完整路径
    b_input_csv_file_name = f'{directory}b_filtered_otus.csv'
    b_color_file_name = f'{directory}b_filtered_network_color.xlsx'
    b_pdb_file_name = f'{directory}b_network_SpiecEasi.pdf'
    b_edge_weight_file_name = f'{directory}b_edge_weight.csv'
    # 读取颜色文件以便在 network_metrics 函数中使用
    b_color_file = pd.read_excel(b_color_file_name)
    # 调用 R 脚本 'SpiecEasi_stage_python_phyla.R' 来计算网络度量。
    # 传入过滤后的 OTU 数据和对应的颜色文件。
    b_result_tb_obs_otu_full = subprocess.check_output(
        ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_python_phyla.R', b_input_csv_file_name, b_color_file_name],
        universal_newlines=True,
        stderr=subprocess.DEVNULL
    )
    # 使用 network_metrics 函数解析 R 脚本的输出，获取边密度和网络度量 DataFrame
    b_edge_density, b_df = network_metrics(b_result_tb_obs_otu_full, b_color_file)
    # 打印当前文件的信息
    print(f"For the data in {b_input_csv_file_name} file")
    # 打印边密度
    print("B_edge_density: ", b_edge_density)
    # 打印度、接近中心性和中介中心性的结果
    print("Degree, closeness, betweeness results are as follows: ")
    print(b_df)
    # 将边密度保存为 CSV 文件
    b_edge_density_df = pd.DataFrame([b_edge_density], columns=["edge density"])
    b_edge_density_df.to_csv(f'{directory}b_edge_density.csv', index = False)
    # 将包含网络度量值的 DataFrame 保存为 CSV 文件
    b_df.to_csv(f'{directory}b_network_metrics.csv', index=False)
    # 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制网络图。
    # 传入过滤后的 OTU 数据、颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
    subprocess.check_output(
        ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R', b_input_csv_file_name, b_color_file_name, b_pdb_file_name, b_edge_weight_file_name],
        universal_newlines=True,
        stderr=subprocess.DEVNULL
    )

# 为生态位构建网络 (Network for ecosystems)
b_ecosystem = metadata[['Sample ID', 'Niche']]
# 读取转置后的 OTU 稀有化相对丰度数据
b_transposed_OTU_rar_relative = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_transposed_OTU_rar_relative.csv")
# 合并数据以获取所有样本的生态系统类别
# 将 ecosystem DataFrame 与 transposed_OTU_rar_relative DataFrame 合并。
# 使用 "Sample ID " 和 "Unnamed: 0" 列进行连接。
b_merged_data = pd.merge(b_ecosystem, b_transposed_OTU_rar_relative, left_on="Sample ID", right_on="Unnamed: 0")
# 为每个生态系统类别创建单独的 DataFrame
# 获取 'Niche' 列中所有独特的生态系统类别
b_ecosystem_classes = b_merged_data['Niche'].unique()
# 使用字典推导式为每个生态系统类别创建一个 DataFrame，并删除原始的生态系统编码和样本 ID 列
b_ecosystems_dfs = {b_ecosystem: b_merged_data[b_merged_data['Niche'] == b_ecosystem].drop(columns=['Niche', 'Sample ID']) for b_ecosystem in b_ecosystem_classes}
# 显示每个类别的初始样本数量
b_initial_counts = {b_ecosystem: len(b_df) for b_ecosystem, b_df in b_ecosystems_dfs.items()}
print("Initial Sample Counts:", b_initial_counts)
# 预期输出：Initial Sample Counts: 
# 找到所有 DataFrame 中最小的样本数量
b_min_sample_count = min(b_initial_counts.values())
# 打印最小样本数量
print(b_min_sample_count)
# 预期输出：
# 随机剔除样本，使所有生态系统类别的样本数量与最小数量相等。
b_equalized_dfs = {} # 初始化字典，用于存储样本数量均等化后的 DataFrame
for b_ecosystem, b_df in b_ecosystems_dfs.items():
    if len(b_df) > b_min_sample_count:
        # 如果当前生态系统的样本数量大于最小数量，则随机抽样，使数量等于最小数量
        # random_state=42 保证结果可复现
        b_equalized_dfs[b_ecosystem] = b_df.sample(n= b_min_sample_count, random_state=42)
    else:
        # 否则，直接使用原始 DataFrame (因为它的样本数量已经等于或小于最小数量)
        b_equalized_dfs[b_ecosystem] = b_df
# 移除每个 DataFrame 中所有样本中出现次数为零的 OTU
b_final_dfs = {} # 初始化字典，用于存储最终处理后的 DataFrame
b_final_otu_counts = {} # 初始化字典，用于存储最终的 OTU 数量
for b_ecosystem, b_df in b_equalized_dfs.items():
    # 筛选列：只保留至少在一个样本中存在（非零）的 OTU (列)
    b_df = b_df.loc[:, (b_df != 0).any(axis=0)]
    b_final_dfs[b_ecosystem] = b_df
    # 记录每个生态系统处理后的 OTU 数量
    b_final_otu_counts[b_ecosystem] = b_df.shape[1]
# 显示最终 DataFrame 结构和样本数量
b_final_counts = {b_ecosystem: len(b_df) for b_ecosystem, b_df in b_final_dfs.items()}
print("Final Sample Counts:", b_final_counts)
print("Final OTU Counts:", b_final_otu_counts)
# 预期输出：
# Final Sample Counts:
# Final OTU Counts: 
# 由于移除了在所有样本中出现次数为零的 OTU 后，OTU 数量仍然很高
# 移除在少于或等于阈值数量样本中存在的 OTU
b_final_dfs = {} # 重新初始化，因为下面会根据新的阈值再次过滤
b_otu_presence_counts = {} # 初始化字典，用于存储 OTU 在不同阈值下的存在计数
b_thresholds = [0, 1, 2, 3, 4, 5] # 定义要测试的样本存在阈值列表
for b_ecosystem, b_df in b_equalized_dfs.items():
    # 再次筛选列：只保留至少在一个样本中存在（非零）的 OTU (列)
    b_df = b_df.loc[:, (b_df != 0).any(axis=0)]
    b_final_dfs[b_ecosystem] = b_df
    b_num_samples = len(b_df) # 当前生态系统的样本数量
    b_otu_presence_counts[b_ecosystem] = {} # 初始化当前生态系统的 OTU 存在计数字典
    for b_threshold in b_thresholds: # 遍历每个阈值
        # 计算每个 OTU 在超过阈值数量样本中存在的 OTU 数量
        b_count = (b_df != 0).sum(axis=0) > b_threshold
        b_otu_presence_counts[b_ecosystem][f"{int(b_threshold)} count"] = b_count.sum()
# 显示最终的 DataFrame 结构和 OTU 非零计数
print("OTU Presence with non zero Counts:")
for b_ecosystem, b_counts in b_otu_presence_counts.items():
    print(f"{b_ecosystem}: {b_counts}")
# 预期输出：
# OTU Presence with non zero Counts:

# 最终的计数表明，阈值设置为 0 可以得到一个可管理的 OTU 数量。
# 最终过滤 OTU 并为每个生态系统创建文件和目录
b_final_dfs = {}
b_otu_presence_counts = {}
b_ecosystem_directories = [] # 列表用于存储已创建的生态系统目录路径
b_threshold = 1  # 设置感兴趣的阈值 (OTU 至少在 1 个样本中出现)
for b_ecosystem, b_df in b_equalized_dfs.items():
    if b_ecosystem == "Unknown": # 跳过 'Unknown' 生态系统
        continue
    # 再次过滤掉在所有样本中出现次数为零的 OTU
    b_df = b_df.loc[:, (b_df != 0).any(axis=0)]
    b_final_dfs[b_ecosystem] = b_df
    # 计算每个 OTU 在超过阈值数量样本中存在的 OTU 数量
    b_presence_count = (b_df != 0).sum(axis=0) > b_threshold
    b_otu_presence_counts[b_ecosystem] = b_presence_count.sum() # 存储满足阈值的 OTU 数量
    # 检查生态系统对应的目录是否存在，如果不存在则创建
    directory = f'/mnt/d/study/master/Main_Figure_tables/Figure_5/{b_ecosystem}/'
    if not os.path.exists(directory):
        os.makedirs(directory)
    b_ecosystem_directories.append(directory) # 将新创建的目录添加到列表中
    # 过滤 DataFrame，只保留在超过阈值数量样本中存在的 OTU
    b_filtered_df = b_df.loc[:, (b_df != 0).sum(axis=0) > b_threshold]
    # 将过滤后的 DataFrame 保存为 CSV 文件，不包含索引
    b_filtered_df.to_csv(f'{directory}b_filtered_otus.csv', index = False)
# 显示满足阈值  的 OTU 数量
print("OTU Presence with non-zero counts above the threshold of 0:")
for b_ecosystem, b_count in b_otu_presence_counts.items():
    print(f"{b_ecosystem}: {b_count}")
# 预期输出：
# OTU Presence with non-zero counts above the threshold of 1:

# 修复网络颜色文件 (针对每个生态系统)
for directory in b_ecosystem_directories: # 遍历每个生态系统的目录
    # 重新读取原始的 OTU 网络颜色文件 (按门类)
    b_otu_network_color = pd.read_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_otu_network_color_phyla.xlsx")
    # 读取当前过滤后的 OTU 文件 (这个文件中的列名是 OTU IDs)
    b_db_otu = pd.read_csv(f"{directory}b_filtered_otus.csv")
    # 从列名中提取 OTU ID
    # 假设第一列是样本 ID，所以从第二列开始才是 OTU ID。
    # 提取 OTU ID 的数字部分并转换为整数。
    b_otus = b_db_otu.columns.tolist()[1:]
    b_otu_ids = [int(b_otu.split('_')[1]) for b_otu in b_otus] # 假设 OTU_ID 格式为 "OTU_数字"
    # 过滤颜色数据，只保留 'id' 列与当前文件中 OTU ID 匹配的行
    b_db_color_filtered = b_otu_network_color[b_otu_network_color['id'].isin(b_otu_ids)]
    # 将过滤后的颜色数据保存为 Excel 文件
    b_db_color_filtered.to_excel(f'{directory}b_filtered_network_color.xlsx', index=False)

# 这个函数将使用 SpiecEasi_stage_python.R 文件找到网络度量指标。
# 这个函数使用 OS subprocess 来调用 R 脚本。R 脚本必须与这个 Python 文件在同一个文件夹中。
for directory in b_ecosystem_directories: # 遍历每个生态系统的目录
    # 构建输入和输出文件的完整路径
    b_input_csv_file_name = f'{directory}b_filtered_otus.csv'
    b_color_file_name = f'{directory}b_filtered_network_color.xlsx'
    b_pdb_file_name = f'{directory}b_network_SpiecEasi.pdf'
    b_edge_weight_file_name = f'{directory}b_edge_weight.csv'
    # 读取颜色文件以便在 network_metrics 函数中使用
    b_color_file = pd.read_excel(b_color_file_name)
    # 调用 R 脚本 'SpiecEasi_stage_python_phyla.R' 来计算网络度量。
    # 传入过滤后的 OTU 数据和对应的颜色文件。
    b_result_tb_obs_otu_full = subprocess.check_output(
        ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_python_phyla.R', b_input_csv_file_name, b_color_file_name],
        universal_newlines=True,
        stderr=subprocess.DEVNULL
    )
    # 使用 network_metrics 函数解析 R 脚本的输出，获取边密度和网络度量 DataFrame
    b_edge_density, b_df = network_metrics(b_result_tb_obs_otu_full, b_color_file)
    # 打印当前文件的信息
    print(f"For the data in {b_input_csv_file_name} file")
    # 打印边密度
    print("B_edge_density: ", b_edge_density)
    # 打印度、接近中心性和中介中心性的结果
    print("Degree, closeness, betweeness results are as follows: ")
    print(b_df)
    # 将边密度保存为 CSV 文件
    b_edge_density_df = pd.DataFrame([b_edge_density], columns=["edge density"])
    b_edge_density_df.to_csv(f'{directory}b_edge_density.csv', index = False)
    # 将包含网络度量值的 DataFrame 保存为 CSV 文件
    b_df.to_csv(f'{directory}b_network_metrics.csv', index=False)
    # 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制网络图。
    # 传入过滤后的 OTU 数据、颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
    subprocess.check_output(
        ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R', b_input_csv_file_name, b_color_file_name, b_pdb_file_name, b_edge_weight_file_name],
        universal_newlines=True,
        stderr=subprocess.DEVNULL
    )



#真菌共线性
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
fungi_ASV = pd.read_csv("/mnt/d/study/master/meiji/fungi_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 fungi_ASV DataFrame 中选择指定的列
f_ASV = fungi_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
f_ASV.set_index(keys="ASV", inplace=True)
f_tax = fungi_ASV.iloc[:, 1:9].copy()
f_tax.set_index(keys="ASV", inplace=True)
print(len(f_ASV))
# 计算 ASV 相对丰度
f_ASV_df = f_ASV.copy()

#按种
f_species = f_ASV.join(f_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
f_species = f_species.groupby('Species')[sample_ids].sum()
#排序
f_species = f_species.sort_index()
# 将当前的 ASV 索引重置为一个普通列。
f_tax_reset_index = f_tax.reset_index()
# 将 'Species' 列设置为新的索引。
f_tax_species = f_tax_reset_index.groupby('Species').first()
f_tax_species = f_tax_species.drop(columns=['ASV'])
f_tax_species = f_tax_species.sort_index()
# 进行替换操作
f_tax_species['Phylum'] = f_tax_species['Phylum'].replace('p__unclassified_k__Fungi', 'Unclassified')
# 去除 'p__' 前缀
f_tax_species['Phylum'] = f_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
f_unique_species_names = f_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
f_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(f_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 f_species 的索引
f_species = f_species.rename(index= f_species_to_numeric_id)
# 转换 f_tax_species 的索引
f_tax_species = f_tax_species.rename(index= f_species_to_numeric_id)
f_species_df = f_species.copy()
print(len(f_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
f_species_rel = f_species_df.div(f_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
f_rel_long = f_species_rel.stack().reset_index()
f_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
f_rel_long['Origin'] = f_rel_long['Sample'].map(sample_origins_map)
f_rel_long['Niche'] = f_rel_long['Sample'].map(sample_niches_map)
f_species_total_counts = f_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
f_total_counts_across_all_species = f_species_total_counts.sum() # 计算所有 species 的总丰度
f_species_relative_abundance = f_species_total_counts / f_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
f_species_above_overall_threshold = f_species_relative_abundance[f_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
f_rel_long_filtered_by_overall_abundance = f_rel_long[f_rel_long['species'].isin(f_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
f_species_origin_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
f_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
f_species_in_origin = f_species_origin_mean_filtered[f_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
f_species_origin_count = f_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
f_species_niche_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
f_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
f_species_in_niche = f_species_niche_mean_filtered[f_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
f_species_niche_count = f_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
f_species_counts_combined = pd.merge(f_species_origin_count, f_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 3
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
f_core_species_names = f_species_counts_combined[
    (f_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (f_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
f_core_abundance = f_species_rel.loc[f_core_species_names]
f_core_species_raw_counts = f_species.loc[f_core_species_names]
print(len(f_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
f_median_relative_abundance = f_species_relative_abundance.median()
print(f_median_relative_abundance)
# 定义稀有类群的丰度阈值 
f_rare_threshold = 0.000008
# 筛选出稀有 species 
f_rare_species = f_species_relative_abundance[f_species_relative_abundance < f_rare_threshold]
# 获取稀有 species 的名称（索引）
f_rare_species_names = f_rare_species.index.tolist()
len(f_rare_species_names)
f_species_df_transposed = f_species_df.T 
f_speciesdfmelt = f_species_df_transposed.stack().reset_index()# 长宽表格转换
f_speciesdfmelt.columns = ['Sample', 'species', 'value']
f_speciesdfmelt = f_speciesdfmelt[f_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
f_control_species = f_speciesdfmelt.copy()
f_control_species['Group'] = f_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
f_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
f_controlspecies_unique = f_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
f_controlspecies_unique = f_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
f_controlspecies_unique = f_controlspecies_unique[f_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
f_unique_species_names_single_group = f_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
f_set_rare_species = set(f_rare_species_names)
f_set_unique_species_single_group = set(f_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
f_rare_unique_species_names = list(f_set_rare_species.intersection(f_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
f_rare_unique_abundance = f_species_df.loc[f_rare_unique_species_names]
f_rare_unique_species_raw_counts = f_species.loc[f_rare_unique_species_names]
print(len(f_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度置换检验与 Z-score 计算
num_permutations = 1000 
f_niche_breadth_results = classify_generalist_specialist(
    f_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (含 Z-score, 前5个 species) ")
print(f_niche_breadth_results.head())
f_generalists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
f_specialists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(f_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(f_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_generalists_species:
    print("无广适类群 species 数据。")
else:
    f_generalists_raw_counts = f_species.loc[f_generalists_species]
    f_generalists_abundance = f_species_rel.loc[f_generalists_species]
    print(f_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_specialists_species:
    print("无专适类群 species 数据。")
else:
    f_specialists_raw_counts = f_species.loc[f_specialists_species]
    f_specialists_abundance = f_species_rel.loc[f_specialists_species]
print(f_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

# 加载相对丰度、丰度物种、稀有物种、广适种和狭适种数据
f_df_OTU_rel = f_species_rel 
f_df_abun = f_core_abundance 
f_df_rare = f_rare_unique_abundance
f_df_gen = f_generalists_abundance
f_df_spe = f_specialists_abundance
# 加载分类学数据
f_otu_taxon = f_tax_species
# 从 otu_taxon 中删除不需要的分类学列，只保留 OTU ID 和 Phylum 相关的信息
# 此处drop了除Phylum之外的所有分类学层级和OTUs列
f_otu_network_color = f_otu_taxon.drop(["Kingdom", "Class", "Order", "Family", "Genus"], axis=1)
f_otu_network_color = f_otu_network_color.reset_index(drop=True)
# 为 DataFrame 创建一个新的 'id' 列，其值为当前索引
f_otu_network_color["id"] = f_otu_network_color.index
# 重新排序并只保留 'id' 和 'Phylum' 列
f_otu_network_color = f_otu_network_color[['id', 'Phylum']]
# 将 'id' 列的值加 1 (通常是为了使 ID 从 1 开始而不是从 0 开始，这在某些网络分析软件中很常见)
f_otu_network_color["id"] += 1
f_otu_network_color_phyla = f_otu_network_color.copy()
# 将结果保存为 Excel 文件，不包含索引
f_otu_network_color_phyla.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_otu_network_color_phyla.xlsx", index=False)
# 根据丰度 (abundance) 进行颜色编码
# 读取稀有化后的丰度数据 (或包含Ecotype信息的数据)
# 将 f_species 的索引 (species ID) 转换为一个新列
f_species_ecotype_df = f_species.reset_index()
f_species_ecotype_df = f_species_ecotype_df[['Species']].copy() # 在这里明确创建副本
# 添加 'Ecotype' 列并初始化为 'Others'
f_species_ecotype_df['Ecotype'] = 'Others'
# 根据 f_core_species_names 标记 'Cores' species
f_species_ecotype_df.loc[f_species_ecotype_df['Species'].isin(f_core_species_names), 'Ecotype'] = 'Cores'
# 根据 f_rare_unique_species_names 标记 'Uniques' species
f_species_ecotype_df.loc[f_species_ecotype_df['Species'].isin(f_rare_unique_species_names), 'Ecotype'] = 'Uniques'
f_otu_network_color = f_species_ecotype_df.copy()
# 为 DataFrame 创建一个新的 'id' 列，其值为当前索引
f_otu_network_color["id"] = f_otu_network_color.index
# 重新排序并只保留 'id' 和 'Ecotype' 列
f_otu_network_color = f_otu_network_color[['id', 'Ecotype']].copy()
# 将 'id' 列的值加 1
f_otu_network_color["id"] += 1
f_otu_network_color_abundance = f_otu_network_color.copy()
# 将结果保存为 Excel 文件，不包含索引
f_otu_network_color_abundance.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_otu_network_color_abundance.xlsx", index=False)
# 根据生态位宽度 (niche breadth) 进行颜色编码
# 读取表示基因-物种关系（或包含Ecotype信息）的数据
# 初始化一个新的 DataFrame，只包含 species ID，并作为独立副本
f_species_niche_ecotype_df = pd.DataFrame({'Species': f_species.index.tolist()}).copy()
# 添加 'Ecotype' 列并初始化为 'Others'
f_species_niche_ecotype_df['Ecotype'] = 'Others'
# 标记 'Generalists' species
# 使用 .loc[] 安全地进行条件赋值
f_species_niche_ecotype_df.loc[
    f_species_niche_ecotype_df['Species'].isin(f_generalists_species),
    'Ecotype'
] = 'Generalists'
# 标记 'Specialists' species
f_species_niche_ecotype_df.loc[
    (f_species_niche_ecotype_df['Species'].isin(f_specialists_species)),
    'Ecotype'
] = 'Specialists'
f_otu_network_color = f_species_niche_ecotype_df.copy()
# 为 DataFrame 创建一个新的 'id' 列，其值为当前索引
f_otu_network_color["id"] = f_otu_network_color.index
# 重新排序并只保留 'id' 和 'Ecotype' 列
f_otu_network_color = f_otu_network_color[['id', 'Ecotype']]
# 将 'id' 列的值加 1
f_otu_network_color["id"] += 1
f_otu_network_color_breadth = f_otu_network_color.copy()
# 将结果保存为 Excel 文件，不包含索引
f_otu_network_color_breadth.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_otu_network_color_breadth.xlsx", index=False)
# 处理所有 OTUs
# 读取 OTU 稀有化后的相对丰度数据，并将第一列设置为索引
f_otu_relative = f_species_rel.copy()
# 对 DataFrame 进行转置，即行和列互换
f_otu_relative = f_otu_relative.transpose()
# 将转置后的 DataFrame 保存为一个新的 CSV 文件，保留索引作为第一列
f_otu_relative.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_transposed_OTU_rar_relative.csv", index=True)
# 处理 OTUs 的子集用于测试
# 创建数据子集用于测试
# 读取转置后的 OTU 相对丰度数据
f_transposed_OTU_rar_relative = f_otu_relative.copy()
# 选取 DataFrame 的前 100 行和前 100 列作为子集
f_subset = f_transposed_OTU_rar_relative.iloc[:100, :100]
f_subset_transposed_OTU_rar_relative = f_subset.copy()
# 将子集保存为 CSV 文件，不包含索引
f_subset_transposed_OTU_rar_relative.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_subset_transposed_OTU_rar_relative.csv", index=False)
# 为测试目的填充颜色矩阵
# 读取 OTU 门类别的颜色编码文件
f_otu_network_color = f_otu_network_color_phyla.copy()
# 选取颜色矩阵的前 99 行作为子集
f_otu_network_color_subset = f_otu_network_color.iloc[:99,:]
f_subset_otu_network_color = f_otu_network_color_subset.copy()
# 将颜色矩阵子集保存为 Excel 文件，不包含索引
f_subset_otu_network_color.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_subset_otu_network_color.xlsx", index=False)
# OTU 网络度量函数
# 定义一个函数，用于解析网络分析结果并整理成 DataFrame
# result: 包含网络分析原始输出的字符串
# network_color_file: 包含 ASV id 和颜色信息的 DataFrame (通常是 ASV_network_color_phyla.xlsx 的内容)
def network_metrics(result, network_color_file):
    # 从 network_color_file 中提取 'id' 列并转换为列表
    network_color_list = network_color_file["id"].tolist()

    # 将原始结果字符串按行分割
    lines = result.split('\n')
    # 从第一行中提取边密度 (f_edge_density) 的浮点数值
    # 假设第一行格式为 '[1] 0.1234'，通过切片去除 '[1] '
    f_edge_density = float(lines[0][len('[1] '):])

    # 初始化空字典用于存储度 (degree)、接近中心性 (closeness) 和中介中心性 (betweenness)
    degree = {}
    closeness = {}
    betweenness = {}
    current_stage = '' # 用于标记当前正在解析哪种网络度量
    lines = lines[1:] # 跳过已经处理过的第一行 (边密度)
    a = iter(lines) # 创建一个迭代器，以便可以每次取两行
    # 遍历处理后的行，每次取两行 (line1 包含键，line2 包含值)
    for line1, line2 in zip(a, a):
        # 检查当前行是否包含网络度量类型的标识（'degree', 'closeness', 'betweenness'）
        if 'degree' in line1 or 'closeness' in line1 or 'betweenness' in line1:
            current_stage = line1.strip()  # 更新当前正在处理的度量类型，并去除空白
            continue # 跳过当前行，进入下一轮循环读取数据行
        keys = line1.split() # 将第一行按空格分割，得到键 (ASV ID)
        vals = line2.split() # 将第二行按空格分割，得到值 (度量数值)
        # 遍历键值对，将数据存入相应的字典中
        for k, v in zip(keys, vals):
            if 'degree' in current_stage:
                degree[k] = int(v) # 度为整数
            elif 'closeness' in current_stage:
                closeness[k] = float(v) # 接近中心性为浮点数
            elif 'betweenness' in current_stage:
                betweenness[k] = float(v) # 中介中心性为浮点数

    # 将所有键（ASV ID）转换为 'ASV' 格式，并在排序后添加前缀
    # 收集所有字典中的键，去重并转换为整数后排序 (确保按数字顺序排序)
    all_keys = sorted(set(degree.keys()).union(closeness.keys(), betweenness.keys()), key=int)
    # 为排序后的键添加 'ASV' 前缀
    all_keys = ["ASV" + k for k in all_keys]  # 在排序后添加 'ASV' 前缀

    # 准备用于创建 DataFrame 的数据
    df_data = {
        'id': network_color_list, # 使用传入的颜色文件中的 'id' 列表作为 ID
        # 获取 degree 值。由于 all_keys 带有 'ASV' 前缀，所以需要 k[3:] 去除前缀来匹配字典中的键。
        # 如果键不存在，则默认为 0。
        'degree': [degree.get(k[3:], 0) for k in all_keys],
        'closeness': [closeness.get(k[3:], 0) for k in all_keys],
        'betweenness': [betweenness.get(k[3:], 0) for k in all_keys]
    }
    # 使用准备好的数据创建 Pandas DataFrame
    f_df = pd.DataFrame(df_data)
# 返回边密度和包含网络度量值的 DataFrame
    return f_edge_density, f_df


# 用于测试的子集数据 (Subset for testing)
# 读取子集 OTU 网络颜色数据到 DataFrame
f_subset_otu_network_color_df = f_subset_otu_network_color.copy()
# 定义子集 OTU 网络颜色文件的路径
f_subset_otu_network_color = "/mnt/d/study/master/Main_Figure_tables/Figure_5/f_subset_otu_network_color.xlsx"
# 定义子集 OTU 稀有化相对丰度文件的路径
f_subset_otu_rar_relative = "/mnt/d/study/master/Main_Figure_tables/Figure_5/f_subset_transposed_OTU_rar_relative.csv"
# 生成网络度量指标 (Generate network metrics)
# 调用 R 脚本 'SpiecEasi_stage_python_phyla.R' 来生成网络度量。
# 传入稀有化相对丰度子集和网络颜色子集文件作为参数。
# universal_newlines=True: 将输出解释为文本。
f_subset_otu_network_metrics = subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_python_phyla.R', f_subset_otu_rar_relative, f_subset_otu_network_color],
    universal_newlines=True,
    stderr=subprocess.PIPE
)
# 调用自定义的 network_metrics 函数解析 R 脚本的输出，获取边密度和网络度量 DataFrame
f_edge_density, f_df = network_metrics(f_subset_otu_network_metrics, f_subset_otu_network_color_df)
# 打印边密度
print("F_edge_density: ", f_edge_density)
# 预期输出：F_edge_density:
# 绘制网络图 (Plot network)
# 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制网络图。
# 传入稀有化相对丰度子集、网络颜色子集文件，以及输出 PDF 和边权重 CSV 文件的路径。
subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R',
     f_subset_otu_rar_relative, f_subset_otu_network_color,
     '/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_SpiecEasi_subset.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_weight_subset.csv'],
    universal_newlines=True,
    stderr=subprocess.DEVNULL
)
# 预期输出：'null device \n          1 \n' (R 脚本的默认输出，表示绘图设备已激活)
# 处理所有 OTUs (All OTUs)
# 读取所有 OTU 网络颜色数据到 DataFrame (这里使用的是按门类的数据作为主颜色文件)
f_otu_network_color_df = f_otu_network_color_phyla.copy()
# 定义所有 OTU 网络颜色文件的路径 (按门类)
f_otu_network_color_phyla = "/mnt/d/study/master/Main_Figure_tables/Figure_5/f_otu_network_color_phyla.xlsx"
# 定义所有 OTU 网络颜色文件的路径 (按丰度/分组)
f_otu_network_color_abundance = "/mnt/d/study/master/Main_Figure_tables/Figure_5/f_otu_network_color_abundance.xlsx"
# 定义所有 OTU 网络颜色文件的路径 (按生态位宽度)
f_otu_network_color_breadth= "/mnt/d/study/master/Main_Figure_tables/Figure_5/f_otu_network_color_breadth.xlsx"
# 定义所有 OTU 稀有化相对丰度文件的路径
f_otu_rar_relative = "/mnt/d/study/master/Main_Figure_tables/Figure_5/f_transposed_OTU_rar_relative.csv"
# 生成网络度量指标 (Generate network metrics)
# 调用 R 脚本 'SpiecEasi_stage_python_phyla.R' 来生成所有 OTU 的网络度量。
# 传入所有 OTU 的稀有化相对丰度文件和按门类的颜色文件作为参数。
f_otu_network_metrics = subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_python_phyla.R', f_otu_rar_relative, f_otu_network_color_phyla], # 注意这里使用了f_otu_network_color_phyla
    universal_newlines=True,
    stderr=subprocess.PIPE
)
# 调用自定义的 network_metrics 函数解析 R 脚本的输出，获取边密度和网络度量 DataFrame
f_edge_density, f_df = network_metrics(f_otu_network_metrics, f_otu_network_color_df)
# 打印边密度
print("F_edge_density: ", f_edge_density)
# 预期输出：F_edge_density: 
# 将计算出的边密度保存为 CSV 文件
f_edge_density_df = pd.DataFrame(data={'edge density': [f_edge_density]})
f_edge_density_df.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_density_all_OTUs.csv", index=False)
# 将计算出的网络度量 DataFrame (df) 保存为 CSV 文件
f_df.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_metrics_all_OTUs.csv", index = False)
# 绘制网络图，按门类进行颜色编码 (Plot network color code by phyla)
# 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制网络图。
# 传入所有 OTU 的稀有化相对丰度文件、按门类的颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R',
     f_otu_rar_relative, f_otu_network_color_phyla,
     '/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_SpiecEasi_all_OTUs_phyla.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_weight_all_OTUs.csv'],
    universal_newlines=True,
    stderr=subprocess.PIPE
)
# 预期输出：'null device \n          1 \n'

# 绘制网络图，按丰度进行颜色编码 (Plot network color code by abundance)
# 调用 R 脚本 'SpiecEasi_stage_network_ecotypes.R' 来绘制网络图。
# 注意：这里调用的是 'SpiecEasi_stage_network_ecotypes.R' 脚本。
# 传入所有 OTU 的稀有化相对丰度文件、按丰度的颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_ecotypes.R',
     f_otu_rar_relative, f_otu_network_color_abundance,
     '/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_SpiecEasi_all_OTUs_abundance.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_weight_all_OTUs.csv'],
    universal_newlines=True,
    stderr=subprocess.DEVNULL
)
# 预期输出：'null device \n          1 \n'

# 绘制网络图，按生态位宽度进行颜色编码 (Plot network color code by niche breadth)
# 调用 R 脚本 'SpiecEasi_stage_network_ecotypes.R' 来绘制网络图。
# 传入所有 OTU 的稀有化相对丰度文件、按生态位宽度的颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
subprocess.check_output(
    ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_ecotypes.R',
     f_otu_rar_relative, f_otu_network_color_breadth,
     '/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_SpiecEasi_all_OTUs_breadth.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_weight_all_OTUs.csv'],
    universal_newlines=True,
    stderr=subprocess.DEVNULL
)
# 预期输出：'null device \n          1 \n'

# 根据需要过滤 OTU 以改善网络可视化效果
# 由于网络图过于混乱，通过以下两步过程减少用于分析的 OTU 数量。
# 1. 剔除在 X% 样本中存在的 OTU。
# 2. 剔除与所有其他 OTU 的 Spearman 相关性小于 y 的 OTU。
# # 读取原始的转置 OTU 稀有化相对丰度数据
# f_df = pd.read_csv('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_transposed_OTU_rar_relative.csv', index_col=0)
# # 将 DataFrame 中的所有 0 值替换为 NaN (这样在计数时不会被计算，方便统计存在性)
# f_df.replace(0, np.nan, inplace=True)
# # 第一步：根据 OTU 在样本中的存在比例进行过滤
# # 计算阈值：0.05（5%）乘以总样本数 618，向下取整。
# threshold = math.floor(0.05*618)
# columns_to_drop_1 = [] # 初始化列表，用于存储要剔除的列名
# for col in f_df.columns: # 遍历 DataFrame 的每一列 (即每一个 OTU)
#     # 计算非零条目（非 NaN 值）的数量
#     non_zero_count = f_df[col].count()
#     # 如果非零条目的数量小于阈值，则将该列标记为待剔除
#     if non_zero_count < threshold:
#         columns_to_drop_1.append(col)
# # 打印根据存在性阈值需要剔除的列数
# print("Fol threshold: ", threshold,", the number of cols to be dropped is ", len(columns_to_drop_1))
# # 重新读取原始数据，因为之前的操作替换了 0 为 NaN
# f_df = pd.read_csv('/mnt/d/study/master/Main_Figure_tables/Figure_5/transposed_OTU_rar_relative.csv', index_col=0)
# # 剔除第一步中标记的列
# f_df = f_df.drop(columns=columns_to_drop_1)
# # 第二步：根据 OTU 间的 Spearman 相关性进行过滤
# # 计算 DataFrame 中所有列之间的 Spearman 相关性矩阵
# correlation_matrix = f_df.corr(method='spearman')
# threshold = 0.4 # 设定相关性阈值
# tolerance_list = [0] # 允许具有高相关性的 OTU 数量 (这里是 0，表示如果一个 OTU 没有任何与它高相关的其他 OTU，则被剔除)
# for tolerance in tolerance_list: # 遍历容忍度列表 (这里只有一个值 0)
#     columns_to_drop_2 = [] # 初始化列表，用于存储第二步要剔除的列名
#     for col in correlation_matrix.columns: # 遍历相关性矩阵的每一列 (即每一个 OTU)
#         # 计算与当前 OTU 相关性绝对值大于阈值的其他 OTU 的数量
#         # .drop(col) 是为了排除自身与自身的相关性 (总是 1)
#         high_corr_count = (correlation_matrix[col].drop(col).abs() > threshold).sum()
#         # 如果高相关性 OTU 的数量小于或等于容忍度，则标记该列为待剔除
#         if high_corr_count <= tolerance:
#             columns_to_drop_2.append(col)
#     # 打印根据相关性阈值和容忍度需要剔除的列数
#     print("Fol tolerance: ", tolerance,", the number of cols to be dropped is ", len(columns_to_drop_2))
# # 剔除第二步中标记的列
# f_df = f_df.drop(columns=columns_to_drop_2)
# # 将经过两步过滤后的 DataFrame 保存为新的 CSV 文件
# f_df.to_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/less_than_5_and_4_OTU_data.csv")
# 上述被注释代码的运行结果摘要：
# 表示第一步过滤后，如果 OTU 在少于 30 个样本中存在，将剔除 1974 个 OTU。
print("Fol threshold:  30 , the number of cols to be dropped is  1974")
# 表示第二步过滤后，如果一个 OTU 与任何其他 OTU 的 Spearman 相关性低于 0.4 且高相关性 OTU 数量小于等于 0，将剔除 405 个 OTU。
print("Fol tolerance:  0 , the number of cols to be dropped is  405")
# # (被注释掉的代码块 - 以下代码用于更新颜色文件)
# # 从颜色 CSV 文件中剔除相同的 OTU
# # 合并两步过滤后需要剔除的 OTU 列名列表
# columns_to_drop = columns_to_drop_1 + columns_to_drop_2
# # 将要剔除的 OTU 列名转换为整数 ID (假设列名格式为 "OTU_ID")
# ids_to_drop = [int(col[4:]) for col in columns_to_drop]
# # 读取原始的 OTU 网络颜色文件 (按门类)
# otu_network_color = pd.read_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/otu_network_color_phyla.xlsx")
# # 根据需要剔除的 ID 过滤颜色 DataFrame
# f_df_color_filtered = otu_network_color[~otu_network_color['id'].isin(ids_to_drop)]
# # 将过滤后的颜色 DataFrame 保存为新的 Excel 文件
# f_df_color_filtered.to_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/less_than_5_and_4_otu_network_color.xlsx", index=False)
# # (被注释掉的代码块 - 以下代码用于绘制过滤后的网络图)
# # 加载用于网络构建的输入文件
# # 定义过滤后的 OTU 网络颜色文件路径
# filtered_otu_network_color = "/mnt/d/study/master/Main_Figure_tables/Figure_5/less_than_5_and_4_otu_network_color.xlsx"
# # 定义过滤后的 OTU 稀有化相对丰度文件路径
# filtered_otu_rar_relative = "/mnt/d/study/master/Main_Figure_tables/Figure_5/less_than_5_and_4_OTU_data.csv"
# # 绘制网络图
# # 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制过滤后的网络图。
# # 传入过滤后的数据文件、颜色文件以及输出 PDF 和边权重 CSV 文件的路径。
# subprocess.check_output(['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R',
#                          filtered_otu_rar_relative, filtered_otu_network_color,
#                          '/mnt/d/study/master/Main_Figure_tables/Figure_5/network_SpiecEasi_filtered_OTUs.pdf','/mnt/d/study/master/Main_Figure_tables/Figure_5/edge_weight_filtered_OTUs.csv'], universal_newlines=True, stderr=subprocess.DEVNULL)
# # 预期输出：'null device \n          1 \n' (R 脚本的默认输出，表示绘图设备已激活)

# 为居群构建网络 (Network for ecosystems)
f_ecosystem = metadata[['Sample ID', 'Origin']]
# 读取转置后的 OTU 稀有化相对丰度数据
f_transposed_OTU_rar_relative = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_transposed_OTU_rar_relative.csv")
# 合并数据以获取所有样本的生态系统类别
# 将 ecosystem DataFrame 与 transposed_OTU_rar_relative DataFrame 合并。
# 使用 "Sample ID " 和 "Unnamed: 0" 列进行连接。
f_merged_data = pd.merge(f_ecosystem, f_transposed_OTU_rar_relative, left_on="Sample ID", right_on="Unnamed: 0")
# 为每个生态系统类别创建单独的 DataFrame
# 获取 'Origin' 列中所有独特的生态系统类别
f_ecosystem_classes = f_merged_data['Origin'].unique()
# 使用字典推导式为每个生态系统类别创建一个 DataFrame，并删除原始的生态系统编码和样本 ID 列
f_ecosystems_dfs = {f_ecosystem: f_merged_data[f_merged_data['Origin'] == f_ecosystem].drop(columns=['Origin', 'Sample ID']) for f_ecosystem in f_ecosystem_classes}
# 显示每个类别的初始样本数量
f_initial_counts = {f_ecosystem: len(f_df) for f_ecosystem, f_df in f_ecosystems_dfs.items()}
print("Initial Sample Counts:", f_initial_counts)
# 预期输出：Initial Sample Counts: 
# 找到所有 DataFrame 中最小的样本数量
f_min_sample_count = min(f_initial_counts.values())
# 打印最小样本数量
print(f_min_sample_count)
# 预期输出：
# 随机剔除样本，使所有生态系统类别的样本数量与最小数量相等。
f_equalized_dfs = {} # 初始化字典，用于存储样本数量均等化后的 DataFrame
for f_ecosystem, f_df in f_ecosystems_dfs.items():
    if len(f_df) > f_min_sample_count:
        # 如果当前生态系统的样本数量大于最小数量，则随机抽样，使数量等于最小数量
        # random_state=42 保证结果可复现
        f_equalized_dfs[f_ecosystem] = f_df.sample(n= f_min_sample_count, random_state=42)
    else:
        # 否则，直接使用原始 DataFrame (因为它的样本数量已经等于或小于最小数量)
        f_equalized_dfs[f_ecosystem] = f_df
# 移除每个 DataFrame 中所有样本中出现次数为零的 OTU
f_final_dfs = {} # 初始化字典，用于存储最终处理后的 DataFrame
f_final_otu_counts = {} # 初始化字典，用于存储最终的 OTU 数量
for f_ecosystem, f_df in f_equalized_dfs.items():
    # 筛选列：只保留至少在一个样本中存在（非零）的 OTU (列)
    f_df = f_df.loc[:, (f_df != 0).any(axis=0)]
    f_final_dfs[f_ecosystem] = f_df
    # 记录每个生态系统处理后的 OTU 数量
    f_final_otu_counts[f_ecosystem] = f_df.shape[1]
# 显示最终 DataFrame 结构和样本数量
f_final_counts = {f_ecosystem: len(f_df) for f_ecosystem, f_df in f_final_dfs.items()}
print("Final Sample Counts:", f_final_counts)
print("Final OTU Counts:", f_final_otu_counts)
# 预期输出：
# Final Sample Counts:
# Final OTU Counts: 
# 由于移除了在所有样本中出现次数为零的 OTU 后，OTU 数量仍然很高
# 移除在少于或等于阈值数量样本中存在的 OTU
f_final_dfs = {} # 重新初始化，因为下面会根据新的阈值再次过滤
f_otu_presence_counts = {} # 初始化字典，用于存储 OTU 在不同阈值下的存在计数
f_thresholds = [0, 1, 2, 3, 4, 5] # 定义要测试的样本存在阈值列表
for f_ecosystem, f_df in f_equalized_dfs.items():
    # 再次筛选列：只保留至少在一个样本中存在（非零）的 OTU (列)
    f_df = f_df.loc[:, (f_df != 0).any(axis=0)]
    f_final_dfs[f_ecosystem] = f_df
    f_num_samples = len(f_df) # 当前生态系统的样本数量
    f_otu_presence_counts[f_ecosystem] = {} # 初始化当前生态系统的 OTU 存在计数字典
    for f_threshold in f_thresholds: # 遍历每个阈值
        # 计算每个 OTU 在超过阈值数量样本中存在的 OTU 数量
        f_count = (f_df != 0).sum(axis=0) > f_threshold
        f_otu_presence_counts[f_ecosystem][f"{int(f_threshold)} count"] = f_count.sum()
# 显示最终的 DataFrame 结构和 OTU 非零计数
print("OTU Presence with non zero Counts:")
for f_ecosystem, f_counts in f_otu_presence_counts.items():
    print(f"{f_ecosystem}: {f_counts}")
# 预期输出：
# OTU Presence with non zero Counts:

# 最终的计数表明，阈值设置为 0 可以得到一个可管理的 OTU 数量。
# 最终过滤 OTU 并为每个生态系统创建文件和目录
f_final_dfs = {}
f_otu_presence_counts = {}
f_ecosystem_directories = [] # 列表用于存储已创建的生态系统目录路径
f_threshold = 1  # 设置感兴趣的阈值 (OTU 至少在 1 个样本中出现)
for f_ecosystem, f_df in f_equalized_dfs.items():
    if f_ecosystem == "Unknown": # 跳过 'Unknown' 生态系统
        continue
    # 再次过滤掉在所有样本中出现次数为零的 OTU
    f_df = f_df.loc[:, (f_df != 0).any(axis=0)]
    f_final_dfs[f_ecosystem] = f_df
    # 计算每个 OTU 在超过阈值数量样本中存在的 OTU 数量
    f_presence_count = (f_df != 0).sum(axis=0) > f_threshold
    f_otu_presence_counts[f_ecosystem] = f_presence_count.sum() # 存储满足阈值的 OTU 数量
    # 检查生态系统对应的目录是否存在，如果不存在则创建
    directory = f'/mnt/d/study/master/Main_Figure_tables/Figure_5/{f_ecosystem}/'
    if not os.path.exists(directory):
        os.makedirs(directory)
    f_ecosystem_directories.append(directory) # 将新创建的目录添加到列表中
    # 过滤 DataFrame，只保留在超过阈值数量样本中存在的 OTU
    f_filtered_df = f_df.loc[:, (f_df != 0).sum(axis=0) > f_threshold]
    # 将过滤后的 DataFrame 保存为 CSV 文件，不包含索引
    f_filtered_df.to_csv(f'{directory}f_filtered_otus.csv', index = False)
# 显示满足阈值  的 OTU 数量
print("OTU Presence with non-zero counts above the threshold of 0:")
for f_ecosystem, f_count in f_otu_presence_counts.items():
    print(f"{f_ecosystem}: {f_count}")
# 预期输出：
# OTU Presence with non-zero counts above the threshold of 1:

# 修复网络颜色文件 (针对每个生态系统)
for directory in f_ecosystem_directories: # 遍历每个生态系统的目录
    # 重新读取原始的 OTU 网络颜色文件 (按门类)
    f_otu_network_color = pd.read_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_otu_network_color_phyla.xlsx")
    # 读取当前过滤后的 OTU 文件 (这个文件中的列名是 OTU IDs)
    f_df_otu = pd.read_csv(f"{directory}f_filtered_otus.csv")
    # 从列名中提取 OTU ID
    # 假设第一列是样本 ID，所以从第二列开始才是 OTU ID。
    # 提取 OTU ID 的数字部分并转换为整数。
    f_otus = f_df_otu.columns.tolist()[1:]
    f_otu_ids = [int(f_otu.split('_')[1]) for f_otu in f_otus] # 假设 OTU_ID 格式为 "OTU_数字"
    # 过滤颜色数据，只保留 'id' 列与当前文件中 OTU ID 匹配的行
    f_df_color_filtered = f_otu_network_color[f_otu_network_color['id'].isin(f_otu_ids)]
    # 将过滤后的颜色数据保存为 Excel 文件
    f_df_color_filtered.to_excel(f'{directory}f_filtered_network_color.xlsx', index=False)

# 这个函数将使用 SpiecEasi_stage_python.R 文件找到网络度量指标。
# 这个函数使用 OS subprocess 来调用 R 脚本。R 脚本必须与这个 Python 文件在同一个文件夹中。
for directory in f_ecosystem_directories: # 遍历每个生态系统的目录
    # 构建输入和输出文件的完整路径
    f_input_csv_file_name = f'{directory}f_filtered_otus.csv'
    f_color_file_name = f'{directory}f_filtered_network_color.xlsx'
    f_pdf_file_name = f'{directory}f_network_SpiecEasi.pdf'
    f_edge_weight_file_name = f'{directory}f_edge_weight.csv'
    # 读取颜色文件以便在 network_metrics 函数中使用
    f_color_file = pd.read_excel(f_color_file_name)
    # 调用 R 脚本 'SpiecEasi_stage_python_phyla.R' 来计算网络度量。
    # 传入过滤后的 OTU 数据和对应的颜色文件。
    f_result_tf_obs_otu_full = subprocess.check_output(
        ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_python_phyla.R', f_input_csv_file_name, f_color_file_name],
        universal_newlines=True,
        stderr=subprocess.DEVNULL
    )
    # 使用 network_metrics 函数解析 R 脚本的输出，获取边密度和网络度量 DataFrame
    f_edge_density, f_df = network_metrics(f_result_tf_obs_otu_full, f_color_file)
    # 打印当前文件的信息
    print(f"For the data in {f_input_csv_file_name} file")
    # 打印边密度
    print("F_edge_density: ", f_edge_density)
    # 打印度、接近中心性和中介中心性的结果
    print("Degree, closeness, betweeness results are as follows: ")
    print(f_df)
    # 将边密度保存为 CSV 文件
    f_edge_density_df = pd.DataFrame([f_edge_density], columns=["edge density"])
    f_edge_density_df.to_csv(f'{directory}f_edge_density.csv', index = False)
    # 将包含网络度量值的 DataFrame 保存为 CSV 文件
    f_df.to_csv(f'{directory}f_network_metrics.csv', index=False)
    # 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制网络图。
    # 传入过滤后的 OTU 数据、颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
    subprocess.check_output(
        ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R', f_input_csv_file_name, f_color_file_name, f_pdf_file_name, f_edge_weight_file_name],
        universal_newlines=True,
        stderr=subprocess.DEVNULL
    )

# 为生态位构建网络 (Network for ecosystems)
f_ecosystem = metadata[['Sample ID', 'Niche']]
# 读取转置后的 OTU 稀有化相对丰度数据
f_transposed_OTU_rar_relative = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_transposed_OTU_rar_relative.csv")
# 合并数据以获取所有样本的生态系统类别
# 将 ecosystem DataFrame 与 transposed_OTU_rar_relative DataFrame 合并。
# 使用 "Sample ID " 和 "Unnamed: 0" 列进行连接。
f_merged_data = pd.merge(f_ecosystem, f_transposed_OTU_rar_relative, left_on="Sample ID", right_on="Unnamed: 0")
# 为每个生态系统类别创建单独的 DataFrame
# 获取 'Niche' 列中所有独特的生态系统类别
f_ecosystem_classes = f_merged_data['Niche'].unique()
# 使用字典推导式为每个生态系统类别创建一个 DataFrame，并删除原始的生态系统编码和样本 ID 列
f_ecosystems_dfs = {f_ecosystem: f_merged_data[f_merged_data['Niche'] == f_ecosystem].drop(columns=['Niche', 'Sample ID']) for f_ecosystem in f_ecosystem_classes}
# 显示每个类别的初始样本数量
f_initial_counts = {f_ecosystem: len(f_df) for f_ecosystem, f_df in f_ecosystems_dfs.items()}
print("Initial Sample Counts:", f_initial_counts)
# 预期输出：Initial Sample Counts: 
# 找到所有 DataFrame 中最小的样本数量
f_min_sample_count = min(f_initial_counts.values())
# 打印最小样本数量
print(f_min_sample_count)
# 预期输出：
# 随机剔除样本，使所有生态系统类别的样本数量与最小数量相等。
f_equalized_dfs = {} # 初始化字典，用于存储样本数量均等化后的 DataFrame
for f_ecosystem, f_df in f_ecosystems_dfs.items():
    if len(f_df) > f_min_sample_count:
        # 如果当前生态系统的样本数量大于最小数量，则随机抽样，使数量等于最小数量
        # random_state=42 保证结果可复现
        f_equalized_dfs[f_ecosystem] = f_df.sample(n= f_min_sample_count, random_state=42)
    else:
        # 否则，直接使用原始 DataFrame (因为它的样本数量已经等于或小于最小数量)
        f_equalized_dfs[f_ecosystem] = f_df
# 移除每个 DataFrame 中所有样本中出现次数为零的 OTU
f_final_dfs = {} # 初始化字典，用于存储最终处理后的 DataFrame
f_final_otu_counts = {} # 初始化字典，用于存储最终的 OTU 数量
for f_ecosystem, f_df in f_equalized_dfs.items():
    # 筛选列：只保留至少在一个样本中存在（非零）的 OTU (列)
    f_df = f_df.loc[:, (f_df != 0).any(axis=0)]
    f_final_dfs[f_ecosystem] = f_df
    # 记录每个生态系统处理后的 OTU 数量
    f_final_otu_counts[f_ecosystem] = f_df.shape[1]
# 显示最终 DataFrame 结构和样本数量
f_final_counts = {f_ecosystem: len(f_df) for f_ecosystem, f_df in f_final_dfs.items()}
print("Final Sample Counts:", f_final_counts)
print("Final OTU Counts:", f_final_otu_counts)
# 预期输出：
# Final Sample Counts:
# Final OTU Counts: 
# 由于移除了在所有样本中出现次数为零的 OTU 后，OTU 数量仍然很高
# 移除在少于或等于阈值数量样本中存在的 OTU
f_final_dfs = {} # 重新初始化，因为下面会根据新的阈值再次过滤
f_otu_presence_counts = {} # 初始化字典，用于存储 OTU 在不同阈值下的存在计数
f_thresholds = [0, 1, 2, 3, 4, 5] # 定义要测试的样本存在阈值列表
for f_ecosystem, f_df in f_equalized_dfs.items():
    # 再次筛选列：只保留至少在一个样本中存在（非零）的 OTU (列)
    f_df = f_df.loc[:, (f_df != 0).any(axis=0)]
    f_final_dfs[f_ecosystem] = f_df
    f_num_samples = len(f_df) # 当前生态系统的样本数量
    f_otu_presence_counts[f_ecosystem] = {} # 初始化当前生态系统的 OTU 存在计数字典
    for f_threshold in f_thresholds: # 遍历每个阈值
        # 计算每个 OTU 在超过阈值数量样本中存在的 OTU 数量
        f_count = (f_df != 0).sum(axis=0) > f_threshold
        f_otu_presence_counts[f_ecosystem][f"{int(f_threshold)} count"] = f_count.sum()
# 显示最终的 DataFrame 结构和 OTU 非零计数
print("OTU Presence with non zero Counts:")
for f_ecosystem, f_counts in f_otu_presence_counts.items():
    print(f"{f_ecosystem}: {f_counts}")
# 预期输出：
# OTU Presence with non zero Counts:

# 最终的计数表明，阈值设置为 0 可以得到一个可管理的 OTU 数量。
# 最终过滤 OTU 并为每个生态系统创建文件和目录
f_final_dfs = {}
f_otu_presence_counts = {}
f_ecosystem_directories = [] # 列表用于存储已创建的生态系统目录路径
f_threshold = 1  # 设置感兴趣的阈值 (OTU 至少在 1 个样本中出现)
for f_ecosystem, f_df in f_equalized_dfs.items():
    if f_ecosystem == "Unknown": # 跳过 'Unknown' 生态系统
        continue
    # 再次过滤掉在所有样本中出现次数为零的 OTU
    f_df = f_df.loc[:, (f_df != 0).any(axis=0)]
    f_final_dfs[f_ecosystem] = f_df
    # 计算每个 OTU 在超过阈值数量样本中存在的 OTU 数量
    f_presence_count = (f_df != 0).sum(axis=0) > f_threshold
    f_otu_presence_counts[f_ecosystem] = f_presence_count.sum() # 存储满足阈值的 OTU 数量
    # 检查生态系统对应的目录是否存在，如果不存在则创建
    directory = f'/mnt/d/study/master/Main_Figure_tables/Figure_5/{f_ecosystem}/'
    if not os.path.exists(directory):
        os.makedirs(directory)
    f_ecosystem_directories.append(directory) # 将新创建的目录添加到列表中
    # 过滤 DataFrame，只保留在超过阈值数量样本中存在的 OTU
    f_filtered_df = f_df.loc[:, (f_df != 0).sum(axis=0) > f_threshold]
    # 将过滤后的 DataFrame 保存为 CSV 文件，不包含索引
    f_filtered_df.to_csv(f'{directory}f_filtered_otus.csv', index = False)
# 显示满足阈值  的 OTU 数量
print("OTU Presence with non-zero counts above the threshold of 0:")
for f_ecosystem, f_count in f_otu_presence_counts.items():
    print(f"{f_ecosystem}: {f_count}")
# 预期输出：
# OTU Presence with non-zero counts above the threshold of 1:

# 修复网络颜色文件 (针对每个生态系统)
for directory in f_ecosystem_directories: # 遍历每个生态系统的目录
    # 重新读取原始的 OTU 网络颜色文件 (按门类)
    f_otu_network_color = pd.read_excel("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_otu_network_color_phyla.xlsx")
    # 读取当前过滤后的 OTU 文件 (这个文件中的列名是 OTU IDs)
    f_df_otu = pd.read_csv(f"{directory}f_filtered_otus.csv")
    # 从列名中提取 OTU ID
    # 假设第一列是样本 ID，所以从第二列开始才是 OTU ID。
    # 提取 OTU ID 的数字部分并转换为整数。
    f_otus = f_df_otu.columns.tolist()[1:]
    f_otu_ids = [int(f_otu.split('_')[1]) for f_otu in f_otus] # 假设 OTU_ID 格式为 "OTU_数字"
    # 过滤颜色数据，只保留 'id' 列与当前文件中 OTU ID 匹配的行
    f_df_color_filtered = f_otu_network_color[f_otu_network_color['id'].isin(f_otu_ids)]
    # 将过滤后的颜色数据保存为 Excel 文件
    f_df_color_filtered.to_excel(f'{directory}f_filtered_network_color.xlsx', index=False)

# 这个函数将使用 SpiecEasi_stage_python.R 文件找到网络度量指标。
# 这个函数使用 OS subprocess 来调用 R 脚本。R 脚本必须与这个 Python 文件在同一个文件夹中。
for directory in f_ecosystem_directories: # 遍历每个生态系统的目录
    # 构建输入和输出文件的完整路径
    f_input_csv_file_name = f'{directory}f_filtered_otus.csv'
    f_color_file_name = f'{directory}f_filtered_network_color.xlsx'
    f_pdf_file_name = f'{directory}f_network_SpiecEasi.pdf'
    f_edge_weight_file_name = f'{directory}f_edge_weight.csv'
    # 读取颜色文件以便在 network_metrics 函数中使用
    f_color_file = pd.read_excel(f_color_file_name)
    # 调用 R 脚本 'SpiecEasi_stage_python_phyla.R' 来计算网络度量。
    # 传入过滤后的 OTU 数据和对应的颜色文件。
    f_result_tf_obs_otu_full = subprocess.check_output(
        ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_python_phyla.R', f_input_csv_file_name, f_color_file_name],
        universal_newlines=True,
        stderr=subprocess.DEVNULL
    )
    # 使用 network_metrics 函数解析 R 脚本的输出，获取边密度和网络度量 DataFrame
    f_edge_density, f_df = network_metrics(f_result_tf_obs_otu_full, f_color_file)
    # 打印当前文件的信息
    print(f"For the data in {f_input_csv_file_name} file")
    # 打印边密度
    print("F_edge_density: ", f_edge_density)
    # 打印度、接近中心性和中介中心性的结果
    print("Degree, closeness, betweeness results are as follows: ")
    print(f_df)
    # 将边密度保存为 CSV 文件
    f_edge_density_df = pd.DataFrame([f_edge_density], columns=["edge density"])
    f_edge_density_df.to_csv(f'{directory}f_edge_density.csv', index = False)
    # 将包含网络度量值的 DataFrame 保存为 CSV 文件
    f_df.to_csv(f'{directory}f_network_metrics.csv', index=False)
    # 调用 R 脚本 'SpiecEasi_stage_network_phyla.R' 来绘制网络图。
    # 传入过滤后的 OTU 数据、颜色文件，以及输出 PDF 和边权重 CSV 文件的路径。
    subprocess.check_output(
        ['/usr/bin/Rscript', '/mnt/d/study/master/Main_Figure_tables/Figure_5/SpiecEasi_stage_network_phyla.R', f_input_csv_file_name, f_color_file_name, f_pdf_file_name, f_edge_weight_file_name],
        universal_newlines=True,
        stderr=subprocess.DEVNULL
    )




#图5网络其他
import pandas as pd
import os # 用于操作系统相关功能
import scipy.stats as st # 用于统计检验
import numpy as np # 用于数值计算
import statsmodels.stats.multitest as smt # 用于多重检验校正
import seaborn as sns # 用于绘制统计图形
import matplotlib.pyplot as plt # 用于创建可视化
from collections import defaultdict # 用于创建具有默认值的字典
from scipy.stats import kruskal # 科鲁斯卡尔-沃利斯检验
from statannotations.Annotator import Annotator # 用于在 seaborn 图上添加统计显著性注释
#  辅助函数：计算生态位宽度 (Bj) 
def calculate_levins_bj(df):
    """
    计算每个物种的 Levin's 生态位宽度 Bj。
    
    公式: Bj = 1 / sum(Pij^2)
    其中 Pij 为物种 j 在样本 i 中的相对丰度。
    
    参数:
        df (pd.DataFrame): 行 = OTU/物种, 列 = 样本, 值为相对丰度
    
    返回:
        pd.Series: 每个物种对应的 Bj 值
    """
    matrix = df.astype(float)
    
    # 对每个物种（行），跨所有样本计算 Pij² 的加和
    sum_sq = (matrix ** 2).sum(axis=1)
    
    # 避免除以零，将0替换为NaN后再填回0
    bj = 1.0 / sum_sq.replace(0, np.nan)
    
    return bj.fillna(0)


def classify_generalist_specialist(
    relative_abundance_matrix,
    num_permutations=1000,
    seed=42
):
    """
    基于生态位宽度（Levin's Bj）的置换检验，将物种分类为：
    广适种（Generalist）、专适种（Specialist）或中性种（Neutral）。
    
    分类标准：
        Generalist : 观测 Bj 显著大于零分布（p_generalist < 0.05）
        Specialist : 观测 Bj 显著小于零分布（p_specialist < 0.05）
        Neutral    : 其余物种
    
    参数:
        relative_abundance_matrix (pd.DataFrame):
            行 = OTU/物种, 列 = 样本/生境, 值为相对丰度
        num_permutations (int):
            置换次数，默认 1000 次
        seed (int):
            随机种子，保证结果可重复，默认 42
    
    返回:
        pd.DataFrame: index = 物种名，包含以下列：
            - Bj_Observed  : 观测生态位宽度
            - Bj_Expected  : 零分布的平均生态位宽度
            - Bj_Std       : 零分布的标准差
            - Z_Score      : 标准化得分，衡量偏离零分布的程度
            - p_generalist : 右尾 p 值（观测值是否显著偏大）
            - p_specialist : 左尾 p 值（观测值是否显著偏小）
            - Ecotype      : 分类结果（Generalist / Specialist / Neutral）
    """
    # ---------- 1. 数据准备 ----------
    mat = relative_abundance_matrix.fillna(0).astype(float)
    species = mat.index
    n_species, n_samples = mat.shape
    
    print(f"输入矩阵维度: {n_species} 个物种 × {n_samples} 个样本")

    # ---------- 2. 计算观测 Bj ----------
    observed_bj = calculate_levins_bj(mat)

    # ---------- 3. 置换检验，构建零分布 ----------
    # 使用独立随机生成器，避免影响全局随机状态
    rng = np.random.default_rng(seed=seed)
    
    # 预分配结果数组：行 = 物种，列 = 每次置换的 Bj
    permuted_bj_array = np.zeros((n_species, num_permutations), dtype=float)

    for i in range(num_permutations):
        # 将整个相对丰度矩阵展平后随机打乱
        # 目的：彻底破坏物种与样本之间的对应关系，生成零假设下的分布
        shuffled_flat = mat.values.flatten().copy()
        rng.shuffle(shuffled_flat)
        
        # 重塑回原始矩阵形状，计算该置换下每个物种的 Bj
        shuffled_df = pd.DataFrame(
            shuffled_flat.reshape(n_species, n_samples),
            index=species,
            columns=mat.columns
        )
        permuted_bj_array[:, i] = calculate_levins_bj(shuffled_df).values

    # ---------- 4. 零分布统计量 ----------
    # 零分布的均值：代表随机期望下的 Bj
    expected_bj = permuted_bj_array.mean(axis=1)
    
    # 零分布的标准差（使用无偏估计 ddof=1）
    std_null = permuted_bj_array.std(axis=1, ddof=1)
    
    # Z-score：观测值偏离零分布均值的标准化程度
    # 标准差为 0 时（所有置换结果相同）设为 NaN，避免除以零
    z_score = (observed_bj.values - expected_bj) / np.where(
        std_null == 0, np.nan, std_null
    )

    # ---------- 5. 计算置换 p 值 ----------
    # 使用 +1 修正（Phipson & Smyth 2010），避免 p 值为 0 的情况
    obs_val = observed_bj.values.reshape(-1, 1)
    
    # 右尾 p 值：零分布中 Bj >= 观测值的比例
    # p 值越小，说明观测 Bj 显著偏大 → 倾向于 Generalist
    p_gen = (
        (permuted_bj_array >= obs_val).sum(axis=1) + 1
    ) / (num_permutations + 1)
    
    # 左尾 p 值：零分布中 Bj <= 观测值的比例
    # p 值越小，说明观测 Bj 显著偏小 → 倾向于 Specialist
    p_spe = (
        (permuted_bj_array <= obs_val).sum(axis=1) + 1
    ) / (num_permutations + 1)

    # ---------- 6. 汇总结果 ----------
    results = pd.DataFrame({
        'Bj_Observed' : observed_bj.values,
        'Bj_Expected' : expected_bj,
        'Bj_Std'      : std_null,
        'Z_Score'     : z_score,
        'p_generalist': p_gen,
        'p_specialist': p_spe,
    }, index=species)

    # ---------- 7. 物种分类 ----------
    results['Ecotype'] = 'Neutral'
    results.loc[results['p_generalist'] < 0.05, 'Ecotype'] = 'Generalist'
    results.loc[results['p_specialist'] < 0.05, 'Ecotype'] = 'Specialist'

    # ---------- 8. 输出统计摘要 ----------
    print("\n物种分类统计:")
    print(results['Ecotype'].value_counts().to_string())

    return results



# 打印当前 seaborn 版本，确认环境配置
print(sns.__version__)
# 输出示例: 0.11.2 (表示当前运行环境中的 seaborn 版本)
#细菌共线性其他
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
bacteria_ASV = pd.read_csv("/mnt/d/study/master/meiji/bacteria_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 bacteria_ASV DataFrame 中选择指定的列
b_ASV = bacteria_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
b_ASV.set_index(keys="ASV", inplace=True)
b_tax = bacteria_ASV.iloc[:, 1:9].copy()
b_tax.set_index(keys="ASV", inplace=True)
print(len(b_ASV))
# 计算 ASV 相对丰度
b_ASV_df = b_ASV.copy()

#按种
b_species = b_ASV.join(b_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
b_species = b_species.groupby('Species')[sample_ids].sum()
#排序
b_species = b_species.sort_index()
b_species_df = b_species.copy()
print(len(b_species_df))
# 将当前的 ASV 索引重置为一个普通列。
b_tax_reset_index = b_tax.reset_index()
# 将 'Species' 列设置为新的索引。
b_tax_species = b_tax_reset_index.groupby('Species').first()
b_tax_species = b_tax_species.drop(columns=['ASV'])
b_tax_species = b_tax_species.sort_index()
# 进行替换操作
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__unclassified_k__norank_d__Bacteria', 'Unclassified')
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__SAR324_cladeMarine_group_B', 'SAR324')
# 去除 'p__' 前缀
b_tax_species['Phylum'] = b_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
b_unique_species_names = b_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
b_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(b_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 b_species 的索引
b_species = b_species.rename(index= b_species_to_numeric_id)
# 转换 b_tax_species 的索引
b_tax_species = b_tax_species.rename(index= b_species_to_numeric_id)
b_species_df = b_species.copy()
print(len(b_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
b_species_rel = b_species_df.div(b_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
b_rel_long = b_species_rel.stack().reset_index()
b_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
b_rel_long['Origin'] = b_rel_long['Sample'].map(sample_origins_map)
b_rel_long['Niche'] = b_rel_long['Sample'].map(sample_niches_map)
b_species_total_counts = b_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
b_total_counts_across_all_species = b_species_total_counts.sum() # 计算所有 species 的总丰度
b_species_relative_abundance = b_species_total_counts / b_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
b_species_above_overall_threshold = b_species_relative_abundance[b_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
b_rel_long_filtered_by_overall_abundance = b_rel_long[b_rel_long['species'].isin(b_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
b_species_origin_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
b_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
b_species_in_origin = b_species_origin_mean_filtered[b_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
b_species_origin_count = b_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
b_species_niche_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
b_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
b_species_in_niche = b_species_niche_mean_filtered[b_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
b_species_niche_count = b_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
b_species_counts_combined = pd.merge(b_species_origin_count, b_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 3
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
b_core_species_names = b_species_counts_combined[
    (b_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (b_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
b_core_abundance = b_species_rel.loc[b_core_species_names]
b_core_species_raw_counts = b_species.loc[b_core_species_names]
print(len(b_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
b_median_relative_abundance = b_species_relative_abundance.median()
print(b_median_relative_abundance)
# 定义稀有类群的丰度阈值 
b_rare_threshold = 0.00001
# 筛选出稀有 species 
b_rare_species = b_species_relative_abundance[b_species_relative_abundance < b_rare_threshold]
# 获取稀有 species 的名称（索引）
b_rare_species_names = b_rare_species.index.tolist()
len(b_rare_species_names)
b_species_db_transposed = b_species_df.T 
b_speciesdfmelt = b_species_db_transposed.stack().reset_index()# 长宽表格转换
b_speciesdfmelt.columns = ['Sample', 'species', 'value']
b_speciesdfmelt = b_speciesdfmelt[b_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
b_control_species = b_speciesdfmelt.copy()
b_control_species['Group'] = b_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
b_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
b_controlspecies_unique = b_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
b_controlspecies_unique = b_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
b_controlspecies_unique = b_controlspecies_unique[b_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
b_unique_species_names_single_group = b_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
b_set_rare_species = set(b_rare_species_names)
b_set_unique_species_single_group = set(b_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
b_rare_unique_species_names = list(b_set_rare_species.intersection(b_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
b_rare_unique_abundance = b_species_df.loc[b_rare_unique_species_names]
b_rare_unique_species_raw_counts = b_species.loc[b_rare_unique_species_names]
print(len(b_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度置换检验与 Z-score 计算
num_permutations = 1000 
b_niche_breadth_results = classify_generalist_specialist(
    b_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (含 Z-score, 前5个 species) ")
print(b_niche_breadth_results.head())
b_generalists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
b_specialists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(b_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(b_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_generalists_species:
    print("无广适类群 species 数据。")
else:
    b_generalists_raw_counts = b_species.loc[b_generalists_species]
    b_generalists_abundance = b_species_rel.loc[b_generalists_species]
    print(b_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_specialists_species:
    print("无专适类群 species 数据。")
else:
    b_specialists_raw_counts = b_species.loc[b_specialists_species]
    b_specialists_abundance = b_species_rel.loc[b_specialists_species]
print(b_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

# 加载生态系统数据
b_ecosystem = metadata
b_ecosystem = b_ecosystem.iloc[:, :2]
b_ecosystem.set_index(b_ecosystem.columns[0], inplace=True)

# 加载不同生态系统（和总览）的边密度 (edge density) 数据
b_density_all = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_density_all_OTUs.csv")
b_density_jr = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/JR/b_edge_density.csv")
b_density_jj = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/JJ/b_edge_density.csv")
b_density_tz = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/TZ/b_edge_density.csv")
b_density_pa = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/PA/b_edge_density.csv")
b_density_g = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/G/b_edge_density.csv")
b_density_n = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/N/b_edge_density.csv")

# 加载不同生态系统（和总览）的网络中心性指标 (network metrics) 数据
b_metrics_all = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_metrics_all_OTUs.csv", index_col=0) # 设置第一列为索引
b_metrics_jr = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/JR/b_network_metrics.csv", index_col=0)
b_metrics_jj = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/JJ/b_network_metrics.csv", index_col=0)
b_metrics_tz = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/TZ/b_network_metrics.csv", index_col=0)
b_metrics_pa = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/PA/b_network_metrics.csv", index_col=0)
b_metrics_g = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/G/b_network_metrics.csv", index_col=0)
b_metrics_n = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/N/b_network_metrics.csv", index_col=0)

# 加载不同类群（丰度、稀有、广谱、特有）的相对丰度数据
b_db_OTU_rel = b_species_rel 
b_db_abun = b_core_abundance 
b_db_rare = b_rare_unique_abundance
b_db_gen = b_generalists_abundance
b_db_spe = b_specialists_abundance

# 加载 OTU 的分类学信息
b_taxon = b_tax_species

# 为边密度数据框的索引重新命名，使其代表不同的生态系统类别
b_density_all.index = ['All']
b_density_jr.index = ['Jurong']
b_density_jj.index = ['Jingjiang']
b_density_tz.index = ['Tongzhou']
b_density_pa.index = ['Pan\'an']
b_density_g.index = ['Rhizosphere Soil']
b_density_n.index = ['Bulb']

# 为网络中心性指标数据框的索引添加前缀 "OTU_"
# 这有助于确保索引格式一致性，例如 '123' 变为 'OTU_123'
prefix = 'species_'
b_metrics_all.index = b_metrics_all.index.astype(str).map(lambda x: prefix + x)
b_metrics_jr.index = b_metrics_jr.index.astype(str).map(lambda x: prefix + x)
b_metrics_jj.index = b_metrics_jj.index.astype(str).map(lambda x: prefix + x)
b_metrics_tz.index = b_metrics_tz.index.astype(str).map(lambda x: prefix + x)
b_metrics_pa.index = b_metrics_pa.index.astype(str).map(lambda x: prefix + x)
b_metrics_g.index = b_metrics_g.index.astype(str).map(lambda x: prefix + x)
b_metrics_n.index = b_metrics_n.index.astype(str).map(lambda x: prefix + x)

#定义函数：加载和熔化边权重数据
def load_melt_edge_wt(path_to_the_file):
    """
    加载 SpiecEasi 生成的边权重矩阵文件，并将其转换为长格式（'熔化'）。
    同时对数据进行清洗，移除自连接和权重为零的边。

    Args:
        path_to_the_file (str): 包含边权重数据的 CSV 文件的完整路径。

    Returns:
        pd.DataFrame: 经过处理的边权重数据框，包含 'Sample i'（起始 OTU），
                      'Sample j'（结束 OTU）和 'weight'（边权重）三列。
    """
    # 读取 CSV 文件，并将第一列作为数据框的索引
    edge_wt = pd.read_csv(path_to_the_file, index_col=0)
    # 为索引添加 'V' 前缀，以匹配通常的节点命名约定（例如 V1, V2），方便后续处理
    edge_wt.index = edge_wt.index.astype(str).map(lambda x: 'V' + x)

    # 提取矩阵的上三角形部分（包括对角线），其他部分填充为 NaN
    # 这样做是为了避免重复的边 (因为矩阵是对称的)
    edge_wt_melted = edge_wt.where(np.triu(np.ones(edge_wt.shape)).astype(bool))
    # 将宽格式的数据框“熔化”为长格式。'ignore_index=False' 保留原索引作为一列，
    # 然后 .dropna() 移除所有包含 NaN 的行（即上三角形之外的冗余部分）
    edge_wt_melted = edge_wt_melted.melt(ignore_index=False).dropna()

    # 重命名索引和列，使其更具描述性
    edge_wt_melted.index.name = 'Sample i'
    edge_wt_melted.columns = ['Sample j', 'weight']
    # 将索引 'Sample i' 变为常规列
    edge_wt_melted = edge_wt_melted.reset_index()

    # 过滤数据：移除自身连接的行，即起始 OTU 和结束 OTU 相同的行
    edge_wt_melted_filtered = edge_wt_melted[edge_wt_melted['Sample i'] != edge_wt_melted['Sample j']]
    # 过滤数据：移除权重为 0 的行，因为在网络分析中它们不代表实际连接
    edge_wt_melted_filtered = edge_wt_melted_filtered[edge_wt_melted_filtered['weight'] != 0]

    return edge_wt_melted_filtered

# 加载并处理所有 OTU 的边权重数据
b_edge_wt_all = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_weight_all_OTUs.csv")

# 绘制所有 OTU 边权重的直方图
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
sns.histplot(data= b_edge_wt_all, x='weight', color="#F1C40F", kde=True) # 绘制直方图，带核密度估计曲线
plt.xlabel("") # 清空 X 轴标签
plt.ylabel("Count", fontsize=12) # 设置 Y 轴标签
plt.title("Edge weights (all bacterial species)", fontsize=13) # 设置图表标题
plt.savefig("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_weight_all_hist.pdf", bbox_inches="tight", dpi=600) # 保存图表为 PDF
plt.show() # 显示图表

# 绘制所有 OTU 边权重的局部直方图 (聚焦于权重接近 0 的部分)
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
sns.histplot(data= b_edge_wt_all, x='weight', color="#F1C40F", kde=True) # 绘制直方图，带核密度估计曲线
plt.xlabel("") # 清空 X 轴标签
plt.ylabel("Count", fontsize=12) # 设置 Y 轴标签
plt.title("Edge weights (all bacterial species)", fontsize=13) # 设置图表标题
plt.xlim(-0.1, 0.1) # 设置 X 轴的显示范围，聚焦于 -0.1 到 0.1 之间
plt.savefig("/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_weight_all_hist_sub.pdf", bbox_inches="tight", dpi=300) # 保存图表为 PDF
plt.show() # 显示图表

#定义函数：计算正负边权重数量并绘制饼图
def edge_wt_pos_neg_ct_pie(edge_wt_df, color, title, path_to_file):
    """
    计算给定边权重数据中正权重和负权重的数量，并绘制饼图。

    Args:
        edge_wt_df (pd.DataFrame): 包含 'weight' 列的边权重数据框。
        color (list): 饼图扇形的颜色列表，第一个颜色对应正权重，第二个对应负权重。
        title (str): 饼图的标题。
        path_to_file (str): 保存饼图的路径和文件名 (PDF 格式)。

    Returns:
        tuple: (正权重数量, 负权重数量)
    """
    # 计算权重大于 0 的边的数量 (正连接)
    count_positive = (edge_wt_df['weight'] > 0).sum()
    # 计算权重小于 0 的边的数量 (负连接)
    count_negative = (edge_wt_df['weight'] < 0).sum()

    # 将正负计数转换为 Pandas Series，便于绘制饼图
    counts = pd.Series([count_positive, count_negative], index=['Postive', 'Negative'])

    plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
    # 绘制饼图：autopct 显示百分比，colors 设置颜色，startangle 设置起始角度
    counts.plot(kind='pie', autopct='%1.1f%%', colors=color, startangle=90, textprops={'fontsize': 12})
    plt.title(title, size = 14) # 设置标题
    plt.xlabel('') # 清空 X 轴标签
    plt.ylabel('') # 清空 Y 轴标签
    plt.savefig(path_to_file, bbox_inches="tight", dpi=600) # 保存图表
    plt.show() # 显示图表

    return (count_positive, count_negative) # 返回正负权重数量

# 使用上述函数绘制所有 OTU 边权重的饼图
# 返回的值 (13476, 3447)是正权重和负权重的数量
edge_wt_pos_neg_ct_pie(b_edge_wt_all, ['#F1C40F', '#F9E79F'], 'Edge weights (all bacterial species)', "/mnt/d/study/master/Main_Figure_tables/Figure_5/b_edge_weight_all_pie.pdf")

# 将各个生态系统的边密度数据合并到一个数据框中
b_density_concat = pd.concat([b_density_jr, b_density_jj, b_density_tz, b_density_pa, b_density_g, b_density_n])
b_density_concat.index.names = ['group'] # 将索引命名为 'group'
b_density_concat = b_density_concat.reset_index() # 将索引转换为常规列
b_density_concat = b_density_concat.sort_values(by = 'edge density') # 根据边密度值排序

# 定义不同生态系统的颜色映射
group_colors = {
    'Jurong': '#45B39D',
    'Jingjiang': '#666666',
    'Tongzhou': '#1E90FF',
    'Pan\'an': '#CD6155',
    'Rhizosphere Soil': '#A0522D',
    'Bulb': '#AF7AC5'
}

# 绘制生态系统网络密度的柱状图
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
ax = sns.barplot(data = b_density_concat, x='group', y="edge density", hue='group', palette=group_colors, dodge=False) # 绘制柱状图
ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('') # 清空 Y 轴标签 (此处应是 'Edge Density')
ax.set_title('Bacterial network density of populations and niches', size=13) # 设置标题
plt.xticks(fontsize=13, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和对齐方式
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_density_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

# 加载并处理每个生态系统的边权重数据
b_edge_wt_jr = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/JR/b_edge_weight.csv")
b_edge_wt_n = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/N/b_edge_weight.csv")
b_edge_wt_g = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/G/b_edge_weight.csv")
b_edge_wt_pa = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/PA/b_edge_weight.csv")
b_edge_wt_tz = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/TZ/b_edge_weight.csv")
b_edge_wt_jj = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/JJ/b_edge_weight.csv")

# 为每个生态系统绘制边权重饼图，并获取正负权重计数
b_pos_ct_jr, b_neg_ct_jr = edge_wt_pos_neg_ct_pie(b_edge_wt_jr, ['#45B39D', '#79cbbb'],
                                                      'Jurong',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/JR/b_edge_weight_jr_pie.pdf")

b_pos_ct_n, b_neg_ct_n = edge_wt_pos_neg_ct_pie(b_edge_wt_n, ['#AF7AC5', '#D2B4DE'],
                                                      'Bulb',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/N/b_edge_weight_n_pie.pdf")

b_pos_ct_g, b_neg_ct_g = edge_wt_pos_neg_ct_pie(b_edge_wt_g, ['#A0522D', '#bd866c'],
                                                      'Rhizosphere Soil',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/G/b_edge_weight_g_pie.pdf")

b_pos_ct_pa, b_neg_ct_pa = edge_wt_pos_neg_ct_pie(b_edge_wt_pa, ['#CD6155', '#E6B0AA'],
                                                      'Pan\'an',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/PA/b_edge_weight_pa_pie.pdf")

b_pos_ct_tz, b_neg_ct_tz = edge_wt_pos_neg_ct_pie(b_edge_wt_tz, ['#1E90FF', '#78bcff'],
                                                      'Tongzhou',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/TZ/b_edge_weight_tz_pie.pdf")

b_pos_ct_jj, b_neg_ct_jj = edge_wt_pos_neg_ct_pie(b_edge_wt_jj, ['#666666', '#D5D8DC'],
                                                      'Jingjiang',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/JJ/b_edge_weight_jj_pie.pdf")

#对正负边权重数量执行 Fisher 精确检验
# 导入 rpy2 库，用于在 Python 中调用 R 函数
import rpy2.robjects as robjects
from rpy2.robjects import numpy2ri, default_converter
from rpy2.robjects.packages import importr
stats = importr('stats') # 导入 R 的 'stats' 包
# 构建一个 NumPy 数组，包含每个生态系统正负边权重的计数
b_m = np.array([[b_pos_ct_jr, b_neg_ct_jr],
              [b_pos_ct_jj, b_neg_ct_jj],
              [b_pos_ct_g, b_neg_ct_g],
              [b_pos_ct_pa, b_neg_ct_pa],
              [b_pos_ct_n, b_neg_ct_n],
              [b_pos_ct_tz, b_neg_ct_tz]])
# 执行 Fisher 精确检验，并模拟 P 值（当样本量大时，精确计算可能困难）
# simulate_p_value=True 适用于 2xK 列联表
b_m = np.array([[b_pos_ct_jr, b_neg_ct_jr],
                [b_pos_ct_jj, b_neg_ct_jj],
                [b_pos_ct_g, b_neg_ct_g],
                [b_pos_ct_pa, b_neg_ct_pa],
                [b_pos_ct_n, b_neg_ct_n],
                [b_pos_ct_tz, b_neg_ct_tz]])
# 使用上下文管理器进行转换
with (default_converter + numpy2ri.converter).context():
    # 在这个缩进块内，rpy2 会自动把 numpy 数组识别为 R 矩阵
    res = stats.fisher_test(b_m, simulate_p_value=True)
    
    # 提取 P 值
    # 注意：rpy2 返回的是列表，res[0] 通常是 p-value
    p_val = res[0][0]
    print('p-value: {}'.format(res[0][0])) # 打印 Fisher 检验的 P 值
# 输出示例: p-value: 0.0004997501249375312

# 为每个生态系统的网络中心性数据框添加 'group' 列，标记其所属的生态系统类别
b_metrics_jr['group'] = 'Jurong'
b_metrics_jj['group'] = 'Jingjiang'
b_metrics_tz['group'] = 'Tongzhou'
b_metrics_pa['group'] = 'Pan\'an'
b_metrics_g['group'] = 'Rhizosphere Soil'
b_metrics_n['group'] = 'Bulb'

# 将所有生态系统的网络中心性数据合并到一个数据框中，用于后续的比较分析
b_metrics_cancat = pd.concat([b_metrics_jr, b_metrics_jj, b_metrics_tz, b_metrics_pa, b_metrics_g, b_metrics_n])

#执行 Kruskal-Wallis 检验 

# 初始化字典用于存储 Kruskal-Wallis 检验的 P 值
kw_pval = {}
# 遍历需要进行检验的中心性指标：'degree', 'closeness', 'betweenness'
for i in ['degree', 'closeness', 'betweenness']:
    # 对每个指标执行 Kruskal-Wallis 检验，比较六个生态系统组的差异
    stat, p_value = kruskal(b_metrics_jr[i], b_metrics_jj[i], b_metrics_pa[i], b_metrics_tz[i], b_metrics_g[i], b_metrics_n[i])
    # 打印每个指标在不同生态系统中的中位数和 Kruskal-Wallis 检验的 P 值
    print(i, 'Jurong:', np.median(b_metrics_jr[i]),
          'Jingjiang:', np.median(b_metrics_jj[i]),
          'Pan\'an:', np.median(b_metrics_pa[i]),
           'Tongzhou:', np.median(b_metrics_tz[i]),
           'Rhizosphere Soil:', np.median(b_metrics_g[i]),
           'Bulb:', np.median(b_metrics_n[i]))
    kw_pval[i] = p_value # 将 P 值存储到字典中

# 输出示例 (Kruskal-Wallis 检验结果):
# degree forest: 27.0 Jingjiang: 25.0 Pan\'an: 24.0 Tongzhou: 25.0 Rhizosphere Soil: 27.0 steppe: 27.0
# closeness forest: 4.29367e-06 Jingjiang: 3.559657e-06 Pan\'an: 4.3339205e-06 Tongzhou: 3.043133e-06 Rhizosphere Soil: 4.223972e-06 steppe: 4.4570035e-06
# betweenness forest: 873.0 Jingjiang: 1027.0 Pan\'an: 794.25 Tongzhou: 1044.0 Rhizosphere Soil: 834.0 steppe: 871.5

# 定义用于绘图的生态系统颜色映射字典
group_colors = {
    'Jurong': '#45B39D',
    'Jingjiang': '#666666',
    'Tongzhou': '#1E90FF',
    'Pan\'an': '#CD6155',
    'Rhizosphere Soil': '#A0522D',
    'Bulb': '#AF7AC5'
}

# 定义箱线图的显示顺序
order = ['Pan\'an', 'Jingjiang', 'Jurong', 'Bulb', 'Tongzhou', 'Rhizosphere Soil']
# 定义用于统计比较的组对
pairs=[
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Tongzhou'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Bulb'),
    ('Jurong', 'Tongzhou'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Rhizosphere Soil'),
    ('Jurong', 'Bulb'),
    ('Tongzhou', 'Jingjiang'),
    ('Tongzhou', 'Bulb'), # 调整：原始代码缺少 Tongzhou 与 Rhizosphere Soil 的比较，这里添加了
    ('Tongzhou', 'Rhizosphere Soil'),
    ('Jingjiang', 'Rhizosphere Soil'),
    ('Jingjiang', 'Bulb'),
    ('Rhizosphere Soil', 'Bulb'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
# 绘制箱线图：x 轴为生态系统组，y 轴为节点度，根据 group_colors 设置颜色，不显示异常值 (showfliers=False)
ax = sns.boxplot(data= b_metrics_cancat, x='group', y='degree', order=order,  palette=group_colors, showfliers=False)

# 使用 Annotator 在箱线图上添加统计显著性注释
annot = Annotator(ax, pairs, data=b_metrics_cancat, x='group', y='degree', order=order)
# 配置检验方法（曼-惠特尼 U 检验）和多重比较校正方法（Benjamini-Hochberg）
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用统计检验
annot.annotate() # 在图上添加注释（显著性星号或 P 值）

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Degree', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 在图的左上角添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.96, f'Kruskal-Wallis P = {kw_pval["degree"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和对齐方式
plt.ylim(-5, 145) # 设置 Y 轴的显示范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_degree_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表
print(" 各组 Degree 中位数 ")
for group_name in order:
    # 筛选出当前组的数据
    subset_data = b_metrics_cancat[b_metrics_cancat['group'] == group_name]['degree']
    # 计算中位数
    median_val = subset_data.median()
    # 打印中位数
    print(f"{group_name}: {median_val:.2f}") # 打印两位小数

# 定义箱线图的显示顺序，基于 Closeness 值对生态系统进行排序
order = ['Tongzhou', 'Jingjiang', 'Rhizosphere Soil', 'Jurong', 'Bulb', 'Pan\'an']

# 定义用于 Mann-Whitney U 检验的组对。这些是所有可能的两两比较。
pairs=[
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Bulb'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Tongzhou'),
    ('Rhizosphere Soil', 'Jurong'),
    ('Rhizosphere Soil', 'Bulb'),
    ('Rhizosphere Soil', 'Jingjiang'),
    ('Rhizosphere Soil', 'Tongzhou'),
    ('Jurong', 'Bulb'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Tongzhou'),
    ('Bulb', 'Jingjiang'),
    ('Bulb', 'Tongzhou'),
    ('Jingjiang', 'Tongzhou'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小 (宽x高)
# 绘制箱线图：
# data: 合并后的网络中心性数据框
# x: 分组变量 'group' (生态系统类型)
# y: 数值变量 'closeness' (接近中心性)
# order: 指定 x 轴分组的显示顺序
# palette: 使用预定义的生态系统颜色
# showfliers=False: 不显示离群点 (outliers)
ax = sns.boxplot(data=b_metrics_cancat, x='group', y='closeness', order=order, palette=group_colors, showfliers=False)

# 使用 Annotator 在箱线图上添加统计显著性注释
annot = Annotator(ax, pairs, data=b_metrics_cancat, x='group', y='closeness', order=order)
# 配置统计检验方法为 'Mann-Whitney' (曼-惠特尼 U 检验，非参数两独立样本检验)
# 配置多重比较校正方法为 "Benjamini-Hochberg" (FDR 校正)
# verbose=2: 显示详细的检验过程和结果
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用配置的统计检验
annot.annotate() # 在图上绘制显著性注释（如星号或 P 值）

ax.set_xlabel('') # 清空 X 轴标签，因为信息已在刻度上显示
ax.set_ylabel('Closeness', size=13) # 设置 Y 轴标签及其字体大小
ax.set_title('', size=13) # 设置图表标题及其字体大小
# 在图表内部添加 Kruskal-Wallis 检验的 P 值
# transform=ax.transAxes 表示坐标是相对于坐标轴的比例 (0-1)
plt.text(0.05, 0.97, f'Kruskal-Wallis P = {kw_pval["closeness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和右对齐
plt.ylim(1.6e-6, 1.6e-5) # 设置 Y 轴的显示范围，以更好地展示数据分布
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_closeness_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表为 PDF，紧密裁剪，高 DPI
plt.show() # 显示图表

# 定义箱线图的显示顺序，基于 Betweenness 值对生态系统进行排序
order = ['Pan\'an', 'Bulb', 'Jurong', 'Tongzhou', 'Jingjiang', 'Rhizosphere Soil']

# 定义用于 Mann-Whitney U 检验的组对
pairs=[
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Bulb'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Tongzhou'),
    ('Rhizosphere Soil', 'Jurong'),
    ('Rhizosphere Soil', 'Bulb'),
    ('Rhizosphere Soil', 'Jingjiang'),
    ('Rhizosphere Soil', 'Tongzhou'),
    ('Jurong', 'Bulb'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Tongzhou'),
    ('Bulb', 'Jingjiang'),
    ('Bulb', 'Tongzhou'),
    ('Jingjiang', 'Tongzhou'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
# 绘制箱线图：数据、x轴、y轴、顺序、颜色、不显示异常值
ax = sns.boxplot(data=b_metrics_cancat, x='group', y='betweenness', order=order, palette=group_colors, showfliers=False)

# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=b_metrics_cancat, x='group', y='betweenness', order=order)
# 配置检验方法和多重比较校正
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Betweenness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.97, f'Kruskal-Wallis P = {kw_pval["betweenness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(-500, 8200) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_betweenness_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

# 从所有 OTU 的网络中心性数据中，根据丰度/稀有/广谱/特有类群的索引筛选出相应的数据
b_metrics_abun = b_metrics_all[b_metrics_all.index.isin(b_db_abun.index)].copy()
b_metrics_abun['Ecotype'] = 'Cores' # 添加 'Ecotype' 列并赋值

b_metrics_rare = b_metrics_all[b_metrics_all.index.isin(b_db_rare.index)].copy()
b_metrics_rare['Ecotype'] = 'Uniques'

b_metrics_gen = b_metrics_all[b_metrics_all.index.isin(b_db_gen.index)].copy()
b_metrics_gen['Ecotype'] = 'Generalists'

b_metrics_spe = b_metrics_all[b_metrics_all.index.isin(b_db_spe.index)].copy()
b_metrics_spe['Ecotype'] = 'Specialists'

# 将所有生态类型的节点中心性数据合并到一个数据框中，用于后续的比较分析
b_metrics_ecotype = pd.concat([b_metrics_abun, b_metrics_rare, b_metrics_gen, b_metrics_spe])

# 初始化字典用于存储 Kruskal-Wallis 检验的 P 值
kw_pval = {}
# 遍历需要进行检验的中心性指标
for i in ['degree', 'closeness', 'betweenness']:
    # 对每个指标执行 Kruskal-Wallis 检验，比较四个生态类型组的差异
    stat, p_value = kruskal(b_metrics_abun[i], b_metrics_rare[i], b_metrics_gen[i], b_metrics_spe[i])
    # 打印每个指标在不同生态类型中的中位数和 P 值
    print(i, 'abundant:', np.median(b_metrics_abun[i]),
          'rare:', np.median(b_metrics_rare[i]),
          'generalist:', np.median(b_metrics_gen[i]),
          'specialist:', np.median(b_metrics_spe[i]))
    kw_pval[i] = p_value # 将 P 值存储到字典中

# 输出示例 (Kruskal-Wallis 检验结果):
# degree abundant: 49.0 rare: 53.0 generalist: 54.5 specialist: 54.0
# closeness abundant: 8.548408e-07 rare: 9.008329e-07 generalist: 8.993396000000001e-07 specialist: 8.638988e-07
# betweenness abundant: 1970.0 rare: 2626.0 generalist: 2245.0 specialist: 1886.0

# 定义箱线图的显示顺序（生态类型）
order=['Cores', 'Uniques', 'Generalists', 'Specialists']

# 定义不同生态类型的颜色映射
group_colors = {
    'Cores': '#5499C7', # 蓝色
    'Uniques': '#D4AC0D',     # 黄色
    'Generalists': '#EC7063',   # 红色
    'Specialists': '#45B39D'    # 绿色
}

# 定义用于 Mann-Whitney U 检验的组对 (所有两两比较)
pairs=[
    ('Cores', 'Uniques'),
    ('Cores', 'Generalists'),
    ('Cores', 'Specialists'),
    ('Uniques', 'Generalists'),
    ('Uniques', 'Specialists'),
    ('Generalists', 'Specialists'),
]

plt.rcParams["figure.figsize"] = (3, 4) # 设置图表大小 (宽x高)，相对更窄

#绘制节点度（Degree）箱线图 (针对生态类型) 
ax = sns.boxplot(data=b_metrics_ecotype, x='Ecotype', y='degree', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=b_metrics_ecotype, x='Ecotype', y='degree', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Degree', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["degree"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(0, 150) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_degree_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点中介中心性（Betweenness）箱线图 (针对生态类型) 
ax = sns.boxplot(data=b_metrics_ecotype, x='Ecotype', y='betweenness', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=b_metrics_ecotype, x='Ecotype', y='betweenness', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Betweenness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["betweenness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(-2000, 32000) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_betweenness_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点接近中心性（Closeness）箱线图 (针对生态类型) 
ax = sns.boxplot(data=b_metrics_ecotype, x='Ecotype', y='closeness', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=b_metrics_ecotype, x='Ecotype', y='closeness', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Closeness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["closeness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(0.05e-6, 1.9e-6) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_closeness_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#网络节点指标比较 (针对门 Phyla) 
# 将所有 OTU 的网络中心性指标数据 (b_metrics_all) 与分类学信息 (taxon) 进行合并。
# 合并依据是两个数据框的索引 ('left_index=True', 'right_index=True')。
# 只选择 taxon 数据框中的 'Phylum' 列进行合并，以便将每个 OTU 与其所属的门关联起来。
b_metrics_phyla = pd.merge(b_metrics_all, b_taxon[['Phylum']], left_index=True, right_index=True)

# 按 'Phylum' 列进行分组，并计算每个门的平均节点指标（degree, closeness, betweenness）。
# 这将用于确定绘图时门的排序。
b_metrics_phyla_mean = b_metrics_phyla.groupby('Phylum').mean()

#绘制节点度（Degree）的线点图 (Pointplot) 
# 根据平均度值对门进行降序排序，并获取排序后的门名称列表。
b_order_degree = b_metrics_phyla_mean.sort_values(by='degree', ascending=False).index.to_list()

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小 (宽x高)，垂直方向相对更长，适合显示多个门
# 绘制点图：
# x='degree': X 轴显示节点度
# y='Phylum': Y 轴显示门名称
# data: 合并了门信息的网络中心性数据
# order: 指定 Y 轴上门的显示顺序
# join=False: 不连接点，只显示点和误差线
# capsize=0.2: 误差线帽的宽度
# errwidth=1: 误差线的宽度
# color='#DC7633': 设置点的颜色
sns.pointplot(x='degree', y='Phylum', data=b_metrics_phyla, order=b_order_degree, join=False, capsize=0.2, errwidth=1, color='#DC7633')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=10) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签及其字体大小
plt.xlabel('Degree', size=14) # 设置 X 轴标签及其字体大小
plt.title('', size=14) # 设置图表标题及其字体大小
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_degree_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表为 PDF，紧密裁剪，高 DPI
plt.show() # 显示图表

#绘制节点接近中心性（Closeness）的线点图 (Pointplot) 
# 根据平均接近中心性值对门进行降序排序，并获取排序后的门名称列表。
b_order_closeness = b_metrics_phyla_mean.sort_values(by='closeness', ascending=False).index.to_list()

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小
# 绘制点图：x 轴为接近中心性，y 轴为门，使用指定数据、顺序和颜色
sns.pointplot(x='closeness', y='Phylum', data=b_metrics_phyla, order=b_order_closeness, join=False, capsize=0.2, errwidth=1, color='#BB8FCE')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=10) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签
plt.xlabel('Closeness', size=14) # 设置 X 轴标签
plt.title('', size=14) # 设置图表标题
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_closeness_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点中介中心性（Betweenness）的线点图 (Pointplot) 
# 根据平均中介中心性值对门进行降序排序，并获取排序后的门名称列表。
b_order_betweenness = b_metrics_phyla_mean.sort_values(by='betweenness', ascending=False).index.to_list()

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小
# 绘制点图：x 轴为中介中心性，y 轴为门，使用指定数据、顺序和颜色
sns.pointplot(x='betweenness', y='Phylum', data=b_metrics_phyla, order=b_order_betweenness, join=False, capsize=0.2, errwidth=1, color='#73C6B6')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=10) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签
plt.xlabel('Betweenness', size=14) # 设置 X 轴标签
plt.title('', size=14) # 设置图表标题
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/b_network_betweenness_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表



#真菌共线性其他
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
fungi_ASV = pd.read_csv("/mnt/d/study/master/meiji/fungi_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 fungi_ASV DataFrame 中选择指定的列
f_ASV = fungi_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
f_ASV.set_index(keys="ASV", inplace=True)
f_tax = fungi_ASV.iloc[:, 1:9].copy()
f_tax.set_index(keys="ASV", inplace=True)
print(len(f_ASV))
# 计算 ASV 相对丰度
f_ASV_df = f_ASV.copy()

#按种
f_species = f_ASV.join(f_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
f_species = f_species.groupby('Species')[sample_ids].sum()
#排序
f_species = f_species.sort_index()
# 将当前的 ASV 索引重置为一个普通列。
f_tax_reset_index = f_tax.reset_index()
# 将 'Species' 列设置为新的索引。
f_tax_species = f_tax_reset_index.groupby('Species').first()
f_tax_species = f_tax_species.drop(columns=['ASV'])
f_tax_species = f_tax_species.sort_index()
# 进行替换操作
f_tax_species['Phylum'] = f_tax_species['Phylum'].replace('p__unclassified_k__Fungi', 'Unclassified')
# 去除 'p__' 前缀
f_tax_species['Phylum'] = f_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
f_unique_species_names = f_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
f_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(f_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 f_species 的索引
f_species = f_species.rename(index= f_species_to_numeric_id)
# 转换 f_tax_species 的索引
f_tax_species = f_tax_species.rename(index= f_species_to_numeric_id)
f_species_df = f_species.copy()
print(len(f_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
f_species_rel = f_species_df.div(f_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
f_rel_long = f_species_rel.stack().reset_index()
f_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
f_rel_long['Origin'] = f_rel_long['Sample'].map(sample_origins_map)
f_rel_long['Niche'] = f_rel_long['Sample'].map(sample_niches_map)
f_species_total_counts = f_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
f_total_counts_across_all_species = f_species_total_counts.sum() # 计算所有 species 的总丰度
f_species_relative_abundance = f_species_total_counts / f_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
f_species_above_overall_threshold = f_species_relative_abundance[f_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
f_rel_long_filtered_by_overall_abundance = f_rel_long[f_rel_long['species'].isin(f_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
f_species_origin_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
f_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
f_species_in_origin = f_species_origin_mean_filtered[f_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
f_species_origin_count = f_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
f_species_niche_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
f_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
f_species_in_niche = f_species_niche_mean_filtered[f_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
f_species_niche_count = f_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
f_species_counts_combined = pd.merge(f_species_origin_count, f_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 3
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
f_core_species_names = f_species_counts_combined[
    (f_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (f_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
f_core_abundance = f_species_rel.loc[f_core_species_names]
f_core_species_raw_counts = f_species.loc[f_core_species_names]
print(len(f_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
f_median_relative_abundance = f_species_relative_abundance.median()
print(f_median_relative_abundance)
# 定义稀有类群的丰度阈值 
f_rare_threshold = 0.000008
# 筛选出稀有 species 
f_rare_species = f_species_relative_abundance[f_species_relative_abundance < f_rare_threshold]
# 获取稀有 species 的名称（索引）
f_rare_species_names = f_rare_species.index.tolist()
len(f_rare_species_names)
f_species_df_transposed = f_species_df.T 
f_speciesdfmelt = f_species_df_transposed.stack().reset_index()# 长宽表格转换
f_speciesdfmelt.columns = ['Sample', 'species', 'value']
f_speciesdfmelt = f_speciesdfmelt[f_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
f_control_species = f_speciesdfmelt.copy()
f_control_species['Group'] = f_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
f_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
f_controlspecies_unique = f_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
f_controlspecies_unique = f_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
f_controlspecies_unique = f_controlspecies_unique[f_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
f_unique_species_names_single_group = f_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
f_set_rare_species = set(f_rare_species_names)
f_set_unique_species_single_group = set(f_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
f_rare_unique_species_names = list(f_set_rare_species.intersection(f_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
f_rare_unique_abundance = f_species_df.loc[f_rare_unique_species_names]
f_rare_unique_species_raw_counts = f_species.loc[f_rare_unique_species_names]
print(len(f_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度置换检验与 Z-score 计算
num_permutations = 1000 
f_niche_breadth_results = classify_generalist_specialist(
    f_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (含 Z-score, 前5个 species) ")
print(f_niche_breadth_results.head())
f_generalists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
f_specialists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(f_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(f_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_generalists_species:
    print("无广适类群 species 数据。")
else:
    f_generalists_raw_counts = f_species.loc[f_generalists_species]
    f_generalists_abundance = f_species_rel.loc[f_generalists_species]
    print(f_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_specialists_species:
    print("无专适类群 species 数据。")
else:
    f_specialists_raw_counts = f_species.loc[f_specialists_species]
    f_specialists_abundance = f_species_rel.loc[f_specialists_species]
print(f_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

# 加载生态系统数据
f_ecosystem = metadata
f_ecosystem = f_ecosystem.iloc[:, :2]
f_ecosystem.set_index(f_ecosystem.columns[0], inplace=True)

# 加载不同生态系统（和总览）的边密度 (edge density) 数据
f_density_all = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_density_all_OTUs.csv")
f_density_jr = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/JR/f_edge_density.csv")
f_density_jj = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/JJ/f_edge_density.csv")
f_density_tz = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/TZ/f_edge_density.csv")
f_density_pa = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/PA/f_edge_density.csv")
f_density_g = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/G/f_edge_density.csv")
f_density_n = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/N/f_edge_density.csv")

# 加载不同生态系统（和总览）的网络中心性指标 (network metrics) 数据
f_metrics_all = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_metrics_all_OTUs.csv", index_col=0) # 设置第一列为索引
f_metrics_jr = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/JR/f_network_metrics.csv", index_col=0)
f_metrics_jj = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/JJ/f_network_metrics.csv", index_col=0)
f_metrics_tz = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/TZ/f_network_metrics.csv", index_col=0)
f_metrics_pa = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/PA/f_network_metrics.csv", index_col=0)
f_metrics_g = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/G/f_network_metrics.csv", index_col=0)
f_metrics_n = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/N/f_network_metrics.csv", index_col=0)

# 加载不同类群（丰度、稀有、广谱、特有）的相对丰度数据
f_df_OTU_rel = f_species_rel 
f_df_abun = f_core_abundance 
f_df_rare = f_rare_unique_abundance
f_df_gen = f_generalists_abundance
f_df_spe = f_specialists_abundance

# 加载 OTU 的分类学信息
f_taxon = f_tax_species

# 为边密度数据框的索引重新命名，使其代表不同的生态系统类别
f_density_all.index = ['All']
f_density_jr.index = ['Jurong']
f_density_jj.index = ['Jingjiang']
f_density_tz.index = ['Tongzhou']
f_density_pa.index = ['Pan\'an']
f_density_g.index = ['Rhizosphere Soil']
f_density_n.index = ['Bulb']

# 为网络中心性指标数据框的索引添加前缀 "OTU_"
# 这有助于确保索引格式一致性，例如 '123' 变为 'OTU_123'
prefix = 'species_'
f_metrics_all.index = f_metrics_all.index.astype(str).map(lambda x: prefix + x)
f_metrics_jr.index = f_metrics_jr.index.astype(str).map(lambda x: prefix + x)
f_metrics_jj.index = f_metrics_jj.index.astype(str).map(lambda x: prefix + x)
f_metrics_tz.index = f_metrics_tz.index.astype(str).map(lambda x: prefix + x)
f_metrics_pa.index = f_metrics_pa.index.astype(str).map(lambda x: prefix + x)
f_metrics_g.index = f_metrics_g.index.astype(str).map(lambda x: prefix + x)
f_metrics_n.index = f_metrics_n.index.astype(str).map(lambda x: prefix + x)

#定义函数：加载和熔化边权重数据
def load_melt_edge_wt(path_to_the_file):
    """
    加载 SpiecEasi 生成的边权重矩阵文件，并将其转换为长格式（'熔化'）。
    同时对数据进行清洗，移除自连接和权重为零的边。

    Args:
        path_to_the_file (str): 包含边权重数据的 CSV 文件的完整路径。

    Returns:
        pd.DataFrame: 经过处理的边权重数据框，包含 'Sample i'（起始 OTU），
                      'Sample j'（结束 OTU）和 'weight'（边权重）三列。
    """
    # 读取 CSV 文件，并将第一列作为数据框的索引
    edge_wt = pd.read_csv(path_to_the_file, index_col=0)
    # 为索引添加 'V' 前缀，以匹配通常的节点命名约定（例如 V1, V2），方便后续处理
    edge_wt.index = edge_wt.index.astype(str).map(lambda x: 'V' + x)

    # 提取矩阵的上三角形部分（包括对角线），其他部分填充为 NaN
    # 这样做是为了避免重复的边 (因为矩阵是对称的)
    edge_wt_melted = edge_wt.where(np.triu(np.ones(edge_wt.shape)).astype(bool))
    # 将宽格式的数据框“熔化”为长格式。'ignore_index=False' 保留原索引作为一列，
    # 然后 .dropna() 移除所有包含 NaN 的行（即上三角形之外的冗余部分）
    edge_wt_melted = edge_wt_melted.melt(ignore_index=False).dropna()

    # 重命名索引和列，使其更具描述性
    edge_wt_melted.index.name = 'Sample i'
    edge_wt_melted.columns = ['Sample j', 'weight']
    # 将索引 'Sample i' 变为常规列
    edge_wt_melted = edge_wt_melted.reset_index()

    # 过滤数据：移除自身连接的行，即起始 OTU 和结束 OTU 相同的行
    edge_wt_melted_filtered = edge_wt_melted[edge_wt_melted['Sample i'] != edge_wt_melted['Sample j']]
    # 过滤数据：移除权重为 0 的行，因为在网络分析中它们不代表实际连接
    edge_wt_melted_filtered = edge_wt_melted_filtered[edge_wt_melted_filtered['weight'] != 0]

    return edge_wt_melted_filtered

# 加载并处理所有 OTU 的边权重数据
f_edge_wt_all = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_weight_all_OTUs.csv")

# 绘制所有 OTU 边权重的直方图
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
sns.histplot(data= f_edge_wt_all, x='weight', color="#F1C40F", kde=True) # 绘制直方图，带核密度估计曲线
plt.xlabel("") # 清空 X 轴标签
plt.ylabel("Count", fontsize=12) # 设置 Y 轴标签
plt.title("Edge weights (all fungal species)", fontsize=13) # 设置图表标题
plt.savefig("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_weight_all_hist.pdf", bbox_inches="tight", dpi=600) # 保存图表为 PDF
plt.show() # 显示图表

# 绘制所有 OTU 边权重的局部直方图 (聚焦于权重接近 0 的部分)
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
sns.histplot(data= f_edge_wt_all, x='weight', color="#F1C40F", kde=True) # 绘制直方图，带核密度估计曲线
plt.xlabel("") # 清空 X 轴标签
plt.ylabel("Count", fontsize=12) # 设置 Y 轴标签
plt.title("Edge weights (all fungal species)", fontsize=13) # 设置图表标题
plt.xlim(-0.1, 0.1) # 设置 X 轴的显示范围，聚焦于 -0.1 到 0.1 之间
plt.savefig("/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_weight_all_hist_sub.pdf", bbox_inches="tight", dpi=300) # 保存图表为 PDF
plt.show() # 显示图表

#定义函数：计算正负边权重数量并绘制饼图
def edge_wt_pos_neg_ct_pie(edge_wt_df, color, title, path_to_file):
    """
    计算给定边权重数据中正权重和负权重的数量，并绘制饼图。

    Args:
        edge_wt_df (pd.DataFrame): 包含 'weight' 列的边权重数据框。
        color (list): 饼图扇形的颜色列表，第一个颜色对应正权重，第二个对应负权重。
        title (str): 饼图的标题。
        path_to_file (str): 保存饼图的路径和文件名 (PDF 格式)。

    Returns:
        tuple: (正权重数量, 负权重数量)
    """
    # 计算权重大于 0 的边的数量 (正连接)
    count_positive = (edge_wt_df['weight'] > 0).sum()
    # 计算权重小于 0 的边的数量 (负连接)
    count_negative = (edge_wt_df['weight'] < 0).sum()

    # 将正负计数转换为 Pandas Series，便于绘制饼图
    counts = pd.Series([count_positive, count_negative], index=['Postive', 'Negative'])

    plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
    # 绘制饼图：autopct 显示百分比，colors 设置颜色，startangle 设置起始角度
    counts.plot(kind='pie', autopct='%1.1f%%', colors=color, startangle=90, textprops={'fontsize': 12})
    plt.title(title, size = 14) # 设置标题
    plt.xlabel('') # 清空 X 轴标签
    plt.ylabel('') # 清空 Y 轴标签
    plt.savefig(path_to_file, bbox_inches="tight", dpi=600) # 保存图表
    plt.show() # 显示图表

    return (count_positive, count_negative) # 返回正负权重数量

# 使用上述函数绘制所有 OTU 边权重的饼图
# 返回的值 (13476, 3447)是正权重和负权重的数量
edge_wt_pos_neg_ct_pie(f_edge_wt_all, ['#F1C40F', '#F9E79F'], 'Edge weights (all fungal species)', "/mnt/d/study/master/Main_Figure_tables/Figure_5/f_edge_weight_all_pie.pdf")

# 将各个生态系统的边密度数据合并到一个数据框中
f_density_concat = pd.concat([f_density_jr, f_density_jj, f_density_tz, f_density_pa, f_density_g, f_density_n])
f_density_concat.index.names = ['group'] # 将索引命名为 'group'
f_density_concat = f_density_concat.reset_index() # 将索引转换为常规列
f_density_concat = f_density_concat.sort_values(by = 'edge density') # 根据边密度值排序

# 定义不同生态系统的颜色映射
group_colors = {
    'Jurong': '#45B39D',
    'Jingjiang': '#666666',
    'Tongzhou': '#1E90FF',
    'Pan\'an': '#CD6155',
    'Rhizosphere Soil': '#A0522D',
    'Bulb': '#AF7AC5'
}

# 绘制生态系统网络密度的柱状图
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
ax = sns.barplot(data = f_density_concat, x='group', y="edge density", hue='group', palette=group_colors, dodge=False) # 绘制柱状图
ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('') # 清空 Y 轴标签 (此处应是 'Edge Density')
ax.set_title('Fungal network density of populations and niches', size=13) # 设置标题
plt.xticks(fontsize=13, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和对齐方式
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_density_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

# 加载并处理每个生态系统的边权重数据
f_edge_wt_jr = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/JR/f_edge_weight.csv")
f_edge_wt_n = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/N/f_edge_weight.csv")
f_edge_wt_g = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/G/f_edge_weight.csv")
f_edge_wt_pa = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/PA/f_edge_weight.csv")
f_edge_wt_tz = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/TZ/f_edge_weight.csv")
f_edge_wt_jj = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/JJ/f_edge_weight.csv")

# 为每个生态系统绘制边权重饼图，并获取正负权重计数
f_pos_ct_jr, f_neg_ct_jr = edge_wt_pos_neg_ct_pie(f_edge_wt_jr, ['#45B39D', '#79cbbb'],
                                                      'Jurong',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/JR/f_edge_weight_jr_pie.pdf")

f_pos_ct_n, f_neg_ct_n = edge_wt_pos_neg_ct_pie(f_edge_wt_n, ['#AF7AC5', '#D2B4DE'],
                                                      'Bulb',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/N/f_edge_weight_n_pie.pdf")

f_pos_ct_g, f_neg_ct_g = edge_wt_pos_neg_ct_pie(f_edge_wt_g, ['#A0522D', '#bd866c'],
                                                      'Rhizosphere Soil',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/G/f_edge_weight_g_pie.pdf")

f_pos_ct_pa, f_neg_ct_pa = edge_wt_pos_neg_ct_pie(f_edge_wt_pa, ['#CD6155', '#E6B0AA'],
                                                      'Pan\'an',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/PA/f_edge_weight_pa_pie.pdf")

f_pos_ct_tz, f_neg_ct_tz = edge_wt_pos_neg_ct_pie(f_edge_wt_tz, ['#1E90FF', '#78bcff'],
                                                      'Tongzhou',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/TZ/f_edge_weight_tz_pie.pdf")

f_pos_ct_jj, f_neg_ct_jj = edge_wt_pos_neg_ct_pie(f_edge_wt_jj, ['#666666', '#D5D8DC'],
                                                      'Jingjiang',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/JJ/f_edge_weight_jj_pie.pdf")

# 对正负边权重数量执行 Fisher 精确检验
import rpy2.robjects as robjects
from rpy2.robjects import numpy2ri, default_converter
from rpy2.robjects.packages import importr

# 导入 R 的 'stats' 包，这里不需要立即激活转换器
stats = importr('stats')

# 构建一个 NumPy 数组，包含每个生态系统正负边权重的计数
f_m = np.array([[f_pos_ct_jr, f_neg_ct_jr],
                [f_pos_ct_jj, f_neg_ct_jj],
                [f_pos_ct_g, f_neg_ct_g],
                [f_pos_ct_pa, f_neg_ct_pa],
                [f_pos_ct_n, f_neg_ct_n],
                [f_pos_ct_tz, f_neg_ct_tz]])

# 使用上下文管理器显式激活 numpy 到 R 对象的转换
# 这种方式可以避免旧版本 activate() 方法带来的 DeprecationWarning 报错
with (default_converter + numpy2ri.converter).context():
    # 在这个缩进块内执行 Fisher 精确检验，并模拟 P 值（适用于 2xK 大尺寸列联表）
    res = stats.fisher_test(f_m, simulate_p_value=True)
    
    # 从返回的 R 对象中提取 P 值并打印
    # 注意：rpy2 返回的对象结构通常需要通过 [0][0] 索引取值
    print('p-value: {}'.format(res[0][0]))

# 为每个生态系统的网络中心性数据框添加 'group' 列，标记其所属的生态系统类别
f_metrics_jr['group'] = 'Jurong'
f_metrics_jj['group'] = 'Jingjiang'
f_metrics_tz['group'] = 'Tongzhou'
f_metrics_pa['group'] = 'Pan\'an'
f_metrics_g['group'] = 'Rhizosphere Soil'
f_metrics_n['group'] = 'Bulb'

# 将所有生态系统的网络中心性数据合并到一个数据框中，用于后续的比较分析
f_metrics_cancat = pd.concat([f_metrics_jr, f_metrics_jj, f_metrics_tz, f_metrics_pa, f_metrics_g, f_metrics_n])

#执行 Kruskal-Wallis 检验 

# 初始化字典用于存储 Kruskal-Wallis 检验的 P 值
kw_pval = {}
# 遍历需要进行检验的中心性指标：'degree', 'closeness', 'betweenness'
for i in ['degree', 'closeness', 'betweenness']:
    # 对每个指标执行 Kruskal-Wallis 检验，比较六个生态系统组的差异
    stat, p_value = kruskal(f_metrics_jr[i], f_metrics_jj[i], f_metrics_pa[i], f_metrics_tz[i], f_metrics_g[i], f_metrics_n[i])
    # 打印每个指标在不同生态系统中的中位数和 Kruskal-Wallis 检验的 P 值
    print(i, 'Jurong:', np.median(f_metrics_jr[i]),
          'Jingjiang:', np.median(f_metrics_jj[i]),
          'Pan\'an:', np.median(f_metrics_pa[i]),
           'Tongzhou:', np.median(f_metrics_tz[i]),
           'Rhizosphere Soil:', np.median(f_metrics_g[i]),
           'Bulb:', np.median(f_metrics_n[i]))
    kw_pval[i] = p_value # 将 P 值存储到字典中

# 输出示例 (Kruskal-Wallis 检验结果):
# degree forest: 27.0 Jingjiang: 25.0 Pan\'an: 24.0 Tongzhou: 25.0 Rhizosphere Soil: 27.0 steppe: 27.0
# closeness forest: 4.29367e-06 Jingjiang: 3.559657e-06 Pan\'an: 4.3339205e-06 Tongzhou: 3.043133e-06 Rhizosphere Soil: 4.223972e-06 steppe: 4.4570035e-06
# betweenness forest: 873.0 Jingjiang: 1027.0 Pan\'an: 794.25 Tongzhou: 1044.0 Rhizosphere Soil: 834.0 steppe: 871.5

# 定义用于绘图的生态系统颜色映射字典
group_colors = {
    'Jurong': '#45B39D',
    'Jingjiang': '#666666',
    'Tongzhou': '#1E90FF',
    'Pan\'an': '#CD6155',
    'Rhizosphere Soil': '#A0522D',
    'Bulb': '#AF7AC5'
}

# 定义箱线图的显示顺序
order=['Bulb', 'Pan\'an', 'Tongzhou', 'Jingjiang', 'Jurong', 'Rhizosphere Soil']
# 定义用于统计比较的组对
pairs=[
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Tongzhou'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Bulb'),
    ('Jurong', 'Tongzhou'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Rhizosphere Soil'),
    ('Jurong', 'Bulb'),
    ('Tongzhou', 'Jingjiang'),
    ('Tongzhou', 'Bulb'), # 调整：原始代码缺少 Tongzhou 与 Rhizosphere Soil 的比较，这里添加了
    ('Tongzhou', 'Rhizosphere Soil'),
    ('Jingjiang', 'Rhizosphere Soil'),
    ('Jingjiang', 'Bulb'),
    ('Rhizosphere Soil', 'Bulb'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
# 绘制箱线图：x 轴为生态系统组，y 轴为节点度，根据 group_colors 设置颜色，不显示异常值 (showfliers=False)
ax = sns.boxplot(data= f_metrics_cancat, x='group', y='degree', order=order,  palette=group_colors, showfliers=False)

# 使用 Annotator 在箱线图上添加统计显著性注释
annot = Annotator(ax, pairs, data=f_metrics_cancat, x='group', y='degree', order=order)
# 配置检验方法（曼-惠特尼 U 检验）和多重比较校正方法（Benjamini-Hochberg）
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用统计检验
annot.annotate() # 在图上添加注释（显著性星号或 P 值）

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Degree', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 在图的左上角添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.96, f'Kruskal-Wallis P = {kw_pval["degree"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和对齐方式
plt.ylim(-5, 130) # 设置 Y 轴的显示范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_degree_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

# 定义箱线图的显示顺序，基于 Closeness 值对生态系统进行排序
order=[ 'Bulb', 'Pan\'an', 'Jurong', 'Rhizosphere Soil', 'Tongzhou', 'Jingjiang']

# 定义用于 Mann-Whitney U 检验的组对。这些是所有可能的两两比较。
pairs=[
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Bulb'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Tongzhou'),
    ('Rhizosphere Soil', 'Jurong'),
    ('Rhizosphere Soil', 'Bulb'),
    ('Rhizosphere Soil', 'Jingjiang'),
    ('Rhizosphere Soil', 'Tongzhou'),
    ('Jurong', 'Bulb'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Tongzhou'),
    ('Bulb', 'Jingjiang'),
    ('Bulb', 'Tongzhou'),
    ('Jingjiang', 'Tongzhou'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小 (宽x高)
# 绘制箱线图：
# data: 合并后的网络中心性数据框
# x: 分组变量 'group' (生态系统类型)
# y: 数值变量 'closeness' (接近中心性)
# order: 指定 x 轴分组的显示顺序
# palette: 使用预定义的生态系统颜色
# showfliers=False: 不显示离群点 (outliers)
ax = sns.boxplot(data=f_metrics_cancat, x='group', y='closeness', order=order, palette=group_colors, showfliers=False)

# 使用 Annotator 在箱线图上添加统计显著性注释
annot = Annotator(ax, pairs, data=f_metrics_cancat, x='group', y='closeness', order=order)
# 配置统计检验方法为 'Mann-Whitney' (曼-惠特尼 U 检验，非参数两独立样本检验)
# 配置多重比较校正方法为 "Benjamini-Hochberg" (FDR 校正)
# verbose=2: 显示详细的检验过程和结果
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用配置的统计检验
annot.annotate() # 在图上绘制显著性注释（如星号或 P 值）

ax.set_xlabel('') # 清空 X 轴标签，因为信息已在刻度上显示
ax.set_ylabel('Closeness', size=13) # 设置 Y 轴标签及其字体大小
ax.set_title('', size=13) # 设置图表标题及其字体大小
# 在图表内部添加 Kruskal-Wallis 检验的 P 值
# transform=ax.transAxes 表示坐标是相对于坐标轴的比例 (0-1)
plt.text(0.05, 0.97, f'Kruskal-Wallis P = {kw_pval["closeness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和右对齐
plt.ylim(-1.6e-6, 1.7e-5) # 设置 Y 轴的显示范围，以更好地展示数据分布
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_closeness_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表为 PDF，紧密裁剪，高 DPI
plt.show() # 显示图表

# 定义箱线图的显示顺序，基于 Betweenness 值对生态系统进行排序
order = ['Bulb', 'Jingjiang', 'Tongzhou', 'Pan\'an', 'Jurong', 'Rhizosphere Soil']

# 定义用于 Mann-Whitney U 检验的组对
pairs=[
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Bulb'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Tongzhou'),
    ('Rhizosphere Soil', 'Jurong'),
    ('Rhizosphere Soil', 'Bulb'),
    ('Rhizosphere Soil', 'Jingjiang'),
    ('Rhizosphere Soil', 'Tongzhou'),
    ('Jurong', 'Bulb'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Tongzhou'),
    ('Bulb', 'Jingjiang'),
    ('Bulb', 'Tongzhou'),
    ('Jingjiang', 'Tongzhou'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
# 绘制箱线图：数据、x轴、y轴、顺序、颜色、不显示异常值
ax = sns.boxplot(data=f_metrics_cancat, x='group', y='betweenness', order=order, palette=group_colors, showfliers=False)

# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=f_metrics_cancat, x='group', y='betweenness', order=order)
# 配置检验方法和多重比较校正
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Betweenness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.97, f'Kruskal-Wallis P = {kw_pval["betweenness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(-500, 7000) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_betweenness_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

# 从所有 OTU 的网络中心性数据中，根据丰度/稀有/广谱/特有类群的索引筛选出相应的数据
f_metrics_abun = f_metrics_all[f_metrics_all.index.isin(f_df_abun.index)].copy()
f_metrics_abun['Ecotype'] = 'Cores' # 添加 'Ecotype' 列并赋值

f_metrics_rare = f_metrics_all[f_metrics_all.index.isin(f_df_rare.index)].copy()
f_metrics_rare['Ecotype'] = 'Uniques'

f_metrics_gen = f_metrics_all[f_metrics_all.index.isin(f_df_gen.index)].copy()
f_metrics_gen['Ecotype'] = 'Generalists'

f_metrics_spe = f_metrics_all[f_metrics_all.index.isin(f_df_spe.index)].copy()
f_metrics_spe['Ecotype'] = 'Specialists'

# 将所有生态类型的节点中心性数据合并到一个数据框中，用于后续的比较分析
f_metrics_ecotype = pd.concat([f_metrics_abun, f_metrics_rare, f_metrics_gen, f_metrics_spe])

# 初始化字典用于存储 Kruskal-Wallis 检验的 P 值
kw_pval = {}
# 遍历需要进行检验的中心性指标
for i in ['degree', 'closeness', 'betweenness']:
    # 对每个指标执行 Kruskal-Wallis 检验，比较四个生态类型组的差异
    stat, p_value = kruskal(f_metrics_abun[i], f_metrics_rare[i], f_metrics_gen[i], f_metrics_spe[i])
    # 打印每个指标在不同生态类型中的中位数和 P 值
    print(i, 'abundant:', np.median(f_metrics_abun[i]),
          'rare:', np.median(f_metrics_rare[i]),
          'generalist:', np.median(f_metrics_gen[i]),
          'specialist:', np.median(f_metrics_spe[i]))
    kw_pval[i] = p_value # 将 P 值存储到字典中

# 输出示例 (Kruskal-Wallis 检验结果):
# degree abundant: 49.0 rare: 53.0 generalist: 54.5 specialist: 54.0
# closeness abundant: 8.548408e-07 rare: 9.008329e-07 generalist: 8.993396000000001e-07 specialist: 8.638988e-07
# betweenness abundant: 1970.0 rare: 2626.0 generalist: 2245.0 specialist: 1886.0

# 定义箱线图的显示顺序（生态类型）
order=['Cores', 'Uniques', 'Generalists', 'Specialists']

# 定义不同生态类型的颜色映射
group_colors = {
    'Cores': '#5499C7', # 蓝色
    'Uniques': '#D4AC0D',     # 黄色
    'Generalists': '#EC7063',   # 红色
    'Specialists': '#45B39D'    # 绿色
}

# 定义用于 Mann-Whitney U 检验的组对 (所有两两比较)
pairs=[
    ('Cores', 'Uniques'),
    ('Cores', 'Generalists'),
    ('Cores', 'Specialists'),
    ('Uniques', 'Generalists'),
    ('Uniques', 'Specialists'),
    ('Generalists', 'Specialists'),
]

plt.rcParams["figure.figsize"] = (3, 4) # 设置图表大小 (宽x高)，相对更窄

#绘制节点度（Degree）箱线图 (针对生态类型) 
ax = sns.boxplot(data=f_metrics_ecotype, x='Ecotype', y='degree', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=f_metrics_ecotype, x='Ecotype', y='degree', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Degree', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["degree"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(0, 150) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_degree_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点中介中心性（Betweenness）箱线图 (针对生态类型) 
ax = sns.boxplot(data=f_metrics_ecotype, x='Ecotype', y='betweenness', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=f_metrics_ecotype, x='Ecotype', y='betweenness', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Betweenness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["betweenness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(-2000, 32000) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_betweenness_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点接近中心性（Closeness）箱线图 (针对生态类型) 
ax = sns.boxplot(data=f_metrics_ecotype, x='Ecotype', y='closeness', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=f_metrics_ecotype, x='Ecotype', y='closeness', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Closeness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["closeness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(0.5e-6, 3e-6) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_closeness_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#网络节点指标比较 (针对门 Phyla) 
# 将所有 OTU 的网络中心性指标数据 (f_metrics_all) 与分类学信息 (taxon) 进行合并。
# 合并依据是两个数据框的索引 ('left_index=True', 'right_index=True')。
# 只选择 taxon 数据框中的 'Phylum' 列进行合并，以便将每个 OTU 与其所属的门关联起来。
f_metrics_phyla = pd.merge(f_metrics_all, f_taxon[['Phylum']], left_index=True, right_index=True)

# 按 'Phylum' 列进行分组，并计算每个门的平均节点指标（degree, closeness, betweenness）。
# 这将用于确定绘图时门的排序。
f_metrics_phyla_mean = f_metrics_phyla.groupby('Phylum').mean()

#绘制节点度（Degree）的线点图 (Pointplot) 
# 根据平均度值对门进行降序排序，并获取排序后的门名称列表。
f_order_degree = f_metrics_phyla_mean.sort_values(by='degree', ascending=False).index.to_list()

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小 (宽x高)，垂直方向相对更长，适合显示多个门
# 绘制点图：
# x='degree': X 轴显示节点度
# y='Phylum': Y 轴显示门名称
# data: 合并了门信息的网络中心性数据
# order: 指定 Y 轴上门的显示顺序
# join=False: 不连接点，只显示点和误差线
# capsize=0.2: 误差线帽的宽度
# errwidth=1: 误差线的宽度
# color='#DC7633': 设置点的颜色
sns.pointplot(x='degree', y='Phylum', data=f_metrics_phyla, order=f_order_degree, join=False, capsize=0.2, errwidth=1, color='#DC7633')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=11) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签及其字体大小
plt.xlabel('Degree', size=14) # 设置 X 轴标签及其字体大小
plt.title('', size=14) # 设置图表标题及其字体大小
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_degree_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表为 PDF，紧密裁剪，高 DPI
plt.show() # 显示图表

#绘制节点接近中心性（Closeness）的线点图 (Pointplot) 
# 根据平均接近中心性值对门进行降序排序，并获取排序后的门名称列表。
f_order_closeness = f_metrics_phyla_mean.sort_values(by='closeness', ascending=False).index.to_list()

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小
# 绘制点图：x 轴为接近中心性，y 轴为门，使用指定数据、顺序和颜色
sns.pointplot(x='closeness', y='Phylum', data=f_metrics_phyla, order=f_order_closeness, join=False, capsize=0.2, errwidth=1, color='#BB8FCE')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=11) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签
plt.xlabel('Closeness', size=14) # 设置 X 轴标签
plt.title('', size=14) # 设置图表标题
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_closeness_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点中介中心性（Betweenness）的线点图 (Pointplot) 
# 根据平均中介中心性值对门进行降序排序，并获取排序后的门名称列表。
f_order_betweenness = f_metrics_phyla_mean.sort_values(by='betweenness', ascending=False).index.to_list()

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小
# 绘制点图：x 轴为中介中心性，y 轴为门，使用指定数据、顺序和颜色
sns.pointplot(x='betweenness', y='Phylum', data=f_metrics_phyla, order=f_order_betweenness, join=False, capsize=0.2, errwidth=1, color='#73C6B6')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=11) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签
plt.xlabel('Betweenness', size=14) # 设置 X 轴标签
plt.title('', size=14) # 设置图表标题
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/f_network_betweenness_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表






#跨界网络其他
import pandas as pd
import os # 用于操作系统相关功能
import scipy.stats as st # 用于统计检验
import numpy as np # 用于数值计算
import statsmodels.stats.multitest as smt # 用于多重检验校正
import seaborn as sns # 用于绘制统计图形
import matplotlib.pyplot as plt # 用于创建可视化
from collections import defaultdict # 用于创建具有默认值的字典
from scipy.stats import kruskal # 科鲁斯卡尔-沃利斯检验
from statannotations.Annotator import Annotator # 用于在 seaborn 图上添加统计显著性注释
#  辅助函数：计算生态位宽度 (Bj) 
def calculate_levins_bj(df):
    """
    计算每个物种的 Levin's 生态位宽度 Bj。
    
    公式: Bj = 1 / sum(Pij^2)
    其中 Pij 为物种 j 在样本 i 中的相对丰度。
    
    参数:
        df (pd.DataFrame): 行 = OTU/物种, 列 = 样本, 值为相对丰度
    
    返回:
        pd.Series: 每个物种对应的 Bj 值
    """
    matrix = df.astype(float)
    
    # 对每个物种（行），跨所有样本计算 Pij² 的加和
    sum_sq = (matrix ** 2).sum(axis=1)
    
    # 避免除以零，将0替换为NaN后再填回0
    bj = 1.0 / sum_sq.replace(0, np.nan)
    
    return bj.fillna(0)


def classify_generalist_specialist(
    relative_abundance_matrix,
    num_permutations=1000,
    seed=42
):
    """
    基于生态位宽度（Levin's Bj）的置换检验，将物种分类为：
    广适种（Generalist）、专适种（Specialist）或中性种（Neutral）。
    
    分类标准：
        Generalist : 观测 Bj 显著大于零分布（p_generalist < 0.05）
        Specialist : 观测 Bj 显著小于零分布（p_specialist < 0.05）
        Neutral    : 其余物种
    
    参数:
        relative_abundance_matrix (pd.DataFrame):
            行 = OTU/物种, 列 = 样本/生境, 值为相对丰度
        num_permutations (int):
            置换次数，默认 1000 次
        seed (int):
            随机种子，保证结果可重复，默认 42
    
    返回:
        pd.DataFrame: index = 物种名，包含以下列：
            - Bj_Observed  : 观测生态位宽度
            - Bj_Expected  : 零分布的平均生态位宽度
            - Bj_Std       : 零分布的标准差
            - Z_Score      : 标准化得分，衡量偏离零分布的程度
            - p_generalist : 右尾 p 值（观测值是否显著偏大）
            - p_specialist : 左尾 p 值（观测值是否显著偏小）
            - Ecotype      : 分类结果（Generalist / Specialist / Neutral）
    """
    # ---------- 1. 数据准备 ----------
    mat = relative_abundance_matrix.fillna(0).astype(float)
    species = mat.index
    n_species, n_samples = mat.shape
    
    print(f"输入矩阵维度: {n_species} 个物种 × {n_samples} 个样本")

    # ---------- 2. 计算观测 Bj ----------
    observed_bj = calculate_levins_bj(mat)

    # ---------- 3. 置换检验，构建零分布 ----------
    # 使用独立随机生成器，避免影响全局随机状态
    rng = np.random.default_rng(seed=seed)
    
    # 预分配结果数组：行 = 物种，列 = 每次置换的 Bj
    permuted_bj_array = np.zeros((n_species, num_permutations), dtype=float)

    for i in range(num_permutations):
        # 将整个相对丰度矩阵展平后随机打乱
        # 目的：彻底破坏物种与样本之间的对应关系，生成零假设下的分布
        shuffled_flat = mat.values.flatten().copy()
        rng.shuffle(shuffled_flat)
        
        # 重塑回原始矩阵形状，计算该置换下每个物种的 Bj
        shuffled_df = pd.DataFrame(
            shuffled_flat.reshape(n_species, n_samples),
            index=species,
            columns=mat.columns
        )
        permuted_bj_array[:, i] = calculate_levins_bj(shuffled_df).values

    # ---------- 4. 零分布统计量 ----------
    # 零分布的均值：代表随机期望下的 Bj
    expected_bj = permuted_bj_array.mean(axis=1)
    
    # 零分布的标准差（使用无偏估计 ddof=1）
    std_null = permuted_bj_array.std(axis=1, ddof=1)
    
    # Z-score：观测值偏离零分布均值的标准化程度
    # 标准差为 0 时（所有置换结果相同）设为 NaN，避免除以零
    z_score = (observed_bj.values - expected_bj) / np.where(
        std_null == 0, np.nan, std_null
    )

    # ---------- 5. 计算置换 p 值 ----------
    # 使用 +1 修正（Phipson & Smyth 2010），避免 p 值为 0 的情况
    obs_val = observed_bj.values.reshape(-1, 1)
    
    # 右尾 p 值：零分布中 Bj >= 观测值的比例
    # p 值越小，说明观测 Bj 显著偏大 → 倾向于 Generalist
    p_gen = (
        (permuted_bj_array >= obs_val).sum(axis=1) + 1
    ) / (num_permutations + 1)
    
    # 左尾 p 值：零分布中 Bj <= 观测值的比例
    # p 值越小，说明观测 Bj 显著偏小 → 倾向于 Specialist
    p_spe = (
        (permuted_bj_array <= obs_val).sum(axis=1) + 1
    ) / (num_permutations + 1)

    # ---------- 6. 汇总结果 ----------
    results = pd.DataFrame({
        'Bj_Observed' : observed_bj.values,
        'Bj_Expected' : expected_bj,
        'Bj_Std'      : std_null,
        'Z_Score'     : z_score,
        'p_generalist': p_gen,
        'p_specialist': p_spe,
    }, index=species)

    # ---------- 7. 物种分类 ----------
    results['Ecotype'] = 'Neutral'
    results.loc[results['p_generalist'] < 0.05, 'Ecotype'] = 'Generalist'
    results.loc[results['p_specialist'] < 0.05, 'Ecotype'] = 'Specialist'

    # ---------- 8. 输出统计摘要 ----------
    print("\n物种分类统计:")
    print(results['Ecotype'].value_counts().to_string())

    return results



# 打印当前 seaborn 版本，确认环境配置
print(sns.__version__)
# 输出示例: 0.11.2 (表示当前运行环境中的 seaborn 版本)
#细菌
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
bacteria_ASV = pd.read_csv("/mnt/d/study/master/meiji/bacteria_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 bacteria_ASV DataFrame 中选择指定的列
b_ASV = bacteria_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
b_ASV.set_index(keys="ASV", inplace=True)
b_tax = bacteria_ASV.iloc[:, 1:9].copy()
b_tax.set_index(keys="ASV", inplace=True)
print(len(b_ASV))
# 计算 ASV 相对丰度
b_ASV_df = b_ASV.copy()

#按种
b_species = b_ASV.join(b_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
b_species = b_species.groupby('Species')[sample_ids].sum()
#排序
b_species = b_species.sort_index()
b_species_df = b_species.copy()
print(len(b_species_df))
# 将当前的 ASV 索引重置为一个普通列。
b_tax_reset_index = b_tax.reset_index()
# 将 'Species' 列设置为新的索引。
b_tax_species = b_tax_reset_index.groupby('Species').first()
b_tax_species = b_tax_species.drop(columns=['ASV'])
b_tax_species = b_tax_species.sort_index()
# 进行替换操作
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__unclassified_k__norank_d__Bacteria', 'Unclassified')
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__SAR324_cladeMarine_group_B', 'SAR324')
# 去除 'p__' 前缀
b_tax_species['Phylum'] = b_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
b_unique_species_names = b_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
b_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(b_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 b_species 的索引
b_species = b_species.rename(index= b_species_to_numeric_id)
# 转换 b_tax_species 的索引
b_tax_species = b_tax_species.rename(index= b_species_to_numeric_id)
b_species_df = b_species.copy()
print(len(b_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
b_species_rel = b_species_df.div(b_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
b_rel_long = b_species_rel.stack().reset_index()
b_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
b_rel_long['Origin'] = b_rel_long['Sample'].map(sample_origins_map)
b_rel_long['Niche'] = b_rel_long['Sample'].map(sample_niches_map)
b_species_total_counts = b_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
b_total_counts_across_all_species = b_species_total_counts.sum() # 计算所有 species 的总丰度
b_species_relative_abundance = b_species_total_counts / b_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
b_species_above_overall_threshold = b_species_relative_abundance[b_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
b_rel_long_filtered_by_overall_abundance = b_rel_long[b_rel_long['species'].isin(b_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
b_species_origin_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
b_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
b_species_in_origin = b_species_origin_mean_filtered[b_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
b_species_origin_count = b_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
b_species_niche_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
b_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
b_species_in_niche = b_species_niche_mean_filtered[b_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
b_species_niche_count = b_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
b_species_counts_combined = pd.merge(b_species_origin_count, b_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 3
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
b_core_species_names = b_species_counts_combined[
    (b_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (b_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
b_core_abundance = b_species_rel.loc[b_core_species_names]
b_core_species_raw_counts = b_species.loc[b_core_species_names]
print(len(b_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
b_median_relative_abundance = b_species_relative_abundance.median()
print(b_median_relative_abundance)
# 定义稀有类群的丰度阈值 
b_rare_threshold = 0.00001
# 筛选出稀有 species 
b_rare_species = b_species_relative_abundance[b_species_relative_abundance < b_rare_threshold]
# 获取稀有 species 的名称（索引）
b_rare_species_names = b_rare_species.index.tolist()
len(b_rare_species_names)
b_species_db_transposed = b_species_df.T 
b_speciesdfmelt = b_species_db_transposed.stack().reset_index()# 长宽表格转换
b_speciesdfmelt.columns = ['Sample', 'species', 'value']
b_speciesdfmelt = b_speciesdfmelt[b_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
b_control_species = b_speciesdfmelt.copy()
b_control_species['Group'] = b_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
b_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
b_controlspecies_unique = b_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
b_controlspecies_unique = b_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
b_controlspecies_unique = b_controlspecies_unique[b_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
b_unique_species_names_single_group = b_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
b_set_rare_species = set(b_rare_species_names)
b_set_unique_species_single_group = set(b_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
b_rare_unique_species_names = list(b_set_rare_species.intersection(b_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
b_rare_unique_abundance = b_species_df.loc[b_rare_unique_species_names]
b_rare_unique_species_raw_counts = b_species.loc[b_rare_unique_species_names]
print(len(b_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度置换检验与 Z-score 计算
num_permutations = 1000 
b_niche_breadth_results = classify_generalist_specialist(
    b_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (含 Z-score, 前5个 species) ")
print(b_niche_breadth_results.head())
b_generalists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
b_specialists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(b_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(b_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_generalists_species:
    print("无广适类群 species 数据。")
else:
    b_generalists_raw_counts = b_species.loc[b_generalists_species]
    b_generalists_abundance = b_species_rel.loc[b_generalists_species]
    print(b_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_specialists_species:
    print("无专适类群 species 数据。")
else:
    b_specialists_raw_counts = b_species.loc[b_specialists_species]
    b_specialists_abundance = b_species_rel.loc[b_specialists_species]
print(b_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

#真菌
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
fungi_ASV = pd.read_csv("/mnt/d/study/master/meiji/fungi_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 fungi_ASV DataFrame 中选择指定的列
f_ASV = fungi_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
f_ASV.set_index(keys="ASV", inplace=True)
f_tax = fungi_ASV.iloc[:, 1:9].copy()
f_tax.set_index(keys="ASV", inplace=True)
print(len(f_ASV))
# 计算 ASV 相对丰度
f_ASV_df = f_ASV.copy()

#按种
f_species = f_ASV.join(f_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
f_species = f_species.groupby('Species')[sample_ids].sum()
#排序
f_species = f_species.sort_index()
# 将当前的 ASV 索引重置为一个普通列。
f_tax_reset_index = f_tax.reset_index()
# 将 'Species' 列设置为新的索引。
f_tax_species = f_tax_reset_index.groupby('Species').first()
f_tax_species = f_tax_species.drop(columns=['ASV'])
f_tax_species = f_tax_species.sort_index()
# 进行替换操作
f_tax_species['Phylum'] = f_tax_species['Phylum'].replace('p__unclassified_k__Fungi', 'Unclassified')
# 去除 'p__' 前缀
f_tax_species['Phylum'] = f_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
f_unique_species_names = f_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
f_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(f_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 f_species 的索引
f_species = f_species.rename(index= f_species_to_numeric_id)
# 转换 f_tax_species 的索引
f_tax_species = f_tax_species.rename(index= f_species_to_numeric_id)
f_species_df = f_species.copy()
print(len(f_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
f_species_rel = f_species_df.div(f_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
f_rel_long = f_species_rel.stack().reset_index()
f_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
f_rel_long['Origin'] = f_rel_long['Sample'].map(sample_origins_map)
f_rel_long['Niche'] = f_rel_long['Sample'].map(sample_niches_map)
f_species_total_counts = f_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
f_total_counts_across_all_species = f_species_total_counts.sum() # 计算所有 species 的总丰度
f_species_relative_abundance = f_species_total_counts / f_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
f_species_above_overall_threshold = f_species_relative_abundance[f_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
f_rel_long_filtered_by_overall_abundance = f_rel_long[f_rel_long['species'].isin(f_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
f_species_origin_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
f_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
f_species_in_origin = f_species_origin_mean_filtered[f_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
f_species_origin_count = f_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
f_species_niche_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
f_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
f_species_in_niche = f_species_niche_mean_filtered[f_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
f_species_niche_count = f_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
f_species_counts_combined = pd.merge(f_species_origin_count, f_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 3
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
f_core_species_names = f_species_counts_combined[
    (f_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (f_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
f_core_abundance = f_species_rel.loc[f_core_species_names]
f_core_species_raw_counts = f_species.loc[f_core_species_names]
print(len(f_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
f_median_relative_abundance = f_species_relative_abundance.median()
print(f_median_relative_abundance)
# 定义稀有类群的丰度阈值 
f_rare_threshold = 0.000008
# 筛选出稀有 species 
f_rare_species = f_species_relative_abundance[f_species_relative_abundance < f_rare_threshold]
# 获取稀有 species 的名称（索引）
f_rare_species_names = f_rare_species.index.tolist()
len(f_rare_species_names)
f_species_df_transposed = f_species_df.T 
f_speciesdfmelt = f_species_df_transposed.stack().reset_index()# 长宽表格转换
f_speciesdfmelt.columns = ['Sample', 'species', 'value']
f_speciesdfmelt = f_speciesdfmelt[f_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
f_control_species = f_speciesdfmelt.copy()
f_control_species['Group'] = f_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
f_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
f_controlspecies_unique = f_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
f_controlspecies_unique = f_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
f_controlspecies_unique = f_controlspecies_unique[f_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
f_unique_species_names_single_group = f_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
f_set_rare_species = set(f_rare_species_names)
f_set_unique_species_single_group = set(f_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
f_rare_unique_species_names = list(f_set_rare_species.intersection(f_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
f_rare_unique_abundance = f_species_df.loc[f_rare_unique_species_names]
f_rare_unique_species_raw_counts = f_species.loc[f_rare_unique_species_names]
print(len(f_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度置换检验与 Z-score 计算
num_permutations = 1000 
f_niche_breadth_results = classify_generalist_specialist(
    f_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (含 Z-score, 前5个 species) ")
print(f_niche_breadth_results.head())
f_generalists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
f_specialists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(f_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(f_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_generalists_species:
    print("无广适类群 species 数据。")
else:
    f_generalists_raw_counts = f_species.loc[f_generalists_species]
    f_generalists_abundance = f_species_rel.loc[f_generalists_species]
    print(f_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_specialists_species:
    print("无专适类群 species 数据。")
else:
    f_specialists_raw_counts = f_species.loc[f_specialists_species]
    f_specialists_abundance = f_species_rel.loc[f_specialists_species]
print(f_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

# 加载不同类群（丰度、稀有、广谱、特有）的相对丰度数据
# 合并 Cores 微生物的丰度数据
df_abun = pd.concat([b_core_abundance, f_core_abundance], axis=0)
# 合并 Rare 和 Uniques 微生物的丰度数据
df_rare = pd.concat([b_rare_unique_abundance, f_rare_unique_abundance], axis=0)
# 合并 Generalists 微生物的丰度数据
df_gen = pd.concat([b_generalists_abundance, f_generalists_abundance], axis=0)
# 合并 Specialists 微生物的丰度数据
df_spe = pd.concat([b_specialists_abundance, f_specialists_abundance], axis=0)
# 加载 OTU 的分类学信息
taxon = pd.concat([b_tax_species, f_tax_species], axis=0)

# 加载不同生态系统（和总览）的边密度 (edge density) 数据
density_jr = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_jr/edge_density.csv")
density_jj = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_jj/edge_density.csv")
density_tz = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_tz/edge_density.csv")
density_pa = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_pa/edge_density.csv")
density_g = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_g/edge_density.csv")
density_n = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_n/edge_density.csv")

# 加载不同生态系统（和总览）的网络中心性指标 (network metrics) 数据
# 定义文件路径和对应的变量名后缀
file_paths = {
    "all": "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/nodeG_plot.csv",
    "jr": "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_jr/nodeG_plot.csv",
    "jj": "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_jj/nodeG_plot.csv",
    "tz": "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_tz/nodeG_plot.csv",
    "pa": "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_pa/nodeG_plot.csv",
    "g": "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_g/nodeG_plot.csv",
    "n": "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_n/nodeG_plot.csv",
}
# 存储处理后的 DataFrames
processed_metrics = {}
for suffix, path in file_paths.items():
    try:
        # 读取 CSV 文件，并将第一列设置为索引
        df = pd.read_csv(path, index_col=0)
        # 检查所需列是否存在
        required_cols = ['igraph.degree', 'igraph.closeness', 'igraph.betweenness']
        # 筛选出实际存在且我们需要保留的列
        cols_to_keep = [col for col in required_cols if col in df.columns]
        # 如果所有所需列都不存在，则跳过或发出警告
        if not cols_to_keep:
            print(f"Warning: No required centrality columns found in {path}. Skipping this file.")
            continue 
        # 只保留指定的列
        df_filtered = df[cols_to_keep].copy() # 使用 .copy() 避免 SettingWithCopyWarning
        # 重命名列
        new_column_names = {
            'igraph.degree': 'degree',
            'igraph.closeness': 'closeness',
            'igraph.betweenness': 'betweenness'
        }
        df_renamed = df_filtered.rename(columns=new_column_names)
        # 重命名索引（如果索引有名称的话，通常读取CSV时第一列作为索引后，索引名会是该列的原名，我们统一改为'id'）
        df_renamed.index.name = 'id'
        # 将处理后的 DataFrame 存储到字典中，以原始的变量名后缀作为键
        processed_metrics[f'metrics_{suffix}'] = df_renamed
        print(f"Successfully processed metrics_{suffix}")
    except FileNotFoundError:
        print(f"Error: File not found at {path}. Skipping this file.")
    except Exception as e:
        print(f"An error occurred while processing {path}: {e}")
# 现在，你可以通过字典访问这些处理后的 DataFrame，例如：
metrics_all = processed_metrics.get('metrics_all')
metrics_jr = processed_metrics.get('metrics_jr')
metrics_jj = processed_metrics.get('metrics_jj')
metrics_tz = processed_metrics.get('metrics_tz')
metrics_pa = processed_metrics.get('metrics_pa')
metrics_g = processed_metrics.get('metrics_g')
metrics_n = processed_metrics.get('metrics_n')

# 为边密度数据框的索引重新命名，使其代表不同的生态系统类别
density_jr.index = ['Jurong']
density_jj.index = ['Jingjiang']
density_tz.index = ['Tongzhou']
density_pa.index = ['Pan\'an']
density_g.index = ['Rhizosphere Soil']
density_n.index = ['Bulb']

# 为网络中心性指标数据框的索引添加前缀 "OTU_"
# 这有助于确保索引格式一致性，例如 '123' 变为 'OTU_123'
prefix = 'species_'
metrics_all.index = metrics_all.index.astype(str).map(lambda x: prefix + x)
metrics_jr.index = metrics_jr.index.astype(str).map(lambda x: prefix + x)
metrics_jj.index = metrics_jj.index.astype(str).map(lambda x: prefix + x)
metrics_tz.index = metrics_tz.index.astype(str).map(lambda x: prefix + x)
metrics_pa.index = metrics_pa.index.astype(str).map(lambda x: prefix + x)
metrics_g.index = metrics_g.index.astype(str).map(lambda x: prefix + x)
metrics_n.index = metrics_n.index.astype(str).map(lambda x: prefix + x)

#定义函数：加载和熔化边权重数据
def load_melt_edge_wt(path_to_the_file):
    """
    加载 SpiecEasi 生成的边权重矩阵文件，并将其转换为长格式（'熔化'）。
    同时对数据进行清洗，移除自连接和权重为零的边。

    Args:
        path_to_the_file (str): 包含边权重数据的 CSV 文件的完整路径。

    Returns:
        pd.DataFrame: 经过处理的边权重数据框，包含 'Sample i'（起始 OTU），
                      'Sample j'（结束 OTU）和 'weight'（边权重）三列。
    """
    # 读取 CSV 文件，并将第一列作为数据框的索引
    edge_wt = pd.read_csv(path_to_the_file, index_col=0)
    # 为索引添加 'V' 前缀，以匹配通常的节点命名约定（例如 V1, V2），方便后续处理
    edge_wt.index = edge_wt.index.astype(str).map(lambda x: 'V' + x)

    # 提取矩阵的上三角形部分（包括对角线），其他部分填充为 NaN
    # 这样做是为了避免重复的边 (因为矩阵是对称的)
    edge_wt_melted = edge_wt.where(np.triu(np.ones(edge_wt.shape)).astype(bool))
    # 将宽格式的数据框“熔化”为长格式。'ignore_index=False' 保留原索引作为一列，
    # 然后 .dropna() 移除所有包含 NaN 的行（即上三角形之外的冗余部分）
    edge_wt_melted = edge_wt_melted.melt(ignore_index=False).dropna()

    # 重命名索引和列，使其更具描述性
    edge_wt_melted.index.name = 'Sample i'
    edge_wt_melted.columns = ['Sample j', 'weight']
    # 将索引 'Sample i' 变为常规列
    edge_wt_melted = edge_wt_melted.reset_index()

    # 过滤数据：移除自身连接的行，即起始 OTU 和结束 OTU 相同的行
    edge_wt_melted_filtered = edge_wt_melted[edge_wt_melted['Sample i'] != edge_wt_melted['Sample j']]
    # 过滤数据：移除权重为 0 的行，因为在网络分析中它们不代表实际连接
    edge_wt_melted_filtered = edge_wt_melted_filtered[edge_wt_melted_filtered['weight'] != 0]

    return edge_wt_melted_filtered

# 加载并处理所有 OTU 的边权重数据
edge_wt_all = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/cor_matrix.csv")

# 绘制所有 OTU 边权重的直方图
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
sns.histplot(data= edge_wt_all, x='weight', color="#F1C40F", kde=True) # 绘制直方图，带核密度估计曲线
plt.xlabel("") # 清空 X 轴标签
plt.ylabel("Count", fontsize=12) # 设置 Y 轴标签
plt.title("Inter-domain edge weights (all species)", fontsize=13) # 设置图表标题
plt.xlim(-1, 1)
plt.savefig("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/edge_weight_all_hist.pdf", bbox_inches="tight", dpi=600) # 保存图表为 PDF
plt.show() # 显示图表

# 绘制所有 OTU 边权重的局部直方图 (聚焦于权重接近 0 的部分)
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
sns.histplot(data= edge_wt_all, x='weight', color="#F1C40F", kde=True) # 绘制直方图，带核密度估计曲线
plt.xlabel("") # 清空 X 轴标签
plt.ylabel("Count", fontsize=12) # 设置 Y 轴标签
plt.title("Inter-domain edge weights (all species)", fontsize=13) # 设置图表标题
plt.xlim(0.8, 1) # 设置 X 轴的显示范围，聚焦于 -0.1 到 0.1 之间
plt.savefig("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/edge_weight_all_hist_sub.pdf", bbox_inches="tight", dpi=300) # 保存图表为 PDF
plt.show() # 显示图表

#定义函数：计算正负边权重数量并绘制饼图
def edge_wt_pos_neg_ct_pie(edge_wt_df, color, title, path_to_file):
    """
    计算给定边权重数据中正权重和负权重的数量，并绘制饼图。

    Args:
        edge_wt_df (pd.DataFrame): 包含 'weight' 列的边权重数据框。
        color (list): 饼图扇形的颜色列表，第一个颜色对应正权重，第二个对应负权重。
        title (str): 饼图的标题。
        path_to_file (str): 保存饼图的路径和文件名 (PDF 格式)。

    Returns:
        tuple: (正权重数量, 负权重数量)
    """
    # 计算权重大于 0 的边的数量 (正连接)
    count_positive = (edge_wt_df['weight'] > 0).sum()
    # 计算权重小于 0 的边的数量 (负连接)
    count_negative = (edge_wt_df['weight'] < 0).sum()

    # 将正负计数转换为 Pandas Series，便于绘制饼图
    counts = pd.Series([count_positive, count_negative], index=['Postive', 'Negative'])

    plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
    # 绘制饼图：autopct 显示百分比，colors 设置颜色，startangle 设置起始角度
    counts.plot(kind='pie', autopct='%1.1f%%', colors=color, startangle=90, textprops={'fontsize': 12})
    plt.title(title, size = 14) # 设置标题
    plt.xlabel('') # 清空 X 轴标签
    plt.ylabel('') # 清空 Y 轴标签
    plt.savefig(path_to_file, bbox_inches="tight", dpi=600) # 保存图表
    plt.show() # 显示图表

    return (count_positive, count_negative) # 返回正负权重数量

# 使用上述函数绘制所有 OTU 边权重的饼图
# 返回的值 (13476, 3447)是正权重和负权重的数量
edge_wt_pos_neg_ct_pie(edge_wt_all, ['#F1C40F', '#F9E79F'], 'Inter-domain edge weights (all species)', "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/edge_weight_all_pie.pdf")

# 将各个生态系统的边密度数据合并到一个数据框中
density_concat = pd.concat([density_jr, density_jj, density_tz, density_pa, density_g, density_n])
density_concat.index.names = ['group'] # 将索引命名为 'group'
density_concat = density_concat.reset_index() # 将索引转换为常规列
density_concat = density_concat.sort_values(by = 'edge density') # 根据边密度值排序

# 定义不同生态系统的颜色映射
group_colors = {
    'Jurong': '#45B39D',
    'Jingjiang': '#666666',
    'Tongzhou': '#1E90FF',
    'Pan\'an': '#CD6155',
    'Rhizosphere Soil': '#A0522D',
    'Bulb': '#AF7AC5'
}

# 绘制生态系统网络密度的柱状图
plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
ax = sns.barplot(data = density_concat, x='group', y="edge density", hue='group', palette=group_colors, dodge=False) # 绘制柱状图
ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('') # 清空 Y 轴标签 (此处应是 'Edge Density')
ax.set_title('Inter-domain network density of populations and niches', size=13) # 设置标题
plt.xticks(fontsize=13, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和对齐方式
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/density_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

# 加载并处理每个生态系统的边权重数据
edge_wt_jr = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_jr/cor_matrix.csv")
edge_wt_n = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_n/cor_matrix.csv")
edge_wt_g = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_g/cor_matrix.csv")
edge_wt_pa = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_pa/cor_matrix.csv")
edge_wt_tz = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_tz/cor_matrix.csv")
edge_wt_jj = load_melt_edge_wt("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_jj/cor_matrix.csv")

# 为每个生态系统绘制边权重饼图，并获取正负权重计数
pos_ct_jr, neg_ct_jr = edge_wt_pos_neg_ct_pie(edge_wt_jr, ['#45B39D', '#79cbbb'],
                                                      'Jurong',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_jr/edge_weight_jr_pie.pdf")

pos_ct_n, neg_ct_n = edge_wt_pos_neg_ct_pie(edge_wt_n, ['#AF7AC5', '#D2B4DE'],
                                                      'Bulb',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_n/edge_weight_n_pie.pdf")

pos_ct_g, neg_ct_g = edge_wt_pos_neg_ct_pie(edge_wt_g, ['#A0522D', '#bd866c'],
                                                      'Rhizosphere Soil',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_g/edge_weight_g_pie.pdf")

pos_ct_pa, neg_ct_pa = edge_wt_pos_neg_ct_pie(edge_wt_pa, ['#CD6155', '#E6B0AA'],
                                                      'Pan\'an',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_pa/edge_weight_pa_pie.pdf")

pos_ct_tz, neg_ct_tz = edge_wt_pos_neg_ct_pie(edge_wt_tz, ['#1E90FF', '#78bcff'],
                                                      'Tongzhou',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_tz/edge_weight_tz_pie.pdf")

pos_ct_jj, neg_ct_jj = edge_wt_pos_neg_ct_pie(edge_wt_jj, ['#666666', '#D5D8DC'],
                                                      'Jingjiang',
                                                      "/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_jj/edge_weight_jj_pie.pdf")

# 对正负边权重数量执行 Fisher 精确检验
import rpy2.robjects as robjects
from rpy2.robjects import numpy2ri, default_converter
from rpy2.robjects.packages import importr

# 导入 R 的 'stats' 包，用于后续执行统计检验
stats = importr('stats')

# 构建一个 NumPy 数组，包含每个生态系统正负边权重的计数
# 这里的矩阵 m 包含了 6 个生态系统的正向和负向连接数
m = np.array([[pos_ct_jr, neg_ct_jr],
              [pos_ct_jj, neg_ct_jj],
              [pos_ct_g, neg_ct_g],
              [pos_ct_pa, neg_ct_pa],
              [pos_ct_n, neg_ct_n],
              [pos_ct_tz, neg_ct_tz]])

# 使用上下文管理器显式激活转换器，解决新版本 rpy2 不再支持直接使用 activate() 的问题
# 这种写法会自动处理 Python 对象与 R 对象之间的转换逻辑
with (default_converter + numpy2ri.converter).context():
    # 执行 Fisher 精确检验，并使用蒙特卡洛模拟 P 值（simulate_p_value=True 适用于 2xK 大尺寸列联表）
    res = stats.fisher_test(m, simulate_p_value=True)
    
    # 提取并打印 Fisher 检验得到的 P 值
    # 结果 res 是一个 R 列表对象，通过索引 [0][0] 获取具体的数值
    print('p-value: {}'.format(res[0][0]))
# 为每个生态系统的网络中心性数据框添加 'group' 列，标记其所属的生态系统类别
metrics_jr['group'] = 'Jurong'
metrics_jj['group'] = 'Jingjiang'
metrics_tz['group'] = 'Tongzhou'
metrics_pa['group'] = 'Pan\'an'
metrics_g['group'] = 'Rhizosphere Soil'
metrics_n['group'] = 'Bulb'

# 将所有生态系统的网络中心性数据合并到一个数据框中，用于后续的比较分析
metrics_cancat = pd.concat([metrics_jr, metrics_jj, metrics_tz, metrics_pa, metrics_g, metrics_n])

#执行 Kruskal-Wallis 检验 

# 初始化字典用于存储 Kruskal-Wallis 检验的 P 值
kw_pval = {}
# 遍历需要进行检验的中心性指标：'degree', 'closeness', 'betweenness'
for i in ['degree', 'closeness', 'betweenness']:
    # 对每个指标执行 Kruskal-Wallis 检验，比较六个生态系统组的差异
    stat, p_value = kruskal(metrics_jr[i], metrics_jj[i], metrics_pa[i], metrics_tz[i], metrics_g[i], metrics_n[i])
    # 打印每个指标在不同生态系统中的中位数和 Kruskal-Wallis 检验的 P 值
    print(i, 'Jurong:', np.median(metrics_jr[i]),
          'Jingjiang:', np.median(metrics_jj[i]),
          'Pan\'an:', np.median(metrics_pa[i]),
           'Tongzhou:', np.median(metrics_tz[i]),
           'Rhizosphere Soil:', np.median(metrics_g[i]),
           'Bulb:', np.median(metrics_n[i]))
    kw_pval[i] = p_value # 将 P 值存储到字典中

# 输出示例 (Kruskal-Wallis 检验结果):
# degree forest: 27.0 Jingjiang: 25.0 Pan\'an: 24.0 Tongzhou: 25.0 Rhizosphere Soil: 27.0 steppe: 27.0
# closeness forest: 4.29367e-06 Jingjiang: 3.559657e-06 Pan\'an: 4.3339205e-06 Tongzhou: 3.043133e-06 Rhizosphere Soil: 4.223972e-06 steppe: 4.4570035e-06
# betweenness forest: 873.0 Jingjiang: 1027.0 Pan\'an: 794.25 Tongzhou: 1044.0 Rhizosphere Soil: 834.0 steppe: 871.5

# 定义用于绘图的生态系统颜色映射字典
group_colors = {
    'Jurong': '#45B39D',
    'Jingjiang': '#666666',
    'Tongzhou': '#1E90FF',
    'Pan\'an': '#CD6155',
    'Rhizosphere Soil': '#A0522D',
    'Bulb': '#AF7AC5'
}

# 定义箱线图的显示顺序
order = ['Bulb', 'Rhizosphere Soil', 'Tongzhou', 'Pan\'an', 'Jurong', 'Jingjiang']
# 定义用于统计比较的组对
pairs=[
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Tongzhou'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Bulb'),
    ('Jurong', 'Tongzhou'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Rhizosphere Soil'),
    ('Jurong', 'Bulb'),
    ('Tongzhou', 'Jingjiang'),
    ('Tongzhou', 'Bulb'), # 调整：原始代码缺少 Tongzhou 与 Rhizosphere Soil 的比较，这里添加了
    ('Tongzhou', 'Rhizosphere Soil'),
    ('Jingjiang', 'Rhizosphere Soil'),
    ('Jingjiang', 'Bulb'),
    ('Rhizosphere Soil', 'Bulb'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
# 绘制箱线图：x 轴为生态系统组，y 轴为节点度，根据 group_colors 设置颜色，不显示异常值 (showfliers=False)
ax = sns.boxplot(data= metrics_cancat, x='group', y='degree', order=order,  palette=group_colors, showfliers=False)

# 使用 Annotator 在箱线图上添加统计显著性注释
annot = Annotator(ax, pairs, data=metrics_cancat, x='group', y='degree', order=order)
# 配置检验方法（曼-惠特尼 U 检验）和多重比较校正方法（Benjamini-Hochberg）
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用统计检验
annot.annotate() # 在图上添加注释（显著性星号或 P 值）

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Degree', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 在图的左上角添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.96, f'Kruskal-Wallis P = {kw_pval["degree"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和对齐方式
plt.ylim(-5, 400) # 设置 Y 轴的显示范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_degree_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表
print(" 各组 Degree 中位数 ")
for group_name in order:
    # 筛选出当前组的数据
    subset_data = metrics_cancat[metrics_cancat['group'] == group_name]['degree']
    # 计算中位数
    median_val = subset_data.median()
    # 打印中位数
    print(f"{group_name}: {median_val:.2f}") # 打印两位小数

# 定义箱线图的显示顺序，基于 Closeness 值对生态系统进行排序
order = ['Bulb', 'Rhizosphere Soil', 'Tongzhou', 'Pan\'an', 'Jingjiang', 'Jurong']

# 定义用于 Mann-Whitney U 检验的组对。这些是所有可能的两两比较。
pairs=[
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Bulb'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Tongzhou'),
    ('Rhizosphere Soil', 'Jurong'),
    ('Rhizosphere Soil', 'Bulb'),
    ('Rhizosphere Soil', 'Jingjiang'),
    ('Rhizosphere Soil', 'Tongzhou'),
    ('Jurong', 'Bulb'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Tongzhou'),
    ('Bulb', 'Jingjiang'),
    ('Bulb', 'Tongzhou'),
    ('Jingjiang', 'Tongzhou'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小 (宽x高)
# 绘制箱线图：
# data: 合并后的网络中心性数据框
# x: 分组变量 'group' (生态系统类型)
# y: 数值变量 'closeness' (接近中心性)
# order: 指定 x 轴分组的显示顺序
# palette: 使用预定义的生态系统颜色
# showfliers=False: 不显示离群点 (outliers)
ax = sns.boxplot(data=metrics_cancat, x='group', y='closeness', order=order, palette=group_colors, showfliers=False)

# 使用 Annotator 在箱线图上添加统计显著性注释
annot = Annotator(ax, pairs, data=metrics_cancat, x='group', y='closeness', order=order)
# 配置统计检验方法为 'Mann-Whitney' (曼-惠特尼 U 检验，非参数两独立样本检验)
# 配置多重比较校正方法为 "Benjamini-Hochberg" (FDR 校正)
# verbose=2: 显示详细的检验过程和结果
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用配置的统计检验
annot.annotate() # 在图上绘制显著性注释（如星号或 P 值）

ax.set_xlabel('') # 清空 X 轴标签，因为信息已在刻度上显示
ax.set_ylabel('Closeness', size=13) # 设置 Y 轴标签及其字体大小
ax.set_title('', size=13) # 设置图表标题及其字体大小
# 在图表内部添加 Kruskal-Wallis 检验的 P 值
# transform=ax.transAxes 表示坐标是相对于坐标轴的比例 (0-1)
plt.text(0.05, 0.97, f'Kruskal-Wallis P = {kw_pval["closeness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度字体大小、旋转角度和右对齐
plt.ylim(-0.05, 2.3) # 设置 Y 轴的显示范围，以更好地展示数据分布
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_closeness_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表为 PDF，紧密裁剪，高 DPI
plt.show() # 显示图表

# 定义箱线图的显示顺序，基于 Betweenness 值对生态系统进行排序
order = ['Bulb', 'Rhizosphere Soil', 'Tongzhou', 'Jurong', 'Pan\'an', 'Jingjiang']

# 定义用于 Mann-Whitney U 检验的组对
pairs=[
    ('Pan\'an', 'Rhizosphere Soil'),
    ('Pan\'an', 'Jurong'),
    ('Pan\'an', 'Bulb'),
    ('Pan\'an', 'Jingjiang'),
    ('Pan\'an', 'Tongzhou'),
    ('Rhizosphere Soil', 'Jurong'),
    ('Rhizosphere Soil', 'Bulb'),
    ('Rhizosphere Soil', 'Jingjiang'),
    ('Rhizosphere Soil', 'Tongzhou'),
    ('Jurong', 'Bulb'),
    ('Jurong', 'Jingjiang'),
    ('Jurong', 'Tongzhou'),
    ('Bulb', 'Jingjiang'),
    ('Bulb', 'Tongzhou'),
    ('Jingjiang', 'Tongzhou'),
]

plt.rcParams["figure.figsize"] = (4, 4) # 设置图表大小
# 绘制箱线图：数据、x轴、y轴、顺序、颜色、不显示异常值
ax = sns.boxplot(data=metrics_cancat, x='group', y='betweenness', order=order, palette=group_colors, showfliers=False)

# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=metrics_cancat, x='group', y='betweenness', order=order)
# 配置检验方法和多重比较校正
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Betweenness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.97, f'Kruskal-Wallis P = {kw_pval["betweenness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(-500, 9500) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_betweenness_ecosystems.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

# 从所有 OTU 的网络中心性数据中，根据丰度/稀有/广谱/特有类群的索引筛选出相应的数据
metrics_all_abundance = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_abundance/nodeG_plot.csv", index_col=0)
metrics_all_breadth = pd.read_csv("/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network_breadth/nodeG_plot.csv", index_col=0)

metrics_all_abundance.index = metrics_all_abundance.index.astype(str).map(lambda x: prefix + x)
metrics_all_breadth.index = metrics_all_breadth.index.astype(str).map(lambda x: prefix + x)

df_abun = metrics_all_abundance[metrics_all_abundance['Ecotype'] == 'Cores']
df_rare = metrics_all_abundance[metrics_all_abundance['Ecotype'] == 'Uniques']
df_gen = metrics_all_breadth[metrics_all_breadth['Ecotype'] == 'Generalists']
df_spe = metrics_all_breadth[metrics_all_breadth['Ecotype'] == 'Specialists']

metrics_abun = metrics_all[metrics_all.index.isin(df_abun.index)].copy()
metrics_abun['Ecotype'] = 'Cores' # 添加 'Ecotype' 列并赋值

metrics_rare = metrics_all[metrics_all.index.isin(df_rare.index)].copy()
metrics_rare['Ecotype'] = 'Uniques'

metrics_gen = metrics_all[metrics_all.index.isin(df_gen.index)].copy()
metrics_gen['Ecotype'] = 'Generalists'

metrics_spe = metrics_all[metrics_all.index.isin(df_spe.index)].copy()
metrics_spe['Ecotype'] = 'Specialists'

# 将所有生态类型的节点中心性数据合并到一个数据框中，用于后续的比较分析
metrics_ecotype = pd.concat([metrics_abun, metrics_rare, metrics_gen, metrics_spe])

# 初始化字典用于存储 Kruskal-Wallis 检验的 P 值
kw_pval = {}
# 遍历需要进行检验的中心性指标
for i in ['degree', 'closeness', 'betweenness']:
    # 对每个指标执行 Kruskal-Wallis 检验，比较四个生态类型组的差异
    stat, p_value = kruskal(metrics_abun[i], metrics_rare[i], metrics_gen[i], metrics_spe[i])
    # 打印每个指标在不同生态类型中的中位数和 P 值
    print(i, 'abundant:', np.median(metrics_abun[i]),
          'rare:', np.median(metrics_rare[i]),
          'generalist:', np.median(metrics_gen[i]),
          'specialist:', np.median(metrics_spe[i]))
    kw_pval[i] = p_value # 将 P 值存储到字典中

# 输出示例 (Kruskal-Wallis 检验结果):
# degree abundant: 49.0 rare: 53.0 generalist: 54.5 specialist: 54.0
# closeness abundant: 8.548408e-07 rare: 9.008329e-07 generalist: 8.993396000000001e-07 specialist: 8.638988e-07
# betweenness abundant: 1970.0 rare: 2626.0 generalist: 2245.0 specialist: 1886.0

# 定义箱线图的显示顺序（生态类型）
order=['Cores', 'Uniques', 'Generalists', 'Specialists']

# 定义不同生态类型的颜色映射
group_colors = {
    'Cores': '#5499C7', # 蓝色
    'Uniques': '#D4AC0D',     # 黄色
    'Generalists': '#EC7063',   # 红色
    'Specialists': '#45B39D'    # 绿色
}

# 定义用于 Mann-Whitney U 检验的组对 (所有两两比较)
pairs=[
    ('Cores', 'Uniques'),
    ('Cores', 'Generalists'),
    ('Cores', 'Specialists'),
    ('Uniques', 'Generalists'),
    ('Uniques', 'Specialists'),
    ('Generalists', 'Specialists'),
]

plt.rcParams["figure.figsize"] = (3, 4) # 设置图表大小 (宽x高)，相对更窄

#绘制节点度（Degree）箱线图 (针对生态类型) 
ax = sns.boxplot(data=metrics_ecotype, x='Ecotype', y='degree', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=metrics_ecotype, x='Ecotype', y='degree', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Degree', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["degree"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(-5, 110) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_degree_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点中介中心性（Betweenness）箱线图 (针对生态类型) 
ax = sns.boxplot(data=metrics_ecotype, x='Ecotype', y='betweenness', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=metrics_ecotype, x='Ecotype', y='betweenness', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Betweenness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["betweenness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(-5, 100) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_betweenness_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点接近中心性（Closeness）箱线图 (针对生态类型) 
ax = sns.boxplot(data=metrics_ecotype, x='Ecotype', y='closeness', order=order, palette=group_colors, showfliers=False) # 绘制箱线图
# 使用 Annotator 添加统计显著性注释
annot = Annotator(ax, pairs, data=metrics_ecotype, x='Ecotype', y='closeness', order=order)
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2) # 配置检验方法和校正
annot.apply_test() # 应用检验
annot.annotate() # 绘制注释

ax.set_xlabel('') # 清空 X 轴标签
ax.set_ylabel('Closeness', size=13) # 设置 Y 轴标签
ax.set_title('', size=13) # 设置图表标题
# 添加 Kruskal-Wallis 检验的 P 值
plt.text(0.05, 0.94, f'Kruskal-Wallis P = {kw_pval["closeness"]:.2e}', fontsize=10, ha='left', va='top', transform=ax.transAxes)
plt.xticks(fontsize=12, rotation=45, ha='right') # 设置 X 轴刻度
plt.ylim(-0.05, 2.1) # 设置 Y 轴范围
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_closeness_ecotypes.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#网络节点指标比较 (针对门 Phyla) 
# 将所有 OTU 的网络中心性指标数据 (metrics_all) 与分类学信息 (taxon) 进行合并。
# 合并依据是两个数据框的索引 ('left_index=True', 'right_index=True')。
# 只选择 taxon 数据框中的 'Phylum' 列进行合并，以便将每个 OTU 与其所属的门关联起来。
metrics_phyla = pd.merge(metrics_all, taxon[['Phylum']], left_index=True, right_index=True)

# 按 'Phylum' 列进行分组，并计算每个门的平均节点指标（degree, closeness, betweenness）。
# 这将用于确定绘图时门的排序。
metrics_phyla_mean = metrics_phyla.groupby('Phylum').mean()

#绘制节点度（Degree）的线点图 (Pointplot) 
# 根据平均度值对门进行降序排序，并选择前30个门。
top_30_phyla_degree = metrics_phyla_mean.sort_values(by='degree', ascending=False).index[:30].to_list()
# 使用 isin() 方法过滤 metrics_phyla 数据框，只保留 'Phylum' 在 top_30_phyla_degree 列表中的行
metrics_phyla_filtered = metrics_phyla[metrics_phyla['Phylum'].isin(top_30_phyla_degree)]

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小 (宽x高)，垂直方向相对更长，适合显示多个门
# 绘制点图：
# x='degree': X 轴显示节点度
# y='Phylum': Y 轴显示门名称
# data: 合并了门信息的网络中心性数据
# order: 指定 Y 轴上门的显示顺序
# join=False: 不连接点，只显示点和误差线
# capsize=0.2: 误差线帽的宽度
# errwidth=1: 误差线的宽度
# color='#DC7633': 设置点的颜色
sns.pointplot(x='degree', y='Phylum', data= metrics_phyla_filtered, order= top_30_phyla_degree, join=False, capsize=0.2, errwidth=1, color='#DC7633')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=10) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签及其字体大小
plt.xlabel('Degree', size=14) # 设置 X 轴标签及其字体大小
plt.title('', size=14) # 设置图表标题及其字体大小
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_degree_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表为 PDF，紧密裁剪，高 DPI
plt.show() # 显示图表

#绘制节点接近中心性（Closeness）的线点图 (Pointplot) 
# 根据平均接近中心性值对门进行降序排序，并选择前30个门。
top_30_phyla_closeness = metrics_phyla_mean.sort_values(by='closeness', ascending=False).index[:30].to_list()
metrics_phyla_filtered = metrics_phyla[metrics_phyla['Phylum'].isin(top_30_phyla_closeness)]

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小
# 绘制点图：x 轴为接近中心性，y 轴为门，使用指定数据、顺序和颜色
sns.pointplot(x='closeness', y='Phylum', data= metrics_phyla_filtered, order= top_30_phyla_closeness, join=False, capsize=0.2, errwidth=1, color='#BB8FCE')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=10) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签
plt.xlabel('Closeness', size=14) # 设置 X 轴标签
plt.title('', size=14) # 设置图表标题
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_closeness_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

#绘制节点中介中心性（Betweenness）的线点图 (Pointplot) 
# 根据平均中介中心性值对门进行降序排序，并选择前30个门。
top_30_phyla_betweenness = metrics_phyla_mean.sort_values(by='betweenness', ascending=False).index[:30].to_list()
metrics_phyla_filtered = metrics_phyla[metrics_phyla['Phylum'].isin(top_30_phyla_betweenness)]

plt.rcParams["figure.figsize"] = (3, 6) # 设置图表大小
# 绘制点图：x 轴为中介中心性，y 轴为门，使用指定数据、顺序和颜色
sns.pointplot(x='betweenness', y='Phylum', data= metrics_phyla_filtered, order= top_30_phyla_betweenness, join=False, capsize=0.2, errwidth=1, color='#73C6B6')

plt.xticks(fontsize=11) # 设置 X 轴刻度字体大小
plt.yticks(fontsize=10) # 设置 Y 轴刻度字体大小
plt.ylabel('Phylum', size=14) # 设置 Y 轴标签
plt.xlabel('Betweenness', size=14) # 设置 X 轴标签
plt.title('', size=14) # 设置图表标题
plt.savefig('/mnt/d/study/master/Main_Figure_tables/Figure_5/16S_ITS_network/network_betweenness_phyla.pdf', bbox_inches='tight', dpi=600) # 保存图表
plt.show() # 显示图表

